<a href="https://colab.research.google.com/github/mrfriman666/mrfriman666/blob/main/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22Untitled3_ipynb%22%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# @title 1/3 | SSM2 0.6 - Окружение и согласованные версии { display-mode: "form" }
import hashlib
import json
import os
from pathlib import Path
import re
import shutil
import subprocess
import time
import urllib.request

# Пусто: сохранить установленный Flutter; для новой среды взять 3.35.4.
# Можно указать точную stable-версию. Минимумы SDK проверяются ниже.
FLUTTER_VERSION = ""  # @param {type:"string"}
SDK = Path("/content/android-sdk")
FLUTTER = Path("/content/flutter")
JAVA = Path("/usr/lib/jvm/java-17-openjdk-amd64")
CONFIG = Path("/content/ssm2_fixed_env.json")
VERSIONS = {
    "agp": "8.11.1", "gradle": "8.14.3", "kotlin": "2.2.20",
    "compile_sdk": 36, "target_sdk": 35, "min_sdk": 24,
    "build_tools": "35.0.0", "ndk": "27.0.12077973",
}


def run(args, timeout=1800, input_text=None):
    print("\n>", " ".join(map(str, args)))
    p = subprocess.run(list(map(str, args)), text=True, input=input_text,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                       timeout=timeout)
    print(p.stdout[-5000:])
    if p.returncode:
        raise RuntimeError(f"Команда завершилась с кодом {p.returncode}")
    return p.stdout


def download(url, dest, sha256=None):
    dest = Path(dest)
    if not dest.exists():
        partial = dest.with_suffix(dest.suffix + ".part")
        with urllib.request.urlopen(url, timeout=180) as src, partial.open("wb") as out:
            shutil.copyfileobj(src, out, 1024 * 1024)
        partial.replace(dest)
    if sha256:
        digest = hashlib.sha256()
        with dest.open("rb") as src:
            for chunk in iter(lambda: src.read(1024 * 1024), b""):
                digest.update(chunk)
        if digest.hexdigest() != sha256:
            dest.unlink()
            raise RuntimeError("SHA256 архива не совпал; повторите загрузку")


def version_tuple(value):
    return tuple(int(x) for x in value.split("."))


t0 = time.monotonic()
print("SSM2 0.6 | Окружение. Старый проект и общие кэши не удаляются.")
run(["apt-get", "update", "-qq"], timeout=600)
run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless",
     "xz-utils", "unzip", "zip", "curl", "git"], timeout=1200)
os.environ.update(JAVA_HOME=str(JAVA), ANDROID_HOME=str(SDK), ANDROID_SDK_ROOT=str(SDK))
paths = [str(JAVA / "bin"), str(FLUTTER / "bin"),
         str(SDK / "cmdline-tools/latest/bin"), str(SDK / "platform-tools")]
os.environ["PATH"] = os.pathsep.join(paths + [os.environ.get("PATH", "")])
run([JAVA / "bin/java", "-version"])

manager = SDK / "cmdline-tools/latest/bin/sdkmanager"
if not manager.exists():
    archive = Path("/content/android-command-tools.zip")
    download("https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip", archive)
    stage = Path("/content/ssm2_cmdline_unpack")
    shutil.rmtree(stage, ignore_errors=True)
    run(["unzip", "-q", "-o", archive, "-d", stage])
    manager.parent.parent.parent.mkdir(parents=True, exist_ok=True)
    target = SDK / "cmdline-tools/latest"
    if target.exists():
        target.rename(target.with_name(f"previous-{int(time.time())}"))
    shutil.move(str(stage / "cmdline-tools"), str(target))
run([manager, f"--sdk_root={SDK}", "--licenses"], timeout=900, input_text="y\n" * 150)
run([manager, f"--sdk_root={SDK}", "platform-tools", "platforms;android-36",
     "platforms;android-35", "build-tools;35.0.0", "ndk;27.0.12077973"], timeout=3600)

requested = FLUTTER_VERSION.strip()
if requested or not (FLUTTER / "bin/flutter").exists():
    chosen = requested or "3.35.4"
    with urllib.request.urlopen("https://storage.googleapis.com/flutter_infra_release/releases/releases_linux.json", timeout=60) as response:
        releases = json.load(response)
    release = next((r for r in releases["releases"]
                    if r["version"] == chosen and r["channel"] == "stable"
                    and r.get("dart_sdk_arch", "x64") == "x64"), None)
    if release is None:
        raise RuntimeError(f"Stable Flutter {chosen} для Linux x64 не найден")
    existing = ""
    if (FLUTTER / "bin/flutter").exists():
        run(["git", "config", "--global", "--add", "safe.directory", FLUTTER])
        existing = run([FLUTTER / "bin/flutter", "--version"])
    if not re.search(rf"Flutter\s+{re.escape(chosen)}\b", existing):
        archive = Path(f"/content/flutter-{chosen}.tar.xz")
        download(releases["base_url"] + "/" + release["archive"], archive, release["sha256"])
        stage = Path("/content/ssm2_flutter_unpack")
        stage.mkdir(exist_ok=True)
        run(["tar", "-xJf", archive, "-C", stage], timeout=2400)
        if FLUTTER.exists():
            FLUTTER.rename(Path(f"/content/flutter-backup-{int(time.time())}"))
        shutil.move(str(stage / "flutter"), str(FLUTTER))

run(["git", "config", "--global", "--add", "safe.directory", FLUTTER])
flutter_info = run([FLUTTER / "bin/flutter", "--version", "--machine"])
info = json.loads(flutter_info[flutter_info.index("{"):])
run([FLUTTER / "bin/flutter", "config", "--no-analytics"])
run([FLUTTER / "bin/flutter", "config", f"--android-sdk={SDK}", f"--jdk-dir={JAVA}"])

# Check the INSTALLED SDK, not an assumed Flutter version or a broad log match.
checker = FLUTTER / "packages/flutter_tools/gradle/src/main/kotlin/DependencyVersionChecker.kt"
if checker.exists():
    source = checker.read_text(encoding="utf-8")
    for name, key in [("errorAGPVersion", "agp"), ("errorGradleVersion", "gradle"), ("errorKGPVersion", "kotlin")]:
        match = re.search(rf"{name}\s*[^=]*=\s*(?:AndroidPluginVersion|Version)\(\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*\)", source)
        if match:
            required = ".".join(match.groups())
            print(f"Flutter minimum {key}: {required}; выбрано: {VERSIONS[key]}")
            if version_tuple(VERSIONS[key]) < version_tuple(required):
                raise RuntimeError(f"Flutter {info['frameworkVersion']} требует {key} >= {required}. "
                                   "Этот набор версий не подходит. Укажите FLUTTER_VERSION = '3.35.4' и повторите ячейку. "
                                   "Проверка совместимости намеренно не отключается.")
run([FLUTTER / "bin/flutter", "precache", "--android"], timeout=2400)
required = [JAVA / "bin/java", manager, SDK / "platforms/android-36/android.jar",
            SDK / "build-tools/35.0.0/aapt", SDK / "build-tools/35.0.0/apksigner",
            SDK / "ndk/27.0.12077973/source.properties", FLUTTER / "bin/dart"]
for file in required:
    if not file.exists():
        raise RuntimeError(f"Не найден обязательный файл: {file}")
    print("[OK]", file)
config = {**VERSIONS, "sdk": str(SDK), "java": str(JAVA), "flutter": str(FLUTTER),
          "flutter_version": info["frameworkVersion"], "app": "/content/subaru_ssm2_fixed"}
CONFIG.write_text(json.dumps(config, indent=2), encoding="utf-8")
print(f"\nОкружение подготовлено за {time.monotonic() - t0:.0f} с. Конфигурация: {CONFIG}")
print("Далее выполните ячейку 2. Настоящая проверка Dart и APK будет в ячейке 3.")

SSM2 0.6 | Окружение. Старый проект и общие кэши не удаляются.

> apt-get update -qq
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


> apt-get install -y -qq openjdk-17-jdk-headless xz-utils unzip zip curl git


> /usr/lib/jvm/java-17-openjdk-amd64/bin/java -version
openjdk version "17.0.20" 2026-07-21
OpenJDK Runtime Environment (build 17.0.20+8-1-24.04-Ubuntu)
OpenJDK 64-Bit Server VM (build 17.0.20+8-1-24.04-Ubuntu, mixed mode, sharing)


> /content/android-sdk/cmdline-tools/latest/bin/sdkmanager --sdk_root=/content/android-sdk --licenses
Loading local repository...                                                     
[=========                              ] 25% Loading local repository...       
[=========                              ] 25% Fetch remote repository...        
[==========                             ] 26% Fetch remote reposit

In [7]:
# @title 2/3 | SSM2 0.6 - Полный проект, 28 PID, два транспорта { display-mode: "form" }
import ast
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import time

# Вариант A (bluetooth_classic) подтвержден на вашем автомобиле и оставлен по умолчанию.
# Вариант B остается запасным и не рекомендуется: на нем значения приходили некорректно.
BT_PACKAGE = "bluetooth_classic"  # @param ["bluetooth_classic", "flutter_bluetooth_serial"]
CONFIG = Path("/content/ssm2_fixed_env.json")
if not CONFIG.exists():
    raise RuntimeError("Сначала выполните новую ячейку 1")
CFG = json.loads(CONFIG.read_text(encoding="utf-8"))
APP = Path(CFG["app"])
FLUTTER = Path(CFG["flutter"])
os.environ.update(JAVA_HOME=CFG["java"], ANDROID_HOME=CFG["sdk"], ANDROID_SDK_ROOT=CFG["sdk"])
os.environ["PATH"] = os.pathsep.join([str(FLUTTER / "bin"), CFG["java"] + "/bin", os.environ.get("PATH", "")])


def run(args, timeout=1200):
    result = subprocess.run(list(map(str, args)), cwd=APP if APP.exists() else None,
                            text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            timeout=timeout)
    print(result.stdout[-6000:])
    if result.returncode:
        raise RuntimeError(f"Команда завершилась с кодом {result.returncode}: {args}")
    return result.stdout


def write(relative, text):
    destination = APP / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(text.strip("\n") + "\n", encoding="utf-8")


# Canonical catalog: used to generate Dart and the web catalog. No ECU values here.
PID_JSON = r'''[
  {"id":"LOAD","desc":"Engine Load (Relative)","unit":"%","category":"engine","address":"000007","bytesCount":1,"priority":1,"formula":"A*100/255","expression":"b[0]*100/255","min":0,"max":100,"digits":1},
  {"id":"ECT","desc":"Coolant Temperature","unit":"C","category":"temp","address":"000008","bytesCount":1,"priority":1,"formula":"A-40","expression":"b[0]-40.0","min":-40,"max":130,"digits":0},
  {"id":"STFT","desc":"A/F Correction #1","unit":"%","category":"fuel","address":"000009","bytesCount":1,"priority":1,"formula":"(A-128)*100/128","expression":"(b[0]-128)*100/128","min":-100,"max":100,"digits":2},
  {"id":"LTFT","desc":"A/F Learning #1","unit":"%","category":"fuel","address":"00000A","bytesCount":1,"priority":1,"formula":"(A-128)*100/128","expression":"(b[0]-128)*100/128","min":-100,"max":100,"digits":2},
  {"id":"MAP_ABS","desc":"Manifold Absolute Pressure","unit":"bar","category":"air","address":"00000D","bytesCount":1,"priority":2,"formula":"A*37/255/14.50377","expression":"b[0]*37/255/14.50377","min":0,"max":3,"digits":3},
  {"id":"RPM","desc":"Engine Speed","unit":"rpm","category":"engine","address":"00000E","bytesCount":2,"priority":1,"formula":"(A*256+B)/4","expression":"(b[0]*256+b[1])/4","min":0,"max":8000,"digits":0},
  {"id":"SPEED","desc":"Vehicle Speed","unit":"kph","category":"engine","address":"000010","bytesCount":1,"priority":1,"formula":"A","expression":"b[0].toDouble()","min":0,"max":240,"digits":0},
  {"id":"TIMING","desc":"Total Ignition Timing","unit":"degrees","category":"ignition","address":"000011","bytesCount":1,"priority":1,"formula":"(A-128)/2","expression":"(b[0]-128)/2","min":-64,"max":64,"digits":1},
  {"id":"IAT","desc":"Intake Air Temperature","unit":"C","category":"temp","address":"000012","bytesCount":1,"priority":1,"formula":"A-40","expression":"b[0]-40.0","min":-40,"max":130,"digits":0},
  {"id":"MAF","desc":"Mass Airflow","unit":"g/s","category":"air","address":"000013","bytesCount":2,"priority":1,"formula":"(A*256+B)/100","expression":"(b[0]*256+b[1])/100","min":0,"max":400,"digits":2},
  {"id":"TPS","desc":"Throttle Opening Angle","unit":"%","category":"throttle","address":"000015","bytesCount":1,"priority":1,"formula":"A*100/255","expression":"b[0]*100/255","min":0,"max":100,"digits":1},
  {"id":"O2_F","desc":"Front O2 Sensor #1","unit":"V","category":"fuel","address":"000016","bytesCount":2,"priority":2,"formula":"(A*256+B)/200","expression":"(b[0]*256+b[1])/200","min":0,"max":5,"digits":3},
  {"id":"BATT","desc":"Battery Voltage","unit":"V","category":"electric","address":"00001C","bytesCount":1,"priority":2,"formula":"A*8/100","expression":"b[0]*8/100","min":8,"max":18,"digits":2},
  {"id":"KNOCK_ADV","desc":"Knock Correction Advance","unit":"degrees","category":"ignition","address":"000022","bytesCount":1,"priority":1,"formula":"(A-128)/2","expression":"(b[0]-128)/2","min":-64,"max":64,"digits":1},
  {"id":"BARO","desc":"Atmospheric Pressure","unit":"bar","category":"air","address":"000023","bytesCount":1,"priority":3,"formula":"A*37/255/14.50377","expression":"b[0]*37/255/14.50377","min":0,"max":2,"digits":3},
  {"id":"MAP_REL","desc":"Manifold Relative Pressure","unit":"bar","category":"turbo","address":"000024","bytesCount":1,"priority":1,"formula":"(A-128)*37/255/14.50377","expression":"(b[0]-128)*37/255/14.50377","min":-1.3,"max":1.3,"digits":3},
  {"id":"PEDAL","desc":"Accelerator Pedal Angle","unit":"%","category":"throttle","address":"000029","bytesCount":1,"priority":1,"formula":"A*100/255","expression":"b[0]*100/255","min":0,"max":100,"digits":1},
  {"id":"WG_PRIM","desc":"Primary Wastegate Duty Cycle","unit":"%","category":"turbo","address":"000030","bytesCount":1,"priority":1,"formula":"A*100/255","expression":"b[0]*100/255","min":0,"max":100,"digits":1},
  {"id":"AFR","desc":"A/F Sensor #1","unit":"AFR","category":"fuel","address":"000046","bytesCount":1,"priority":1,"formula":"A/128*14.7","expression":"b[0]/128*14.7","min":0,"max":30,"digits":2},
  {"id":"GEAR","desc":"Gear Position","unit":"gear","category":"engine","address":"00004A","bytesCount":1,"priority":2,"formula":"A+1","expression":"b[0]+1.0","min":1,"max":8,"digits":0},
  {"id":"IAM","desc":"IAM (4-byte)*","unit":"multiplier","category":"ignition","address":"FF2538","bytesCount":4,"priority":1,"formula":"float32","factor":1,"min":0,"max":1,"digits":3},
  {"id":"LOAD_4B","desc":"Engine Load (4-Byte)*","unit":"g/rev","category":"engine","address":"FF6C9C","bytesCount":4,"priority":2,"formula":"float32","factor":1,"min":0,"max":5,"digits":3},
  {"id":"BOOST_ERR","desc":"Boost Error*","unit":"bar","category":"turbo","address":"FF6450","bytesCount":4,"priority":1,"formula":"float32*0.001333224","factor":0.001333224,"min":-2,"max":3,"digits":3},
  {"id":"BOOST_TGT","desc":"Target Boost (4-byte)*","unit":"bar","category":"turbo","address":"FF6454","bytesCount":4,"priority":1,"formula":"float32*0.001333224","factor":0.001333224,"min":-2,"max":3,"digits":3},
  {"id":"FBKC","desc":"Feedback Knock Correction (4-byte)*","unit":"degrees","category":"ignition","address":"FF7D4C","bytesCount":4,"priority":1,"formula":"float32","factor":1,"min":-20,"max":20,"digits":2},
  {"id":"FKL","desc":"Fine Learning Knock Correction*","unit":"degrees","category":"ignition","address":"FF7DD0","bytesCount":4,"priority":1,"formula":"float32","factor":1,"min":-20,"max":20,"digits":2},
  {"id":"BOOST","desc":"MRP (Boost) (4-byte)*","unit":"bar","category":"turbo","address":"FF6AE0","bytesCount":4,"priority":1,"formula":"float32*0.001333224","factor":0.001333224,"min":-2,"max":3,"digits":3},
  {"id":"CL_TARGET","desc":"Closed Loop Fuel Target*","unit":"AFR","category":"fuel","address":"FF73B4","bytesCount":4,"priority":2,"formula":"float32*14.7","factor":14.7,"min":0,"max":30,"digits":2}
]'''
PIDS = json.loads(PID_JSON)
assert len(PIDS) == 28 and len({p["id"] for p in PIDS}) == 28
assert sum("factor" in p for p in PIDS) == 8
for pid in PIDS:
    assert 0 <= int(pid["address"], 16) <= 0xFFFFFF - pid["bytesCount"] + 1

FILES = {}
FILES["lib/pids.dart"] = r'''
import 'dart:typed_data';

typedef PidFormula = double Function(List<int> bytes);

class SubaruPidDef {
  const SubaruPidDef({required this.id, required this.desc, required this.unit,
    required this.category, required this.address, required this.bytesCount,
    required this.priority, required this.formulaText, required this.minValue,
    required this.maxValue, required this.digits, this.formula, this.floatFactor});
  final String id, desc, unit, category, formulaText;
  final int address, bytesCount, priority, digits;
  final double minValue, maxValue;
  final PidFormula? formula;
  final double? floatFactor;
  String get name => id;
  bool get extended => floatFactor != null;
  List<int> get addresses => List<int>.generate(bytesCount, (i) => address + i);

  double? decode(List<int> bytes, {Endian endian = Endian.big}) {
    if (bytes.length != bytesCount || bytes.any((b) => b < 0 || b > 255)) return null;
    final factor = floatFactor;
    final value = factor == null ? formula!(bytes) :
      ByteData.sublistView(Uint8List.fromList(bytes)).getFloat32(0, endian) * factor;
    return value.isFinite ? value : null;
  }
}

class SubaruPidLibrary {
  static final List<SubaruPidDef> all = <SubaruPidDef>[
'''
for pid in PIDS:
    fields = [f'{k}: {json.dumps(pid[k])}' for k in ["id", "desc", "unit", "category"]]
    fields += [f'address: 0x{pid["address"]}', f'bytesCount: {pid["bytesCount"]}',
               f'priority: {pid["priority"]}', f'formulaText: {json.dumps(pid["formula"])}',
               f'minValue: {float(pid["min"])}', f'maxValue: {float(pid["max"])}',
               f'digits: {pid["digits"]}']
    fields.append(f'floatFactor: {float(pid["factor"])}' if "factor" in pid else
                  f'formula: (b) => {pid["expression"]}')
    FILES["lib/pids.dart"] += "    SubaruPidDef(" + ", ".join(fields) + "),\n"
FILES["lib/pids.dart"] += r'''
  ];
  static SubaruPidDef byId(String id) => all.firstWhere((p) => p.id == id);
  static const Set<String> defaults = {'RPM', 'ECT', 'TIMING', 'MAF', 'TPS', 'BATT', 'MAP_REL', 'AFR'};
}
'''

FILES["lib/protocol.dart"] = r'''
String hex2(int byte) => byte.toRadixString(16).padLeft(2, '0').toUpperCase();
String hexAddress(int address) => address.toRadixString(16).padLeft(6, '0').toUpperCase();
String readAddressCommand(int address) {
  if (address < 0 || address > 0xFFFFFF) throw RangeError.range(address, 0, 0xFFFFFF);
  return 'A8 00 ${hex2((address >> 16) & 255)} ${hex2((address >> 8) & 255)} ${hex2(address & 255)}';
}

String visibleText(String text) => text.runes.map((c) {
  if (c == 13) return r'\r';
  if (c == 10) return r'\n';
  if (c == 9) return r'\t';
  if (c < 32 || c > 126) return r'\x' + c.toRadixString(16).padLeft(2, '0');
  return String.fromCharCode(c);
}).join();

class ReplyError implements Exception {
  ReplyError(this.message);
  final String message;
  @override
  String toString() => message;
}

// Intentionally accepts only ATH0 + CAF1, one complete single-address response.
// No searching for E8 inside arbitrary garbage; headers and PCI are not guessed.
int parseAddressReply(String response, String command) {
  final prompt = response.indexOf('>');
  if (prompt < 0) throw ReplyError('INCOMPLETE: no prompt');
  if (response.substring(prompt + 1).trim().isNotEmpty) throw ReplyError('EXTRA_AFTER_PROMPT');
  final lines = response.substring(0, prompt).toUpperCase().split(RegExp(r'[\r\n]+'));
  final payloads = <List<int>>[];
  final echo = command.replaceAll(' ', '').toUpperCase();
  for (var line in lines) {
    line = line.trim();
    if (line.isEmpty) continue;
    if (line.startsWith('SEARCHING...')) {
      line = line.substring('SEARCHING...'.length).trim();
      if (line.isEmpty) continue;
    }
    final compact = line.replaceAll(RegExp(r'[ \t]'), '');
    if (compact == echo) continue;
    if (!RegExp(r'^[0-9A-F]+$').hasMatch(compact) || compact.length.isOdd) {
      throw ReplyError('ELM: $line');
    }
    final bytes = <int>[];
    for (var i = 0; i < compact.length; i += 2) {
      bytes.add(int.parse(compact.substring(i, i + 2), radix: 16));
    }
    if (bytes.first == 0x7F) throw ReplyError('ECU NEGATIVE: ${bytes.map(hex2).join(' ')}');
    payloads.add(bytes);
  }
  if (payloads.length != 1) throw ReplyError('EXPECTED_ONE_REPLY: ${payloads.length}');
  final data = payloads.single;
  if (data.length != 2 || data[0] != 0xE8) {
    throw ReplyError('EXPECTED_E8_PLUS_ONE_BYTE: ${data.map(hex2).join(' ')}');
  }
  return data[1];
}

bool allowedDiagnostic(String command) {
  final c = command.trim().toUpperCase();
  return const {'ATI', 'ATRV', 'ATDP', 'ATDPN', 'AT@1', '0100', '010C', '0105'}.contains(c) ||
    RegExp(r'^A8 00 [0-9A-F]{2} [0-9A-F]{2} [0-9A-F]{2}$').hasMatch(c);
}
'''

FILES["lib/bt_transport.dart"] = r'''
// Uint8List приходит из package:flutter/services.dart, отдельный импорт не нужен.
import 'package:flutter/services.dart';
import 'package:permission_handler/permission_handler.dart';

class BtDevice {
  const BtDevice(this.name, this.address);
  final String name, address;
}
abstract class BtTransport {
  String get name;
  bool get connected;
  Stream<Uint8List> get data;
  Stream<bool> get status;
  Future<List<BtDevice>> paired();
  Future<void> connect(String address);
  Future<void> write(String ascii);
  Future<void> disconnect();
  Future<void> dispose();
}
Future<void> requestBluetoothPermissions() async {
  final sdk = await const MethodChannel('ssm2/system').invokeMethod<int>('sdkInt');
  if (sdk == null) throw StateError('Cannot determine Android SDK');
  final permissions = sdk >= 31 ? <Permission>[Permission.bluetoothConnect, Permission.bluetoothScan] :
    <Permission>[Permission.locationWhenInUse];
  final result = await permissions.request();
  if (result.values.any((s) => !s.isGranted)) throw StateError('Разрешения Bluetooth не выданы');
}
'''

FILES["transport_templates/classic.dart.txt"] = r'''
import 'dart:async';
import 'dart:typed_data';
import 'package:bluetooth_classic/bluetooth_classic.dart';
import 'bt_transport.dart';

class SelectedTransport implements BtTransport {
  SelectedTransport() {
    // The plugin exposes single-subscription controllers. Subscribe exactly once.
    _rx = _bt.onDeviceDataReceived().listen((b) => _data.add(Uint8List.fromList(b)),
      onError: (Object e) { _data.addError(e); _lost(); });
    _st = _bt.onDeviceStatusChanged().listen((code) {
      if (code == 0) _lost();
      if (code == 2) { _connected = true; _status.add(true); }
    }, onError: (Object e) { _data.addError(e); _lost(); });
  }
  final BluetoothClassic _bt = BluetoothClassic();
  final _data = StreamController<Uint8List>.broadcast();
  final _status = StreamController<bool>.broadcast();
  late final StreamSubscription<Uint8List> _rx;
  late final StreamSubscription<int> _st;
  bool _connected = false;
  @override
  String get name => 'A / bluetooth_classic 0.0.4 (local RX patch)';
  @override
  bool get connected => _connected;
  @override
  Stream<Uint8List> get data => _data.stream;
  @override
  Stream<bool> get status => _status.stream;
  void _lost() { _connected = false; _status.add(false); }
  @override
  Future<List<BtDevice>> paired() async {
    await requestBluetoothPermissions();
    final devices = await _bt.getPairedDevices();
    return devices.map((d) => BtDevice(d.name ?? '', d.address)).toList();
  }
  @override
  Future<void> connect(String address) async {
    await requestBluetoothPermissions();
    final accepted = await _bt.connect(address, '00001101-0000-1000-8000-00805f9b34fb');
    if (!accepted) throw StateError('SPP connection rejected');
    _connected = true;
  }
  @override
  Future<void> write(String ascii) async {
    if (!connected) throw StateError('SPP disconnected');
    if (!await _bt.write(ascii)) throw StateError('Native write rejected');
  }
  @override
  Future<void> disconnect() async {
    _connected = false;
    // The vendored native disconnect is null-safe, including after a link loss.
    await _bt.disconnect();
  }
  @override
  Future<void> dispose() async {
    try { await disconnect(); } finally {
      await _rx.cancel(); await _st.cancel();
      await _data.close(); await _status.close();
    }
  }
}
BtTransport createTransport() => SelectedTransport();
'''

FILES["lib/elm.dart"] = r'''
import 'dart:async';
import 'dart:collection';
import 'dart:typed_data';
import 'bt_transport.dart';
import 'pids.dart';
import 'protocol.dart';

class AsyncLock {
  Future<void> _tail = Future<void>.value();
  Future<T> run<T>(Future<T> Function() action) async {
    final previous = _tail;
    final gate = Completer<void>();
    _tail = gate.future;
    await previous;
    try { return await action(); } finally { gate.complete(); }
  }
}

class PidRead {
  PidRead(this.bytes, this.elapsedMs);
  final List<int> bytes;
  final int elapsedMs;
}

class ElmDriver {
  ElmDriver(this.transport) {
    _data = transport.data.listen(_onData, onError: (Object e) => _lost('RX ERROR: $e'));
    _status = transport.status.listen((connected) {
      if (!connected) _lost('SPP disconnected');
    });
  }
  final BtTransport transport;
  final AsyncLock _lock = AsyncLock();
  late final StreamSubscription<Uint8List> _data;
  late final StreamSubscription<bool> _status;
  final Queue<String> trace = Queue<String>();
  Completer<String>? _pending;
  String _rx = '';
  bool _synchronized = false, _settling = false, _disposed = false;
  int timeoutMs = 1200, tx = 0, writeAccepted = 0, rxBytes = 0, prompts = 0, timeouts = 0;
  String identity = '', voltage = '';
  bool get ready => transport.connected && _synchronized && !_disposed;

  void log(String line) {
    trace.add('${DateTime.now().toIso8601String()} $line');
    while (trace.length > 400) { trace.removeFirst(); }
  }
  void _lost(String reason) {
    _synchronized = false;
    log(reason);
    final p = _pending;
    if (p != null && !p.isCompleted) p.completeError(ReplyError(reason));
  }
  void _onData(Uint8List bytes) {
    rxBytes += bytes.length;
    final text = String.fromCharCodes(bytes);
    log('RX ${visibleText(text)}');
    final p = _pending;
    if (p == null || p.isCompleted) {
      if (!_settling && text.trim().isNotEmpty) {
        _synchronized = false;
        log('UNSOLICITED: reconnect required');
      }
      return;
    }
    _rx += text;
    if (_rx.length > 16384) {
      _lost('RX OVERFLOW');
      return;
    }
    final end = _rx.indexOf('>');
    if (end >= 0) {
      prompts++;
      if (_rx.substring(end + 1).trim().isNotEmpty) _synchronized = false;
      p.complete(_rx);
    }
  }

  Future<String> _exchange(String command, {int? timeout}) async {
    if (!ready) throw ReplyError('Нет синхронизации. Переподключите адаптер.');
    if (command.contains('\r') || command.contains('\n') || command.trim().isEmpty) {
      throw ArgumentError('One nonempty command is required');
    }
    final p = Completer<String>();
    _pending = p;
    _rx = '';
    final watch = Stopwatch()..start();
    tx++;
    log('TX ${visibleText('$command\r')}');
    try {
      // Both futures are observed immediately; write completion is not an ECU acknowledgement.
      final values = await Future.wait<Object>([
        transport.write('$command\r').then<Object>((_) {
          writeAccepted++;
          log('WRITE_OK (native transport, not ECU acknowledgement)');
          return true;
        }), p.future,
      ], eagerError: true).timeout(Duration(milliseconds: timeout ?? timeoutMs));
      final raw = values[1] as String;
      if (!ready) throw ReplyError('LINK_LOST_OR_EXTRA_DATA');
      log('PROMPT ${watch.elapsedMilliseconds}ms');
      return raw;
    } on TimeoutException {
      timeouts++;
      _synchronized = false;
      log('TIMEOUT ${watch.elapsedMilliseconds}ms partial=${visibleText(_rx)}');
      if (!p.isCompleted) p.complete('');
      throw ReplyError('TIMEOUT: reconnect required; partial response rejected');
    } catch (e) {
      _synchronized = false;
      if (!p.isCompleted) p.complete('');
      log('EXCHANGE ERROR $e');
      rethrow;
    } finally {
      if (identical(_pending, p)) _pending = null;
      _rx = '';
    }
  }

  Future<void> initialize(String address) => _lock.run(() async {
    if (_disposed) throw StateError('Disposed');
    _settling = true;
    _synchronized = false;
    try {
      await transport.disconnect();
      await transport.connect(address);
      await Future<void>.delayed(const Duration(milliseconds: 300));
      _synchronized = true;
      await _exchange('ATZ', timeout: 3500);
      for (final command in ['ATE0', 'ATL0', 'ATS1', 'ATH0', 'ATSP6', 'ATSH 7E0',
        'ATCRA 7E8', 'ATCAF1', 'ATCFC1', 'ATST 64', 'ATAT1']) {
        final answer = await _exchange(command);
        if (!answer.toUpperCase().split(RegExp(r'[\r\n>]+')).any((l) => l.trim() == 'OK')) {
          throw ReplyError('$command не принят: ${visibleText(answer)}');
        }
      }
      identity = (await _exchange('ATI')).replaceAll(RegExp(r'[\r\n>]'), ' ').trim();
      if (identity.isEmpty || identity == '?') throw ReplyError('ATI has no identity');
      voltage = (await _exchange('ATRV')).replaceAll(RegExp(r'[\r\n>]'), ' ').trim();
      log('READY: CAN 7E0/7E8, ATH0, CAF1, A8 single address');
    } catch (e) {
      _synchronized = false;
      try { await transport.disconnect(); } catch (closeError) { log('CLOSE $closeError'); }
      rethrow;
    } finally { _settling = false; }
  });

  Future<PidRead> readRange(int address, int length) => _lock.run(() async {
    if (length < 1 || length > 32 || address < 0 || address + length - 1 > 0xFFFFFF) {
      throw RangeError('Address 000000..FFFFFF, length 1..32');
    }
    final watch = Stopwatch()..start();
    final bytes = <int>[];
    for (var offset = 0; offset < length; offset++) {
      final cmd = readAddressCommand(address + offset);
      final response = await _exchange(cmd);
      try {
        bytes.add(parseAddressReply(response, cmd));
      } catch (e) {
        log('REJECT $cmd: $e');
        // NO DATA has a complete prompt, so the link is still synchronized.
        rethrow;
      }
    }
    return PidRead(List<int>.unmodifiable(bytes), watch.elapsedMilliseconds);
  });
  Future<PidRead> readPid(SubaruPidDef pid) => readRange(pid.address, pid.bytesCount);
  Future<String> diagnostic(String command) {
    final normalized = command.trim().toUpperCase();
    if (!allowedDiagnostic(normalized)) throw ArgumentError('Разрешены только диагностические команды чтения');
    return _lock.run(() => _exchange(normalized));
  }
  Future<void> disconnect() async {
    _lost('Disconnect requested');
    await transport.disconnect();
  }
  Future<void> dispose() async {
    if (_disposed) return;
    _disposed = true;
    try { await disconnect(); } catch (e) { log('CLOSE ERROR $e'); }
    try { await _lock.run(() async {}); } finally {
      await _data.cancel(); await _status.cancel();
      await transport.dispose();
    }
  }
}
'''

FILES["lib/engine.dart"] = r'''
import 'dart:async';
import 'package:flutter/foundation.dart';
import 'elm.dart';
import 'pids.dart';
import 'samples.dart';

export 'samples.dart';

class SsmEngine extends ChangeNotifier {
  SsmEngine(this.elm);
  final ElmDriver elm;
  final Set<String> enabled = {...SubaruPidLibrary.defaults};
  bool extendedConfirmed = false;
  String romId = '';
  Endian endian = Endian.big;
  bool running = false, _disposed = false;
  int _generation = 0;
  Future<void>? _loop;
  final latest = <String, PidSample>{};
  final attempts = <String, PidSample>{};
  final history = <String, List<PidSample>>{};
  final _events = StreamController<PidSample>.broadcast();
  Stream<PidSample> get events => _events.stream;
  int goodCount = 0, failedCount = 0;
  String message = 'Нет данных';
  final _replyTimes = <DateTime>[];
  List<SubaruPidDef> get active => SubaruPidLibrary.all.where((p) => enabled.contains(p.id) &&
    (!p.extended || extendedConfirmed)).toList();
  double get quality => goodCount + failedCount == 0 ? 0 : goodCount * 100 / (goodCount + failedCount);
  double get pidReadsPerSecond {
    final now = DateTime.now();
    _replyTimes.removeWhere((t) => now.difference(t).inSeconds >= 10);
    if (_replyTimes.length < 2) return 0;
    final seconds = now.difference(_replyTimes.first).inMilliseconds / 1000;
    return seconds > 0 ? (_replyTimes.length - 1) / seconds : 0;
  }
  double frequency(String id) {
    final h = history[id];
    if (h == null || h.length < 2) return 0;
    final end = h.last.time;
    if (DateTime.now().difference(end).inSeconds > 10) return 0;
    final start = h.length > 10 ? h.length - 10 : 0;
    final seconds = end.difference(h[start].time).inMilliseconds / 1000;
    return seconds > 0 ? (h.length - 1 - start) / seconds : 0;
  }
  int? ageMs(String id) {
    final sample = latest[id];
    return sample == null ? null : DateTime.now().difference(sample.time).inMilliseconds;
  }
  bool stale(String id) => (ageMs(id) ?? 999999) > 3000;
  void _notify() { if (!_disposed) notifyListeners(); }
  void _publish(PidSample sample) {
    attempts[sample.pid.id] = sample;
    if (sample.good) {
      latest[sample.pid.id] = sample;
      final list = history.putIfAbsent(sample.pid.id, () => []);
      list.add(sample);
      if (list.length > 300) list.removeAt(0);
      goodCount++;
      _replyTimes.add(sample.time);
      if (_replyTimes.length > 300) _replyTimes.removeAt(0);
    } else { failedCount++; }
    if (!_disposed) _events.add(sample);
    _notify();
  }
  Future<void> start() async {
    if (running || _disposed) return;
    if (_loop != null) await _loop;
    if (_disposed || !elm.ready) throw StateError('Сначала подключите адаптер');
    if (active.isEmpty) throw StateError('Выберите хотя бы один PID');
    final generation = ++_generation;
    running = true;
    message = 'Опрос A8';
    _loop = _poll(generation);
    _notify();
  }
  Future<void> _poll(int generation) async {
    var round = 0;
    try {
      while (generation == _generation && running && elm.ready) {
        final list = active;
        if (list.isEmpty) break;
        for (final pid in list) {
          if (generation != _generation || !running || !elm.ready) return;
          final interval = pid.priority == 1 ? 1 : pid.priority == 2 ? 2 : 5;
          if (round % interval != 0) continue;
          try {
            final read = await elm.readPid(pid);
            if (generation != _generation || _disposed) return;
            final value = pid.decode(read.bytes, endian: endian);
            _publish(PidSample(pid, DateTime.now(), value, read.bytes, read.elapsedMs,
              value == null ? 'INVALID_FLOAT_OR_LENGTH' : ''));
          } catch (e) {
            if (generation != _generation || _disposed) return;
            message = '$e';
            _publish(PidSample(pid, DateTime.now(), null, const [], 0, '$e'));
          }
          await Future<void>.delayed(const Duration(milliseconds: 5));
        }
        round++;
        await Future<void>.delayed(const Duration(milliseconds: 10));
      }
    } finally {
      if (generation == _generation) {
        running = false;
        if (!elm.ready) message = 'Опрос остановлен: переподключите адаптер';
        _notify();
      }
    }
  }
  Future<void> stop() async {
    _generation++;
    running = false;
    final current = _loop;
    if (current != null) await current;
    if (identical(current, _loop)) _loop = null;
    _notify();
  }
  Future<void> configure(Set<String> ids, bool confirm, String rom, Endian byteOrder) async {
    await stop();
    enabled..clear()..addAll(ids);
    extendedConfirmed = confirm && rom.trim().isNotEmpty;
    if (!extendedConfirmed) {
      enabled.removeWhere((id) => SubaruPidLibrary.all.any((p) => p.id == id && p.extended));
    }
    romId = rom.trim(); endian = byteOrder;
    latest.clear(); attempts.clear(); history.clear(); _replyTimes.clear();
    goodCount = 0; failedCount = 0;
    _notify();
  }
  @override
  void dispose() {
    _disposed = true;
    running = false; _generation++;
    unawaited(_events.close());
    super.dispose();
  }
}
'''

FILES["lib/model.dart"] = r'''
import 'dart:async';
import 'dart:convert';
import 'dart:io';
import 'package:flutter/foundation.dart';
import 'package:path_provider/path_provider.dart';
import 'package:share_plus/share_plus.dart';
import 'bt_transport.dart';
import 'elm.dart';
import 'engine.dart';
import 'pids.dart';
import 'protocol.dart';

String csvCell(Object? value) => '"${(value?.toString() ?? '').replaceAll('"', '""')}"';

class CsvLogger {
  IOSink? _sink;
  Future<void> _writes = Future<void>.value();
  File? file;
  bool active = false;
  int count = 0, _pending = 0;
  String error = '';
  Future<void> start() async {
    await stop();
    final dir = await getApplicationDocumentsDirectory();
    file = File('${dir.path}/ssm2_${DateTime.now().millisecondsSinceEpoch}.csv');
    final sink = file!.openWrite();
    _sink = sink;
    unawaited(sink.done.catchError((Object e) { error = '$e'; active = false; }));
    sink.writeln('timestamp,pid,value,unit,address,raw,read_ms,status');
    await sink.flush();
    count = 0; error = ''; active = true;
  }
  void add(PidSample sample) {
    final sink = _sink;
    if (!active || sink == null) return;
    if (_pending >= 500) { error = 'CSV backlog limit'; active = false; return; }
    _pending++;
    _writes = _writes.then((_) async {
      sink.writeln([sample.time.toIso8601String(), sample.pid.id, sample.value,
        sample.pid.unit, hexAddress(sample.pid.address), sample.raw.map(hex2).join(' '),
        sample.readMs, sample.good ? 'fresh' : sample.error].map(csvCell).join(','));
      count++;
      if (count % 10 == 0) await sink.flush();
    }).catchError((Object e) { error = '$e'; active = false; }).whenComplete(() { _pending--; });
  }
  Future<void> stop() async {
    active = false;
    await _writes;
    final sink = _sink;
    _sink = null;
    if (sink != null) {
      try { await sink.flush(); await sink.close(); } catch (e) { error = '$e'; }
    }
  }
}

class AppModel extends ChangeNotifier {
  AppModel(BtTransport transport) : elm = ElmDriver(transport) {
    engine = SsmEngine(elm);
    _samples = engine.events.listen(logger.add);
    _link = transport.status.listen((connected) {
      if (!connected && !busy && !_disposed) {
        message = 'SPP разорван. Переподключите адаптер.';
        unawaited(logger.stop());
        changed();
      }
    });
  }
  final ElmDriver elm;
  late final SsmEngine engine;
  final logger = CsvLogger();
  late final StreamSubscription<PidSample> _samples;
  late final StreamSubscription<bool> _link;
  List<BtDevice> devices = [];
  String? selected;
  String message = 'Сопрягите SPP-адаптер в настройках Android';
  String terminal = '', scanner = '', chartId = 'RPM';
  bool busy = false, foreground = true, _disposed = false;
  void changed() { if (!_disposed) notifyListeners(); }
  Future<void> perform(Future<void> Function() action) async {
    if (busy || _disposed) return;
    busy = true; changed();
    try { await action(); } catch (e) { message = '$e'; elm.log('APP $e'); }
    finally { busy = false; changed(); }
  }
  Future<void> restore() async {
    busy = true;
    try {
      final dir = await getApplicationSupportDirectory();
      final file = File('${dir.path}/ssm2_settings.json');
      if (!await file.exists() || _disposed) return;
      final json = jsonDecode(await file.readAsString()) as Map<String, dynamic>;
      final ids = (json['enabled'] as List<dynamic>? ?? []).whereType<String>().where(
        (id) => SubaruPidLibrary.all.any((p) => p.id == id)).toSet();
      await engine.configure(ids, json['confirmed'] == true, json['rom'] as String? ?? '',
        json['endian'] == 'little' ? Endian.little : Endian.big);
      changed();
    } catch (e) { message = 'Настройки не загружены: $e'; }
    finally { busy = false; changed(); }
  }
  Future<void> saveSettings() async {
    final dir = await getApplicationSupportDirectory();
    await File('${dir.path}/ssm2_settings.json').writeAsString(jsonEncode({
      'enabled': engine.enabled.toList(), 'confirmed': engine.extendedConfirmed,
      'rom': engine.romId, 'endian': engine.endian == Endian.big ? 'big' : 'little',
    }), flush: true);
  }
  Future<void> refreshDevices() => perform(() async {
    devices = await elm.transport.paired();
    if (!devices.any((d) => d.address == selected)) selected = devices.isEmpty ? null : devices.first.address;
    message = devices.isEmpty ? 'Нет сопряженных устройств' : 'Выберите адаптер';
  });
  Future<void> connect() => perform(() async {
    final address = selected;
    if (address == null) throw StateError('Выберите устройство');
    await elm.disconnect(); await engine.stop(); await logger.stop();
    await engine.configure({...engine.enabled}, engine.extendedConfirmed, engine.romId, engine.endian);
    message = 'Инициализация ELM327'; changed();
    await elm.initialize(address);
    message = 'Адаптер отвечает. Подтверждение ECU: только по ответам E8.';
    if (engine.active.isNotEmpty && foreground) await engine.start();
  });
  Future<void> disconnect() => perform(() async {
    await elm.disconnect(); await engine.stop(); await logger.stop(); message = 'Отключено';
  });
  Future<void> togglePolling() => perform(() async {
    if (engine.running) { await engine.stop(); message = 'Опрос на паузе'; }
    else { await engine.start(); message = 'Опрос запущен'; }
  });
  Future<void> selectPid(String id, bool value) => perform(() async {
    final next = {...engine.enabled};
    if (value) { next.add(id); } else { next.remove(id); }
    await engine.configure(next, engine.extendedConfirmed, engine.romId, engine.endian);
    await saveSettings(); message = 'Выбор сохранен. Нажмите Старт для опроса.';
  });
  Future<void> preset(bool all) => perform(() async {
    await engine.configure(all ? SubaruPidLibrary.all.where((p) => !p.extended).map((p) => p.id).toSet() :
      {...SubaruPidLibrary.defaults}, engine.extendedConfirmed, engine.romId, engine.endian);
    await saveSettings(); message = 'Набор сохранен; опрос на паузе';
  });
  Future<void> configureExtended(bool confirm, String rom, Endian endian) => perform(() async {
    if (confirm && rom.trim().isEmpty) throw ArgumentError('Введите ROM ID из своего def-файла');
    await engine.configure({...engine.enabled}, confirm, rom, endian);
    await saveSettings(); message = 'Настройки ROM сохранены. Автопроверка ROM не выполнялась.';
  });
  Future<void> sendDiagnostic(String command) => perform(() async {
    await engine.stop();
    terminal = '';
    final reply = await elm.diagnostic(command);
    terminal = '> ${command.trim().toUpperCase()}\\r\n${visibleText(reply)}';
    message = 'Ручной запрос завершен; опрос остается на паузе';
  });
  Future<void> scan(String start, String count) => perform(() async {
    final address = int.parse(start.trim().replaceFirst(RegExp(r'^0[xX]'), ''), radix: 16);
    final length = int.parse(count);
    if (address >= 0xFF0000 && !engine.extendedConfirmed) throw StateError('Сначала подтвердите ROM');
    await engine.stop();
    scanner = '';
    final result = await elm.readRange(address, length);
    scanner = List<String>.generate(result.bytes.length, (i) =>
      '0x${hexAddress(address + i)}   ${hex2(result.bytes[i])}   ${result.bytes[i]}').join('\n');
    message = 'Прочитано ${result.bytes.length} байт за ${result.elapsedMs} мс. Опрос на паузе.';
  });
  Future<void> toggleLog() => perform(() async {
    if (logger.active) { await logger.stop(); }
    else {
      if (!engine.running || !foreground) throw StateError('Сначала запустите опрос в открытом приложении');
      await logger.start();
      if (!foreground) await logger.stop();
    }
  });
  Future<void> exportCsv() => perform(() async {
    await logger.stop();
    final file = logger.file;
    if (file == null || logger.count == 0) throw StateError('Нет записей CSV');
    await Share.shareXFiles([XFile(file.path)], text: 'SSM2 PID log');
  });
  Future<void> exportTrace() => perform(() async {
    final dir = await getApplicationDocumentsDirectory();
    final file = File('${dir.path}/ssm2_trace_${DateTime.now().millisecondsSinceEpoch}.txt');
    await file.writeAsString('${elm.transport.name}\nROM (user): ${engine.romId}\n'
      'Endian: ${engine.endian == Endian.big ? 'big' : 'little'}\n'
      '${elm.trace.join('\n')}\n', flush: true);
    await Share.shareXFiles([XFile(file.path)], text: 'SSM2 TX/RX diagnostic trace');
  });
  Future<void> shutdown() async {
    if (_disposed) return;
    _disposed = true;
    await _link.cancel();
    try { await elm.disconnect(); } catch (e) { elm.log('SHUTDOWN $e'); }
    await engine.stop(); await _samples.cancel(); await logger.stop();
    try { await elm.dispose(); } catch (e) { elm.log('DISPOSE $e'); }
    engine.dispose(); super.dispose();
  }
}
'''

FILES["lib/main.dart"] = r'''
import 'dart:async';
import 'dart:math' as math;
import 'package:flutter/foundation.dart';
import 'package:flutter/material.dart';
import 'package:flutter/services.dart';
import 'analyzer.dart';
import 'derived.dart';
import 'engine.dart';
import 'model.dart';
import 'pids.dart';
import 'protocol.dart';
import 'transport_selected.dart';

void main() { WidgetsFlutterBinding.ensureInitialized(); runApp(const SsmApp()); }
const cyan = Color(0xFF22D3EE);
const muted = Color(0xFF9AAAC0);

class SsmApp extends StatelessWidget {
  const SsmApp({super.key});
  @override
  Widget build(BuildContext context) => MaterialApp(
    title: 'SSM2 Telemetry 0.5', debugShowCheckedModeBanner: false,
    theme: ThemeData(colorScheme: ColorScheme.fromSeed(seedColor: cyan, brightness: Brightness.dark),
      scaffoldBackgroundColor: const Color(0xFF080D18), useMaterial3: true),
    home: const HomeShell(),
  );
}

class HomeShell extends StatefulWidget {
  const HomeShell({super.key});
  @override
  State<HomeShell> createState() => _HomeShellState();
}
class _HomeShellState extends State<HomeShell> with WidgetsBindingObserver {
  late final AppModel model;
  late final Listenable changes;
  late final Timer timer;
  int tab = 0;
  @override
  void initState() {
    super.initState();
    model = AppModel(createTransport());
    changes = Listenable.merge([model, model.engine]);
    WidgetsBinding.instance.addObserver(this);
    unawaited(model.restore());
    timer = Timer.periodic(const Duration(milliseconds: 500), (_) { if (mounted) setState(() {}); });
  }
  @override
  void didChangeAppLifecycleState(AppLifecycleState state) {
    if (state == AppLifecycleState.resumed) model.foreground = true;
    if (state == AppLifecycleState.paused) {
      model.foreground = false;
      unawaited(model.engine.stop());
      unawaited(model.logger.stop());
    }
  }
  @override
  void dispose() {
    timer.cancel(); WidgetsBinding.instance.removeObserver(this);
    unawaited(model.shutdown()); super.dispose();
  }
  @override
  Widget build(BuildContext context) => AnimatedBuilder(animation: changes, builder: (context, _) {
    final engine = model.engine;
    final lastTimes = engine.latest.values.map((s) => s.time).toList()..sort();
    final live = model.elm.ready && lastTimes.isNotEmpty && DateTime.now().difference(lastTimes.last).inSeconds < 3;
    final pages = <Widget>[AdapterPage(model), DashboardPage(model), PidPage(model),
      LoggerPage(model), GraphPage(model), AnalyzerPage(model), DiagnosticPage(model)];
    return Scaffold(
      appBar: AppBar(title: const Text('SSM2 TELEMETRY', style: TextStyle(fontSize: 17, letterSpacing: 2)),
        actions: [Icon(Icons.circle, size: 10, color: live ? Colors.greenAccent : muted), const SizedBox(width: 16)]),
      body: SafeArea(child: Column(children: [
        if (model.busy) const LinearProgressIndicator(minHeight: 2),
        Padding(padding: const EdgeInsets.fromLTRB(16, 4, 16, 8), child: Align(alignment: Alignment.centerLeft,
          child: Text(model.message, maxLines: 3, overflow: TextOverflow.ellipsis,
            style: const TextStyle(color: muted, fontSize: 12)))),
        Expanded(child: pages[tab]),
      ])),
      bottomNavigationBar: NavigationBar(selectedIndex: tab, labelBehavior: NavigationDestinationLabelBehavior.onlyShowSelected,
        onDestinationSelected: (i) => setState(() => tab = i), destinations: const [
          NavigationDestination(icon: Icon(Icons.bluetooth), label: 'Адаптер'),
          NavigationDestination(icon: Icon(Icons.speed), label: 'Дашборд'),
          NavigationDestination(icon: Icon(Icons.tune), label: 'PID'),
          NavigationDestination(icon: Icon(Icons.fiber_manual_record_outlined), label: 'CSV'),
          NavigationDestination(icon: Icon(Icons.show_chart), label: 'График'),
          NavigationDestination(icon: Icon(Icons.insights), label: 'Анализ'),
          NavigationDestination(icon: Icon(Icons.terminal), label: 'Диагн.'),
        ]),
    );
  });
}

Widget section(String title, List<Widget> children) => Padding(
  padding: const EdgeInsets.fromLTRB(16, 16, 16, 10), child: Column(crossAxisAlignment: CrossAxisAlignment.start,
    children: [Text(title, style: const TextStyle(fontSize: 15, fontWeight: FontWeight.bold)),
      const SizedBox(height: 12), ...children]));
Widget detail(String key, String value) => Padding(padding: const EdgeInsets.symmetric(vertical: 4),
  child: Row(crossAxisAlignment: CrossAxisAlignment.start, children: [
    Expanded(flex: 2, child: Text(key, style: const TextStyle(color: muted, fontSize: 12))),
    const SizedBox(width: 10), Expanded(flex: 3, child: Text(value, style: const TextStyle(fontSize: 12))),
  ]));
String ageText(int? age) => age == null ? 'нет данных' : '${(age / 1000).toStringAsFixed(1)} с';

class AdapterPage extends StatelessWidget {
  const AdapterPage(this.model, {super.key});
  final AppModel model;
  @override
  Widget build(BuildContext context) => ListView(children: [
    section('Bluetooth SPP', [
      Text(model.elm.transport.name, style: const TextStyle(color: cyan, fontSize: 12)),
      const SizedBox(height: 10),
      const Text('Только CAN 11 bit / 500 kbit. Зажигание включено, автомобиль стоит.'),
      const SizedBox(height: 10),
      Wrap(spacing: 8, children: [
        OutlinedButton.icon(onPressed: model.busy ? null : model.refreshDevices,
          icon: const Icon(Icons.refresh), label: const Text('Сопряженные')),
        TextButton(onPressed: () => model.perform(() async {
          await const MethodChannel('ssm2/system').invokeMethod<void>('bluetoothSettings');
        }), child: const Text('Настройки Android')),
      ]),
      if (model.devices.isEmpty) const Padding(padding: EdgeInsets.all(12), child: Text('Обновите список устройств')),
      for (final device in model.devices) ListTile(
        contentPadding: EdgeInsets.zero, leading: const Icon(Icons.bluetooth, color: cyan),
        title: Text(device.name.isEmpty ? 'Без имени' : device.name), subtitle: Text(device.address),
        trailing: model.selected == device.address ? const Icon(Icons.check, color: cyan) : null,
        onTap: model.busy ? null : () { model.selected = device.address; model.changed(); }),
      Wrap(spacing: 8, children: [
        FilledButton(onPressed: model.busy || model.selected == null ? null : model.connect,
          child: Text(model.elm.ready ? 'Переподключить' : 'Подключить')),
        OutlinedButton(onPressed: model.busy ? null : model.disconnect, child: const Text('Отключить')),
      ]),
    ]),
    section('Состояние', [
      detail('Адаптер', model.elm.identity.isEmpty ? 'не опрошен' : model.elm.identity),
      detail('ATRV', model.elm.voltage.isEmpty ? 'не опрошен' : model.elm.voltage),
      detail('SPP', model.elm.transport.connected ? 'открыт' : 'закрыт'),
      detail('Синхронизация', model.elm.ready ? 'готово' : 'нет'),
      const Text('Открытый SPP не доказывает связь с ECU. Нужен полный ответ E8 на запрос A8.',
        style: TextStyle(color: muted, height: 1.5)),
    ]),
  ]);
}

class DashboardPage extends StatelessWidget {
  const DashboardPage(this.model, {super.key});
  final AppModel model;
  @override
  Widget build(BuildContext context) {
    final engine = model.engine;
    final pids = engine.active;
    final fuel = estimateFuel(
      maf: engine.latest['MAF']?.value,
      afr: engine.latest['AFR']?.value,
      speed: engine.latest['SPEED']?.value,
      rpm: engine.latest['RPM']?.value,
      pedal: engine.latest['PEDAL']?.value,
    );
    return Column(children: [
      Padding(padding: const EdgeInsets.symmetric(horizontal: 16), child: Row(children: [
        Expanded(child: Text('A8 / ${pids.length} PID\n${engine.pidReadsPerSecond.toStringAsFixed(1)} PID-обновл./с',
          style: const TextStyle(fontSize: 12, color: muted))),
        FilledButton.tonalIcon(onPressed: model.busy || !model.elm.ready ? null : model.togglePolling,
          icon: Icon(engine.running ? Icons.pause : Icons.play_arrow), label: Text(engine.running ? 'Пауза' : 'Старт')),
      ])),
      FuelCard(fuel: fuel, mafStale: engine.stale('MAF')),
      if (pids.isEmpty) const Expanded(child: Center(child: Text('Выберите параметры во вкладке PID')))
      else Expanded(child: GridView.builder(padding: const EdgeInsets.all(12), itemCount: pids.length,
        gridDelegate: const SliverGridDelegateWithMaxCrossAxisExtent(maxCrossAxisExtent: 270, mainAxisExtent: 188,
          mainAxisSpacing: 10, crossAxisSpacing: 10), itemBuilder: (context, i) {
          final p = pids[i];
          final sample = engine.latest[p.id];
          // Ответ E8 FF не показываем как измерение: на скриншотах это давало
          // «детонация 63.5» и «передача 256».
          final unsupported = sample?.allOnes ?? false;
          final value = unsupported ? null : sample?.value;
          final outdated = engine.stale(p.id);
          final failed = engine.attempts[p.id]?.error.isNotEmpty ?? false;
          final tone = outdated || failed || unsupported ? muted : cyan;
          return Material(color: const Color(0xFF101A2B), borderRadius: BorderRadius.circular(14),
            child: InkWell(borderRadius: BorderRadius.circular(14), onTap: () => showModalBottomSheet<void>(context: context,
              isScrollControlled: true, builder: (context) => SafeArea(child: SingleChildScrollView(child: section(p.id, [
                Text(p.desc), detail('Адреса', p.addresses.map((a) => '0x${hexAddress(a)}').join(', ')),
                detail('Формула', p.formulaText), detail('Сырые байты', engine.latest[p.id]?.raw.map(hex2).join(' ') ?? 'нет'),
                detail('Возраст', ageText(engine.ageMs(p.id))),
                detail('Частота этого PID', '${engine.frequency(p.id).toStringAsFixed(2)} Hz'),
                detail('Окно чтения', '${engine.latest[p.id]?.readMs ?? 0} ms'),
                detail('Последняя ошибка', engine.attempts[p.id]?.error ?? ''),
                if (p.bytesCount > 1) const Text('Байты прочитаны последовательно, не атомарным снимком ECU.'),
              ])))), child: Padding(padding: const EdgeInsets.all(14), child: Column(crossAxisAlignment: CrossAxisAlignment.start,
                children: [
                  Row(children: [Expanded(child: Text(p.id, style: const TextStyle(fontSize: 13, fontWeight: FontWeight.bold))),
                    if (outdated && value != null) const Icon(Icons.schedule, size: 14, color: muted)]),
                  Text('${p.unit} / 0x${hexAddress(p.address)}', style: const TextStyle(fontSize: 10, color: muted)),
                  const Spacer(),
                  SizedBox(height: 46, width: double.infinity, child: FittedBox(fit: BoxFit.scaleDown,
                    alignment: Alignment.centerLeft, child: Text(value?.toStringAsFixed(p.digits) ?? '--',
                      style: TextStyle(color: tone, fontSize: 38, fontWeight: FontWeight.w700)))),
                  const SizedBox(height: 10),
                  LinearProgressIndicator(value: value == null ? 0 :
                    ((value - p.minValue) / (p.maxValue - p.minValue)).clamp(0.0, 1.0).toDouble(), color: tone, minHeight: 3),
                  const SizedBox(height: 8),
                  Text(unsupported
                      ? '0xFF: адрес не поддерживается'
                      : failed
                          ? 'ошибка / ${ageText(engine.ageMs(p.id))}'
                          : ageText(engine.ageMs(p.id)),
                    style: TextStyle(
                      color: unsupported ? const Color(0xFFD4B57F) : muted, fontSize: 10)),
                ]))));
        })),
    ]);
  }
}

class FuelCard extends StatelessWidget {
  const FuelCard({super.key, required this.fuel, required this.mafStale});
  final FuelEstimate? fuel;
  final bool mafStale;

  @override
  Widget build(BuildContext context) {
    final f = fuel;
    return Container(
      margin: const EdgeInsets.fromLTRB(12, 10, 12, 0),
      padding: const EdgeInsets.all(14),
      decoration: BoxDecoration(
        color: const Color(0xFF101A2B),
        borderRadius: BorderRadius.circular(14),
        border: Border.all(color: const Color(0xFF24435A)),
      ),
      child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
        Row(children: [
          const Expanded(child: Text('МГНОВЕННЫЙ РАСХОД',
            style: TextStyle(fontSize: 11, letterSpacing: 1.2, color: muted))),
          Text(f == null ? 'нет MAF' : 'расчет по MAF',
            style: const TextStyle(fontSize: 10, color: muted)),
        ]),
        const SizedBox(height: 10),
        if (f == null)
          const Text('Включите PID MAF и запустите опрос.',
            style: TextStyle(fontSize: 12, color: muted))
        else ...[
          Row(crossAxisAlignment: CrossAxisAlignment.end, children: [
            Text(f.litresPerHour.toStringAsFixed(2),
              style: TextStyle(fontSize: 38, fontWeight: FontWeight.w700,
                color: mafStale ? muted : cyan, height: 1)),
            const Padding(padding: EdgeInsets.only(left: 8, bottom: 4),
              child: Text('л/ч', style: TextStyle(fontSize: 13, color: muted))),
            const Spacer(),
            Text(f.litresPer100km == null
                ? 'на месте'
                : '${f.litresPer100km!.toStringAsFixed(1)} л/100км',
              style: TextStyle(fontSize: 16, fontWeight: FontWeight.w600,
                color: mafStale ? muted : Colors.white)),
          ]),
          const SizedBox(height: 10),
          Text('AFR ${f.afrUsed.toStringAsFixed(1)}'
              '${f.afrMeasured ? ' (из ECU)' : ' (стехиометрия, PID AFR не выбран)'}'
              ' · плотность $kPetrolDensityGramsPerLitre г/л',
            style: const TextStyle(fontSize: 10, color: muted, height: 1.5)),
          if (f.speedKph != null && f.speedKph! < 5)
            const Text('Скорость ниже 5 км/ч: л/100км не считается.',
              style: TextStyle(fontSize: 10, color: muted)),
          if (f.possibleCutoff)
            const Text('Похоже на отсечку подачи: расчет по MAF завышен.',
              style: TextStyle(fontSize: 10, color: Color(0xFFD4B57F))),
          if (mafStale)
            const Text('Данные MAF устарели: показан последний расчет.',
              style: TextStyle(fontSize: 10, color: Color(0xFFD4B57F))),
        ],
      ]),
    );
  }
}

class AnalyzerPage extends StatefulWidget {
  const AnalyzerPage(this.model, {super.key});
  final AppModel model;
  @override
  State<AnalyzerPage> createState() => _AnalyzerPageState();
}

class _AnalyzerPageState extends State<AnalyzerPage> {
  LogAnalysis? report;

  Color _levelColor(FindingLevel level) => switch (level) {
    FindingLevel.critical => const Color(0xFFE08A7A),
    FindingLevel.warning => const Color(0xFFD4B57F),
    FindingLevel.info => muted,
  };
  String _levelName(FindingLevel level) => switch (level) {
    FindingLevel.critical => 'КРИТИЧНО',
    FindingLevel.warning => 'ВНИМАНИЕ',
    FindingLevel.info => 'ИНФО',
  };

  @override
  Widget build(BuildContext context) {
    final engine = widget.model.engine;
    final result = report;
    return ListView(children: [
      section('Анализ журнала', [
        const Text('Правила с фиксированными порогами: детонация, бедная смесь под '
          'наддувом, топливные коррекции, перегрев, напряжение, отклонение наддува, '
          'ответы 0xFF, замершие каналы и скорость обновления.',
          style: TextStyle(color: muted, height: 1.6)),
        const SizedBox(height: 6),
        const Text('Это не машинное обучение и не диагноз. Каждый вывод содержит '
          'измеренные числа, которые можно проверить вручную.',
          style: TextStyle(color: muted, fontSize: 11, height: 1.6)),
        const SizedBox(height: 12),
        Wrap(spacing: 8, children: [
          FilledButton.icon(
            onPressed: engine.history.isEmpty ? null : () => setState(() {
              report = analyzeLog(engine.history, attempts: engine.attempts);
            }),
            icon: const Icon(Icons.insights, size: 18),
            label: const Text('Проанализировать')),
          if (result != null)
            OutlinedButton(onPressed: () => setState(() => report = null),
              child: const Text('Очистить')),
        ]),
        if (engine.history.isEmpty)
          const Padding(padding: EdgeInsets.only(top: 12),
            child: Text('Журнал пуст. Запустите опрос на вкладке дашборда.',
              style: TextStyle(color: muted))),
      ]),
      if (result != null) ...[
        section('Итог', [
          detail('Значений в памяти', '${result.sampleCount}'),
          detail('Длительность', '${result.spanSeconds.toStringAsFixed(1)} с'),
          detail('Каналов с данными', '${result.channels}'),
          detail('Критично / внимание / инфо',
            '${result.count(FindingLevel.critical)} / '
            '${result.count(FindingLevel.warning)} / '
            '${result.count(FindingLevel.info)}'),
        ]),
        if (result.findings.isEmpty)
          section('Замечаний нет', [
            const Text('Ни одно правило не сработало на текущих данных. '
              'Это не гарантия исправности: проверяются только перечисленные пороги.',
              style: TextStyle(color: muted, height: 1.6)),
          ])
        else
          ...result.findings.map((f) => Padding(
            padding: const EdgeInsets.fromLTRB(16, 0, 16, 12),
            child: Container(
              padding: const EdgeInsets.all(14),
              decoration: BoxDecoration(
                color: const Color(0xFF101A2B),
                borderRadius: BorderRadius.circular(12),
                border: Border(left: BorderSide(color: _levelColor(f.level), width: 3),
                  top: const BorderSide(color: Color(0xFF1E2B3A)),
                  right: const BorderSide(color: Color(0xFF1E2B3A)),
                  bottom: const BorderSide(color: Color(0xFF1E2B3A))),
              ),
              child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
                Text(_levelName(f.level), style: TextStyle(fontSize: 9,
                  letterSpacing: 1.2, color: _levelColor(f.level))),
                const SizedBox(height: 6),
                Text(f.title, style: const TextStyle(fontSize: 14,
                  fontWeight: FontWeight.w600)),
                const SizedBox(height: 8),
                Text(f.detail, style: const TextStyle(fontSize: 12, height: 1.6)),
                const SizedBox(height: 8),
                Text(f.evidence, style: const TextStyle(fontSize: 11,
                  color: muted, height: 1.6, fontFamily: 'monospace')),
                const SizedBox(height: 8),
                Text(f.advice, style: const TextStyle(fontSize: 12,
                  color: Color(0xFFB6C7D4), height: 1.6)),
              ]),
            ))),
      ],
    ]);
  }
}

class PidPage extends StatefulWidget {
  const PidPage(this.model, {super.key});
  final AppModel model;
  @override
  State<PidPage> createState() => _PidPageState();
}
class _PidPageState extends State<PidPage> {
  String search = '';
  @override
  Widget build(BuildContext context) {
    final m = widget.model;
    return ListView(children: [section('Библиотека / 28 PID', [
      const Text('Сначала 8 базовых. Больше параметров означает ниже частоту каждого PID.', style: TextStyle(color: muted)),
      Wrap(spacing: 8, children: [
        TextButton(onPressed: m.busy ? null : () => m.preset(false), child: const Text('8 базовых')),
        TextButton(onPressed: m.busy ? null : () => m.preset(true), child: const Text('Все 20 обычных')),
        TextButton(onPressed: m.busy ? null : () => _settings(context), child: const Text('ROM / Float32')),
      ]),
      TextField(decoration: const InputDecoration(labelText: 'Поиск PID или адреса', prefixIcon: Icon(Icons.search)),
        onChanged: (value) => setState(() => search = value.toUpperCase())),
      const SizedBox(height: 8),
      for (final p in SubaruPidLibrary.all.where((p) => '${p.id} ${p.desc} 0x${hexAddress(p.address)}'.toUpperCase().contains(search.trim())))
        CheckboxListTile(contentPadding: EdgeInsets.zero, controlAffinity: ListTileControlAffinity.leading,
          title: Text('${p.id}${p.extended ? ' *' : ''} / ${p.unit}', style: const TextStyle(fontSize: 13)),
          subtitle: Text('0x${hexAddress(p.address)} / ${p.bytesCount} B / P${p.priority}\n${p.formulaText}',
            style: const TextStyle(fontSize: 10, color: muted)), value: m.engine.enabled.contains(p.id),
          onChanged: m.busy || (p.extended && !m.engine.extendedConfirmed) ? null :
            (value) => m.selectPid(p.id, value ?? false)),
      const Text('* Extended зависит от ROM. Подтверждение пользователя не является автоматической проверкой адресов.',
        style: TextStyle(color: Colors.amber, fontSize: 12)),
    ])]);
  }
  Future<void> _settings(BuildContext context) async {
    final m = widget.model;
    final controller = TextEditingController(text: m.engine.romId);
    var confirm = m.engine.extendedConfirmed;
    var little = m.engine.endian == Endian.little;
    final accepted = await showDialog<bool>(context: context, builder: (context) => StatefulBuilder(builder: (context, change) =>
      AlertDialog(title: const Text('Extended / ROM'), content: SingleChildScrollView(child: Column(mainAxisSize: MainAxisSize.min,
        children: [
          const Text('Введите ROM ID из вашего def-файла. Адреса 0xFF... не универсальны. Endian не угадывается.'),
          TextField(controller: controller, decoration: const InputDecoration(labelText: 'ROM ID')),
          CheckboxListTile(title: const Text('Адреса сверены с моим ROM'), value: confirm,
            onChanged: (value) => change(() => confirm = value ?? false)),
          SwitchListTile(title: const Text('Float32 little-endian'), subtitle: const Text('Выключено = big-endian'), value: little,
            onChanged: (value) => change(() => little = value)),
        ])), actions: [
          TextButton(onPressed: () => Navigator.pop(context, false), child: const Text('Отмена')),
          FilledButton(onPressed: () => Navigator.pop(context, true), child: const Text('Сохранить')),
        ])));
    final rom = controller.text;
    controller.dispose();
    if (accepted == true) await m.configureExtended(confirm, rom, little ? Endian.little : Endian.big);
  }
}

class LoggerPage extends StatelessWidget {
  const LoggerPage(this.model, {super.key});
  final AppModel model;
  @override
  Widget build(BuildContext context) => ListView(children: [section('Потоковый CSV', [
    Text('${model.logger.count}', style: const TextStyle(fontSize: 54, fontWeight: FontWeight.bold, color: cyan)),
    Text(model.logger.active ? 'Идет запись' : 'Запись остановлена'),
    const SizedBox(height: 18),
    Wrap(spacing: 8, children: [
      FilledButton.icon(onPressed: model.busy ? null : model.toggleLog,
        icon: Icon(model.logger.active ? Icons.stop : Icons.fiber_manual_record),
        label: Text(model.logger.active ? 'Стоп' : 'Записать')),
      OutlinedButton.icon(onPressed: model.busy || model.logger.count == 0 ? null : model.exportCsv,
        icon: const Icon(Icons.share), label: const Text('Экспорт CSV')),
    ]),
    const SizedBox(height: 16),
    const Text('Одна строка на попытку чтения PID: время, значение, сырые байты, адрес, длительность, статус. '
      'При ошибке значение пустое. Старые значения не записываются как свежие.', style: TextStyle(color: muted, height: 1.6)),
    if (model.logger.error.isNotEmpty) Text(model.logger.error, style: const TextStyle(color: Colors.amber)),
    if (model.logger.file != null) SelectableText(model.logger.file!.path, style: const TextStyle(fontSize: 11)),
    const SizedBox(height: 16),
    const Text('В фоне опрос и запись останавливаются. Для длительной записи оставьте приложение открытым.'),
  ])]);
}

class GraphPage extends StatelessWidget {
  const GraphPage(this.model, {super.key});
  final AppModel model;
  @override
  Widget build(BuildContext context) {
    final pid = SubaruPidLibrary.byId(model.chartId);
    final samples = List<PidSample>.of(model.engine.history[pid.id] ?? []);
    return ListView(children: [section('График / реальные временные метки', [
      DropdownButton<String>(value: model.chartId, isExpanded: true,
        items: SubaruPidLibrary.all.map((p) => DropdownMenuItem(value: p.id, child: Text('${p.id} / ${p.unit}'))).toList(),
        onChanged: (value) { if (value != null) { model.chartId = value; model.changed(); } }),
      const SizedBox(height: 18),
      SizedBox(height: 270, child: samples.length < 2 ? const Center(child: Text('Нужны хотя бы два свежих значения')) :
        CustomPaint(painter: TelemetryPainter(samples, pid.digits), size: Size.infinite)),
      const SizedBox(height: 12),
      detail('Частота этого PID', '${model.engine.frequency(pid.id).toStringAsFixed(2)} Hz'),
      detail('Возраст', ageText(model.engine.ageMs(pid.id))),
      detail('Точек', '${samples.length} / 300'),
      const Text('Разрывы более 3 секунд не соединяются. Последнее значение не копируется в график при ошибках.',
        style: TextStyle(color: muted, fontSize: 12)),
    ])]);
  }
}
class TelemetryPainter extends CustomPainter {
  TelemetryPainter(this.samples, this.digits);
  final List<PidSample> samples;
  final int digits;
  void label(Canvas canvas, String text, Offset point) {
    final painter = TextPainter(text: TextSpan(text: text, style: const TextStyle(color: muted, fontSize: 10)),
      textDirection: TextDirection.ltr)..layout();
    painter.paint(canvas, point);
  }
  @override
  void paint(Canvas canvas, Size size) {
    final values = samples.map((s) => s.value!).toList();
    final minimum = values.reduce(math.min), maximum = values.reduce(math.max);
    final pad = math.max((maximum - minimum) * 0.1, 0.5);
    final lo = minimum - pad, hi = maximum + pad;
    final first = samples.first.time.millisecondsSinceEpoch;
    final span = math.max(1, samples.last.time.millisecondsSinceEpoch - first);
    final width = math.max(1.0, size.width - 65), height = size.height - 38;
    final grid = Paint()..color = const Color(0xFF233045)..strokeWidth = 1;
    for (var i = 0; i <= 4; i++) {
      final y = 8 + height * i / 4;
      canvas.drawLine(Offset(52, y), Offset(size.width - 8, y), grid);
      label(canvas, (hi - (hi - lo) * i / 4).toStringAsFixed(digits > 1 ? 1 : digits), Offset(0, y - 5));
    }
    final path = Path();
    for (var i = 0; i < samples.length; i++) {
      final sample = samples[i];
      final x = 52 + width * (sample.time.millisecondsSinceEpoch - first) / span;
      final y = 8 + height * (hi - sample.value!) / (hi - lo);
      canvas.drawCircle(Offset(x, y), 2, Paint()..color = cyan);
      if (i == 0 || sample.time.difference(samples[i - 1].time).inMilliseconds > 3000) { path.moveTo(x, y); }
      else { path.lineTo(x, y); }
    }
    canvas.drawPath(path, Paint()..color = cyan..strokeWidth = 2..style = PaintingStyle.stroke);
    label(canvas, '0 s', Offset(52, height + 20));
    label(canvas, '${(span / 1000).toStringAsFixed(1)} s', Offset(size.width - 55, height + 20));
  }
  @override
  bool shouldRepaint(covariant TelemetryPainter oldDelegate) => true;
}

class DiagnosticPage extends StatefulWidget {
  const DiagnosticPage(this.model, {super.key});
  final AppModel model;
  @override
  State<DiagnosticPage> createState() => _DiagnosticPageState();
}
class _DiagnosticPageState extends State<DiagnosticPage> {
  final command = TextEditingController(text: 'ATI');
  final address = TextEditingController(text: '000008');
  final count = TextEditingController(text: '1');
  @override
  void dispose() { command.dispose(); address.dispose(); count.dispose(); super.dispose(); }
  @override
  Widget build(BuildContext context) {
    final m = widget.model;
    final e = m.engine;
    return ListView(children: [
      section('Качество и синхронизация', [
        detail('Режим', 'A8 00 / ATH0 / CAF1 / 7E0-7E8'),
        detail('TX попыток / WRITE_OK', '${m.elm.tx} / ${m.elm.writeAccepted}'), detail('RX байт', '${m.elm.rxBytes}'),
        detail('Приглашений / таймаутов', '${m.elm.prompts} / ${m.elm.timeouts}'),
        detail('Полных PID / ошибок', '${e.goodCount} / ${e.failedCount}'),
        detail('Качество PID-чтений', '${e.quality.toStringAsFixed(1)} %'),
        detail('Состояние опроса', e.message),
        const Text('NO DATA или неполный HEX не доказывают поломку адаптера. Проверяйте TX/RX, команду, '
          'режим ECU и карту. После таймаута обязательно переподключение.', style: TextStyle(color: Colors.amber, fontSize: 12)),
      ]),
      section('Терминал / только чтение', [
        TextField(controller: command, autocorrect: false, enableSuggestions: false,
          decoration: const InputDecoration(labelText: 'ATI / ATRV / 010C / A8 00 00 00 08')),
        const SizedBox(height: 8),
        FilledButton(onPressed: m.busy || !m.elm.ready ? null : () => m.sendDiagnostic(command.text), child: const Text('Отправить')),
        SelectableText(m.terminal.isEmpty ? 'Команд еще нет' : m.terminal,
          style: const TextStyle(fontFamily: 'monospace', fontSize: 12)),
      ]),
      section('Сканер адресов / A8', [
        Row(children: [Expanded(child: TextField(controller: address, decoration: const InputDecoration(labelText: 'Адрес HEX'))),
          const SizedBox(width: 12), SizedBox(width: 90, child: TextField(controller: count,
            keyboardType: TextInputType.number, decoration: const InputDecoration(labelText: '1..32 байт')))]),
        const SizedBox(height: 8),
        OutlinedButton(onPressed: m.busy || !m.elm.ready ? null : () => m.scan(address.text, count.text), child: const Text('Прочитать')),
        SelectableText(m.scanner.isEmpty ? 'Нет дампа' : m.scanner, style: const TextStyle(fontFamily: 'monospace', fontSize: 12)),
      ]),
      section('TX/RX / последние 400 событий', [
        OutlinedButton.icon(onPressed: m.busy ? null : m.exportTrace, icon: const Icon(Icons.share), label: const Text('Экспорт журнала')),
        const Text(r'CR показан как \r, LF как \n. Это не байты полезной нагрузки.', style: TextStyle(fontSize: 11, color: muted)),
        const SizedBox(height: 8),
        SelectableText(m.elm.trace.toList().reversed.take(100).join('\n'),
          style: const TextStyle(fontFamily: 'monospace', fontSize: 10, height: 1.5)),
      ]),
    ]);
  }
}
'''

FILES["test/protocol_test.dart"] = r'''
import 'dart:async';
import 'dart:typed_data';
import 'package:flutter_test/flutter_test.dart';
import 'package:subaru_ssm2/analyzer.dart';
import 'package:subaru_ssm2/bt_transport.dart';
import 'package:subaru_ssm2/derived.dart';
import 'package:subaru_ssm2/elm.dart';
import 'package:subaru_ssm2/engine.dart';
import 'package:subaru_ssm2/pids.dart';
import 'package:subaru_ssm2/protocol.dart';

// Synthetic fixtures. These are not measurements from the user's vehicle.
class FakeTransport implements BtTransport {
  bool _connected = false, drop = false;
  final received = StreamController<Uint8List>.broadcast(sync: true);
  final states = StreamController<bool>.broadcast(sync: true);
  final writes = <String>[];
  @override
  String get name => 'Synthetic test transport';
  @override
  bool get connected => _connected;
  @override
  Stream<Uint8List> get data => received.stream;
  @override
  Stream<bool> get status => states.stream;
  @override
  Future<List<BtDevice>> paired() async => [const BtDevice('Test', '00:00:00:00:00:00')];
  @override
  Future<void> connect(String address) async { _connected = true; states.add(true); }
  @override
  Future<void> disconnect() async { _connected = false; states.add(false); }
  void emit(String text) => received.add(Uint8List.fromList(text.codeUnits));
  @override
  Future<void> write(String ascii) async {
    if (!_connected) throw StateError('Not connected');
    writes.add(ascii);
    if (drop) return;
    final cmd = ascii.trim();
    final reply = cmd == 'ATI' || cmd == 'ATZ' ? 'ELM327 TEST\r>' : cmd == 'ATRV' ? '13.5V\r>' :
      cmd.startsWith('A8') ? 'E8 ${cmd.endsWith('0E') ? '0D' : 'E6'}\r>' : 'OK\r>';
    emit(reply.substring(0, 1));
    await Future<void>.delayed(const Duration(milliseconds: 1));
    emit(reply.substring(1));
  }
  @override
  Future<void> dispose() async { await received.close(); await states.close(); }
}

void main() {
  const command = 'A8 00 00 00 08';
  group('Strict ATH0 / CAF1 parser', () {
    test('one byte, CR, LF and split-independent assembled text', () {
      expect(parseAddressReply('E8 6D\r\n>', command), 109);
      expect(parseAddressReply('E86D\r\r>', command), 109);
      expect(parseAddressReply('$command\rE8 6D\r>', command), 109);
      expect(parseAddressReply('SEARCHING...\rE86D\r>', command), 109);
      expect(parseAddressReply('SEARCHING...E86D\r>', command), 109);
    });
    test('ordinary hex bytes are not translated to words', () {
      expect(parseAddressReply('E8 4E>', command), 0x4E);
      expect(parseAddressReply('E8 4F>', command), 0x4F);
      expect(parseAddressReply('E8 E8>', command), 0xE8);
      expect(parseAddressReply('E8 7F>', command), 0x7F);
    });
    for (final raw in ['>', 'E86D', '1D1\r>', 'DDA\r>', 'NO DATA\r>', '?\r>',
      '7F A8 11\r>', 'E8 01 02>', '7E8 02 E8 6D>', 'E8 6D\rE8 6D>',
      'E86D>junk', 'garbage E86D>', '02 E8 6D>']) {
      test('reject ${visibleText(raw)}', () => expect(() => parseAddressReply(raw, command), throwsA(isA<ReplyError>())));
    }
    test('only single-address A8, full 24-bit address', () {
      expect(readAddressCommand(8), command);
      expect(readAddressCommand(0xFF2538), 'A8 00 FF 25 38');
      expect(readAddressCommand(0xFFFFFF), 'A8 00 FF FF FF');
      expect(() => readAddressCommand(0x1000000), throwsRangeError);
      expect(() => readAddressCommand(-1), throwsRangeError);
    });
    test('only read diagnostics; no hidden configuration changes', () {
      expect(allowedDiagnostic('ATI'), isTrue);
      expect(allowedDiagnostic('010C'), isTrue);
      expect(allowedDiagnostic(command), isTrue);
      for (final cmd in ['ATZ', 'ATCAF0', 'ATH1', 'ATSP5', '04', 'B8 00 00 00 01', 'ATI\rATZ']) {
        expect(allowedDiagnostic(cmd), isFalse);
      }
    });
  });
  group('All 28 PID', () {
    test('identity, addresses and sizes', () {
      final all = SubaruPidLibrary.all;
      expect(all.length, 28); expect(all.map((p) => p.id).toSet().length, 28);
      expect(all.where((p) => p.extended).length, 8);
      expect(SubaruPidLibrary.byId('RPM').addresses, [0xE, 0xF]);
      expect(SubaruPidLibrary.byId('TIMING').address, 0x11);
      expect(SubaruPidLibrary.byId('MAP_ABS').bytesCount, 1);
      expect(SubaruPidLibrary.byId('IAM').addresses, [0xFF2538, 0xFF2539, 0xFF253A, 0xFF253B]);
      for (final pid in all) {
        expect(pid.address + pid.bytesCount - 1, lessThanOrEqualTo(0xFFFFFF));
        expect(pid.decode([]), isNull);
        expect(pid.decode(List<int>.filled(pid.bytesCount + 1, 0)), isNull);
        expect(pid.decode(List<int>.filled(pid.bytesCount, 256)), isNull);
      }
    });
    test('integer formulas, synthetic vectors', () {
      final vectors = <String, (List<int>, double)>{
        'LOAD': ([255], 100), 'ECT': ([109], 69), 'STFT': ([128], 0), 'LTFT': ([64], -50),
        'MAP_ABS': ([33], 33 * 37 / 255 / 14.50377), 'RPM': ([13, 230], 889.5),
        'SPEED': ([90], 90), 'TIMING': ([137], 4.5), 'IAT': ([80], 40),
        'MAF': ([1, 194], 4.5), 'TPS': ([255], 100), 'O2_F': ([0, 200], 1),
        'BATT': ([169], 13.52), 'KNOCK_ADV': ([120], -4),
        'BARO': ([100], 100 * 37 / 255 / 14.50377), 'MAP_REL': ([128], 0),
        'PEDAL': ([0], 0), 'WG_PRIM': ([255], 100), 'AFR': ([128], 14.7), 'GEAR': ([2], 3),
      };
      for (final entry in vectors.entries) {
        expect(SubaruPidLibrary.byId(entry.key).decode(entry.value.$1), closeTo(entry.value.$2, 0.000001), reason: entry.key);
      }
    });
    for (final pid in SubaruPidLibrary.all.where((p) => p.extended)) {
      test('${pid.id} float32, selectable endian, NaN/Infinity', () {
        expect(pid.decode([0x3F, 0x80, 0, 0]), closeTo(pid.floatFactor!, 0.0000001));
        expect(pid.decode([0, 0, 0x80, 0x3F], endian: Endian.little), closeTo(pid.floatFactor!, 0.0000001));
        expect(pid.decode([0xBF, 0x80, 0, 0]), closeTo(-pid.floatFactor!, 0.0000001));
        expect(pid.decode([0x7F, 0xC0, 0, 0]), isNull);
        expect(pid.decode([0x7F, 0x80, 0, 0]), isNull);
        expect(pid.decode([0, 0, 0, 0]), 0);
      });
    }
  });
  group('One physical command queue', () {
    test('lock recovers after exceptions', () async {
      final lock = AsyncLock(); final seen = <int>[];
      final first = lock.run(() async { seen.add(1); await Future<void>.delayed(const Duration(milliseconds: 2)); seen.add(2); });
      final second = lock.run(() async { seen.add(3); });
      await Future.wait([first, second]); expect(seen, [1, 2, 3]);
      await expectLater(lock.run(() async { throw StateError('test'); }), throwsStateError);
      expect(await lock.run(() async => 4), 4);
    });
    test('multi-byte PID cannot interleave with terminal', () async {
      final transport = FakeTransport(); final driver = ElmDriver(transport);
      await driver.initialize('test'); transport.writes.clear();
      final pid = driver.readPid(SubaruPidLibrary.byId('RPM'));
      final terminal = driver.diagnostic('ATI');
      expect((await pid).bytes, [13, 230]); await terminal;
      expect(transport.writes, ['A8 00 00 00 0E\r', 'A8 00 00 00 0F\r', 'ATI\r']);
      await driver.dispose();
    });
    test('fuel: MAF and AFR to litres, synthetic values', () {
      final idle = estimateFuel(maf: 4.5, afr: 14.7, speed: 0);
      expect(idle!.fuelGramsPerSecond, closeTo(4.5 / 14.7, 1e-9));
      expect(idle.litresPerHour, closeTo(4.5 / 14.7 * 3600 / 745, 1e-9));
      expect(idle.litresPer100km, isNull, reason: 'стоя л/100км не считается');
      expect(idle.afrMeasured, isTrue);
      final cruise = estimateFuel(maf: 9.0, afr: 14.7, speed: 90);
      expect(cruise!.litresPer100km, closeTo(cruise.litresPerHour / 90 * 100, 1e-9));
      final fallback = estimateFuel(maf: 4.5, afr: 999, speed: 0);
      expect(fallback!.afrUsed, kStoichAfr);
      expect(fallback.afrMeasured, isFalse);
      expect(estimateFuel(maf: null, afr: 14.7), isNull);
      expect(estimateFuel(maf: 0, afr: 14.7), isNull);
      expect(estimateFuel(maf: double.nan, afr: 14.7), isNull);
      expect(estimateFuel(maf: 6, afr: 14.7, speed: 60, rpm: 2500, pedal: 0)!
          .possibleCutoff, isTrue);
      expect(estimateFuel(maf: 6, afr: 14.7, speed: 60, rpm: 2500, pedal: 30)!
          .possibleCutoff, isFalse);
    });
    test('analyzer: rules fire only on their own evidence', () {
      final start = DateTime(2026, 1, 1, 12);
      PidSample make(String id, double? value, List<int> raw, int second) =>
          PidSample(SubaruPidLibrary.byId(id), start.add(Duration(seconds: second)),
              value, raw, 30, value == null ? 'INVALID' : '');
      final history = <String, List<PidSample>>{
        'FBKC': [for (var i = 0; i < 6; i++) make('FBKC', i == 3 ? -3.5 : 0, [0x3F, 0x80, 0, 0], i * 4)],
        'KNOCK_ADV': [for (var i = 0; i < 6; i++) make('KNOCK_ADV', 63.5, [0xFF], i * 4)],
        'ECT': [for (var i = 0; i < 6; i++) make('ECT', 107, [0x93], i * 4)],
        'STFT': [for (var i = 0; i < 6; i++) make('STFT', -12, [0x70], i * 4)],
        'LTFT': [for (var i = 0; i < 6; i++) make('LTFT', -11, [0x71], i * 4)],
      };
      final report = analyzeLog(history);
      String? find(String needle) {
        for (final f in report.findings) {
          if (f.title.contains(needle)) return f.title;
        }
        return null;
      }
      expect(find('FBKC: коррекция по детонации'), isNotNull);
      expect(find('KNOCK_ADV: ответ 0xFF'), isNotNull,
          reason: '0xFF должен объявляться неподдерживаемым, а не 63.5 градуса');
      expect(find('KNOCK_ADV: коррекция'), isNull,
          reason: 'мусор 0xFF не должен порождать вывод о детонации');
      expect(find('Температура ОЖ'), isNotNull);
      expect(find('Топливная коррекция'), isNotNull);
      expect(find('Бедная смесь'), isNull, reason: 'нет данных AFR и наддува');
      expect(report.findings.first.level, FindingLevel.critical);
      expect(report.sampleCount, 30);
      expect(analyzeLog(const <String, List<PidSample>>{}).findings
          .any((f) => f.title == 'Мало данных'), isTrue);
    });
    test('timeout stops queued TX; late bytes are never next response', () async {
      final transport = FakeTransport(); final driver = ElmDriver(transport);
      await driver.initialize('test'); transport.writes.clear(); transport.drop = true; driver.timeoutMs = 10;
      await expectLater(driver.readRange(8, 1), throwsA(isA<ReplyError>()));
      expect(driver.ready, isFalse); expect(driver.timeouts, 1);
      transport.emit('E8 6D\r>');
      await expectLater(driver.readRange(9, 1), throwsA(isA<ReplyError>()));
      expect(transport.writes.length, 1);
      transport.drop = false;
      await driver.initialize('test'); expect(driver.ready, isTrue);
      await driver.dispose();
    });
  });
}
'''

FILES["tool/prepare_bt.py"] = r'''
import hashlib
import json
from pathlib import Path
import re
import shutil
import sys
import tarfile
import urllib.request

APP = Path(__file__).resolve().parents[1]
PACKAGES = {
    "bluetooth_classic": ("0.0.4", "c92e10fb8f8114a19603c48b02ddfda448b41d6a9cf79eb192f82c6ed3bc0df5", "com.matteogassend.bluetooth_classic", "classic"),
    "flutter_bluetooth_serial": ("0.4.0", "85ae82c4099b2b1facdc54e75e1bcfa88dc7f719e55dc886bb0b648cb16636b1", "io.github.edufolly.flutterbluetoothserial", "serial"),
}


def replace_method(source, pattern, replacement, required=True):
    match = re.search(pattern, source)
    if match is None:
        if required:
            raise RuntimeError(f"Pinned plugin source does not match: {pattern}")
        return source
    start = source.index("{", match.start())
    depth, end = 1, start + 1
    while end < len(source) and depth:
        if source[end] == "{": depth += 1
        if source[end] == "}": depth -= 1
        end += 1
    if depth:
        raise RuntimeError("Unbalanced plugin method")
    return source[:match.start()] + replacement + source[end:]


def prepare(name):
    if name not in PACKAGES: raise ValueError(name)
    version, checksum, namespace, short = PACKAGES[name]
    cache = Path('/content/ssm2_downloads')
    cache.mkdir(exist_ok=True)
    archive = cache / f'{name}-{version}.tar.gz'
    if not archive.exists():
        temp = archive.with_suffix('.part')
        with urllib.request.urlopen(f'https://pub.dev/api/archives/{name}-{version}.tar.gz', timeout=120) as response, temp.open('wb') as out:
            shutil.copyfileobj(response, out)
        temp.replace(archive)
    if hashlib.sha256(archive.read_bytes()).hexdigest() != checksum:
        archive.unlink()
        raise RuntimeError('Bluetooth package SHA256 mismatch')
    vendor = APP / 'vendor' / name
    shutil.rmtree(vendor, ignore_errors=True)
    vendor.mkdir(parents=True)
    with tarfile.open(archive, 'r:gz') as tar:
        members = tar.getmembers()
        for member in members:
            target = (vendor / member.name).resolve()
            if not target.is_relative_to(vendor.resolve()) or not (member.isfile() or member.isdir()):
                raise RuntimeError('Unsafe package archive member')
        tar.extractall(vendor, members=members)
    manifest = vendor / 'android/src/main/AndroidManifest.xml'
    if manifest.exists():
        manifest.write_text(re.sub(r'\s+package="[^"]*"', '', manifest.read_text()), encoding='utf-8')
    kotlin = "id 'org.jetbrains.kotlin.android'" if short == 'classic' else ''
    kotlin_options = 'kotlinOptions { jvmTarget = "17" }' if short == 'classic' else ''
    build = f"""plugins {{
    id 'com.android.library'
    {kotlin}
}}
android {{
    namespace '{namespace}'
    compileSdk 36
    defaultConfig {{ minSdk 24 }}
    compileOptions {{
        sourceCompatibility JavaVersion.VERSION_17
        targetCompatibility JavaVersion.VERSION_17
    }}
    {kotlin_options}
}}
dependencies {{
    implementation 'androidx.core:core:1.13.1'
    implementation 'androidx.appcompat:appcompat:1.7.0'
}}
"""
    (vendor / 'android/build.gradle').write_text(build, encoding='utf-8')
    if short == 'classic':
        file = next(vendor.rglob('BluetoothClassicPlugin.kt'))
        source = file.read_text(encoding='utf-8')
        pattern = r'Handler\(Looper\.getMainLooper\(\)\)\.post\s*\{\s*publishBluetoothData\(ByteArray\(numBytes\)\s*\{\s*buffer\[it\]\s*\}\)\s*\}'
        replacement = """if (numBytes < 0) throw IOException("SPP EOF")
                if (numBytes == 0) continue
                val packet = buffer.copyOf(numBytes)
                Handler(Looper.getMainLooper()).post {
                    if (thread === this@ConnectedThread && readStream) publishBluetoothData(packet)
                }"""
        source, count = re.subn(pattern, replacement, source)
        if count != 1: raise RuntimeError('Classic RX patch did not match the pinned source')
        source = replace_method(source, r'private fun disconnect\(result:\s*Result\)\s*\{', """private fun disconnect(result: Result) {
        try {
            thread?.readStream = false
            socket?.close()
            thread = null
            socket = null
            device = null
            publishBluetoothStatus(0)
            result.success(true)
        } catch (e: IOException) {
            result.error("disconnect_error", e.message, null)
        }
    }""")
        source = replace_method(source, r'fun write\(bytes:\s*ByteArray\)\s*\{', """fun write(bytes: ByteArray) {
            outputStream.write(bytes)
            outputStream.flush()
        }""")
        source = replace_method(source, r'private fun write\(result:\s*Result,\s*message:\s*String\)\s*\{', """private fun write(result: Result, message: String) {
        try {
            val activeThread = thread ?: throw IOException("SPP is not connected")
            activeThread.write(message.toByteArray(Charsets.US_ASCII))
            result.success(true)
        } catch (e: IOException) {
            publishBluetoothStatus(0)
            result.error("write_error", e.message, null)
        }
    }""")
        source, count = re.subn(r'Handler\(looper\)\.post\s*\{\s*publishBluetoothStatus\(0\)\s*\}',
            'Handler(looper).post { if (thread === this@ConnectedThread) publishBluetoothStatus(0) }', source)
        if count != 1: raise RuntimeError('Classic generation guard did not match')
        source = source.replace('var readStream = true', '@Volatile var readStream = true')
        file.write_text(source, encoding='utf-8')
        print('[PATCH] A: immutable RX chunk, EOF, safe close, propagated write error')
    else:
        file = next(vendor.rglob('FlutterBluetoothSerialPlugin.java'))
        source = file.read_text(encoding='utf-8')
        source = re.sub(r'(?m)^\s*import io\.flutter\.plugin\.common\.PluginRegistry\.Registrar;\s*$', '', source)
        source = replace_method(source, r'public static void registerWith\s*\([^)]*\)\s*\{', '', required=False)
        if re.search(r'\bRegistrar\b', source):
            raise RuntimeError('Unhandled Flutter embedding v1 reference in serial plugin')
        source = replace_method(source, r'private void ensurePermissions\(EnsurePermissionsCallback callbacks\)\s*\{', """private void ensurePermissions(EnsurePermissionsCallback callbacks) {
        if (android.os.Build.VERSION.SDK_INT >= 31) {
            boolean granted = ContextCompat.checkSelfPermission(activity, Manifest.permission.BLUETOOTH_CONNECT) == PackageManager.PERMISSION_GRANTED
                && ContextCompat.checkSelfPermission(activity, Manifest.permission.BLUETOOTH_SCAN) == PackageManager.PERMISSION_GRANTED;
            callbacks.onResult(granted);
        } else {
            callbacks.onResult(ContextCompat.checkSelfPermission(activity, Manifest.permission.ACCESS_FINE_LOCATION) == PackageManager.PERMISSION_GRANTED);
        }
    }""")
        file.write_text(source, encoding='utf-8')
        connection = next(vendor.rglob('BluetoothConnection.java'))
        source = connection.read_text(encoding='utf-8')
        source, count = re.subn(r'bytes\s*=\s*input\.read\(buffer\);',
            'if (input == null) break;\n                    bytes = input.read(buffer);\n'
            '                    if (bytes < 0) break;\n                    if (bytes == 0) continue;', source)
        if count != 1: raise RuntimeError('Serial EOF patch did not match')
        source = replace_method(source, r'public void write\(byte\[\]\s+bytes\)\s*\{', """public void write(byte[] bytes) throws IOException {
            if (output == null) throw new IOException("SPP output unavailable");
            output.write(bytes);
            output.flush();
        }""")
        source = source.replace('private boolean requestedClosing = false;', 'private volatile boolean requestedClosing = false;')
        connection.write_text(source, encoding='utf-8')
        print('[PATCH] B: original namespace, embedding v2, Android 12+ permissions, EOF, write errors')
    # Keep both templates outside lib; the active file is always recreated.
    shutil.copyfile(APP / f'transport_templates/{short}.dart.txt', APP / 'lib/transport_selected.dart')
    pubspec = (APP / 'pubspec.template.yaml').read_text(encoding='utf-8')
    pubspec = pubspec.replace('__BLUETOOTH_DEPENDENCY__', f'  {name}:\n    path: vendor/{name}')
    (APP / 'pubspec.yaml').write_text(pubspec, encoding='utf-8')
    (APP / 'selected_transport.json').write_text(json.dumps({'package': name, 'version': version, 'patch': '0.5'}), encoding='utf-8')
    print(f'[OK] Selected {name} {version}; common pub-cache untouched')


if __name__ == '__main__':
    prepare(sys.argv[1])
'''

FILES["android/settings.gradle.kts"] = r'''
pluginManagement {
    val flutterSdkPath = run {
        val properties = java.util.Properties()
        file("local.properties").inputStream().use { properties.load(it) }
        requireNotNull(properties.getProperty("flutter.sdk")) { "flutter.sdk missing" }
    }
    includeBuild("$flutterSdkPath/packages/flutter_tools/gradle")
    repositories { google(); mavenCentral(); gradlePluginPortal() }
}
plugins {
    id("dev.flutter.flutter-plugin-loader") version "1.0.0"
    id("com.android.application") version "8.11.1" apply false
    id("com.android.library") version "8.11.1" apply false
    id("org.jetbrains.kotlin.android") version "2.2.20" apply false
}
include(":app")
'''

FILES["android/build.gradle.kts"] = r'''
allprojects {
    repositories { google(); mavenCentral() }
}
val newBuildDir = rootProject.layout.buildDirectory.dir("../../build").get()
rootProject.layout.buildDirectory.value(newBuildDir)
subprojects {
    project.layout.buildDirectory.value(newBuildDir.dir(project.name))
}
subprojects { project.evaluationDependsOn(":app") }
tasks.register<Delete>("clean") { delete(rootProject.layout.buildDirectory) }
'''

FILES["android/app/build.gradle.kts"] = r'''
plugins {
    id("com.android.application")
    id("org.jetbrains.kotlin.android")
    id("dev.flutter.flutter-gradle-plugin")
}
android {
    namespace = "com.subaru.ssm2_fixed"
    compileSdk = 36
    ndkVersion = "27.0.12077973"
    compileOptions {
        sourceCompatibility = JavaVersion.VERSION_17
        targetCompatibility = JavaVersion.VERSION_17
    }
    kotlinOptions { jvmTarget = "17" }
    defaultConfig {
        applicationId = "com.subaru.ssm2_fixed"
        minSdk = 24
        targetSdk = 35
        versionCode = flutter.versionCode
        versionName = flutter.versionName
    }
    buildTypes {
        release {
            // Personal sideload build, not a Play Store signing configuration.
            signingConfig = signingConfigs.getByName("debug")
            isMinifyEnabled = false
            isShrinkResources = false
        }
    }
}
flutter { source = "../.." }
'''

FILES["android/gradle.properties"] = r'''
org.gradle.jvmargs=-Xmx4g -XX:MaxMetaspaceSize=1g -XX:+HeapDumpOnOutOfMemoryError
org.gradle.workers.max=2
org.gradle.caching=true
android.useAndroidX=true
'''

FILES["android/gradle/wrapper/gradle-wrapper.properties"] = r'''
distributionBase=GRADLE_USER_HOME
distributionPath=wrapper/dists
zipStoreBase=GRADLE_USER_HOME
zipStorePath=wrapper/dists
distributionUrl=https\://services.gradle.org/distributions/gradle-8.14.3-bin.zip
networkTimeout=120000
validateDistributionUrl=true
'''

FILES["android/app/src/main/kotlin/com/subaru/ssm2_fixed/MainActivity.kt"] = r'''
package com.subaru.ssm2_fixed

import android.content.Intent
import android.os.Build
import android.provider.Settings
import io.flutter.embedding.android.FlutterActivity
import io.flutter.embedding.engine.FlutterEngine
import io.flutter.plugin.common.MethodChannel

class MainActivity : FlutterActivity() {
    override fun configureFlutterEngine(flutterEngine: FlutterEngine) {
        super.configureFlutterEngine(flutterEngine)
        MethodChannel(flutterEngine.dartExecutor.binaryMessenger, "ssm2/system")
            .setMethodCallHandler { call, result ->
                when (call.method) {
                    "sdkInt" -> result.success(Build.VERSION.SDK_INT)
                    "bluetoothSettings" -> {
                        startActivity(Intent(Settings.ACTION_BLUETOOTH_SETTINGS))
                        result.success(null)
                    }
                    else -> result.notImplemented()
                }
            }
    }
}
'''

FILES["android/app/src/main/AndroidManifest.xml"] = r'''
<manifest xmlns:android="http://schemas.android.com/apk/res/android">
    <uses-permission android:name="android.permission.BLUETOOTH" android:maxSdkVersion="30"/>
    <uses-permission android:name="android.permission.BLUETOOTH_ADMIN" android:maxSdkVersion="30"/>
    <uses-permission android:name="android.permission.ACCESS_FINE_LOCATION" android:maxSdkVersion="30"/>
    <uses-permission android:name="android.permission.ACCESS_COARSE_LOCATION" android:maxSdkVersion="30"/>
    <uses-permission android:name="android.permission.BLUETOOTH_SCAN" android:usesPermissionFlags="neverForLocation"/>
    <uses-permission android:name="android.permission.BLUETOOTH_CONNECT"/>
    <uses-feature android:name="android.hardware.bluetooth" android:required="true"/>
    <application android:label="SSM2 Fixed" android:name="${applicationName}" android:icon="@mipmap/ic_launcher">
        <activity android:name=".MainActivity" android:exported="true" android:launchMode="singleTop"
            android:theme="@style/LaunchTheme" android:hardwareAccelerated="true"
            android:configChanges="orientation|keyboardHidden|keyboard|screenSize|smallestScreenSize|locale|layoutDirection|fontScale|screenLayout|density|uiMode"
            android:windowSoftInputMode="adjustResize">
            <meta-data android:name="io.flutter.embedding.android.NormalTheme" android:resource="@style/NormalTheme"/>
            <intent-filter>
                <action android:name="android.intent.action.MAIN"/>
                <category android:name="android.intent.category.LAUNCHER"/>
            </intent-filter>
        </activity>
        <meta-data android:name="flutterEmbedding" android:value="2"/>
    </application>
    <queries>
        <intent><action android:name="android.intent.action.PROCESS_TEXT"/><data android:mimeType="text/plain"/></intent>
    </queries>
</manifest>
'''

FILES["pubspec.template.yaml"] = r'''
name: subaru_ssm2
description: "Read-only Subaru SSM2 CAN telemetry with 28 user-supplied PID definitions"
publish_to: 'none'
version: 0.6.0+6
environment:
  sdk: '>=3.3.0 <4.0.0'
dependencies:
  flutter:
    sdk: flutter
__BLUETOOTH_DEPENDENCY__
  permission_handler: 11.3.1
  path_provider: 2.1.5
  share_plus: 10.1.4
dev_dependencies:
  flutter_test:
    sdk: flutter
flutter:
  uses-material-design: true
'''

FILES["lib/samples.dart"] = r'''
// Одно измерение PID. Файл намеренно без зависимостей от Flutter:
// анализатор и расчеты можно проверять обычным `dart test`.

import 'pids.dart';

class PidSample {
  PidSample(this.pid, this.time, this.value, this.raw, this.readMs, this.error);
  final SubaruPidDef pid;
  final DateTime time;
  final double? value;
  final List<int> raw;
  final int readMs;
  final String error;
  bool get good => value != null && error.isEmpty;
  // Ответ вида E8 FF почти всегда означает «этот адрес не поддерживается ROM».
  // Без такой пометки получаются значения вроде 63.5 градуса или 327.68 В.
  bool get allOnes => raw.isNotEmpty && raw.every((b) => b == 0xFF);
}
'''

FILES["lib/derived.dart"] = r'''
// Мгновенный расход топлива. Это РАСЧЕТ по MAF, а не показание ECU:
// ECU из вашего списка PID не отдает расход напрямую.
//
//   масса топлива [г/с] = MAF [г/с] / AFR
//   объем [л/ч]        = г/с * 3600 / плотность [г/л]
//   [л/100км]          = [л/ч] / скорость [км/ч] * 100
//
// Плотность бензина принята 745 г/л (типично для 15 C). AFR берется из PID
// AFR, если он в разумных пределах; иначе используется стехиометрия 14.7.

const double kPetrolDensityGramsPerLitre = 745.0;
const double kStoichAfr = 14.7;

class FuelEstimate {
  const FuelEstimate({
    required this.fuelGramsPerSecond,
    required this.litresPerHour,
    required this.litresPer100km,
    required this.afrUsed,
    required this.afrMeasured,
    required this.speedKph,
    required this.possibleCutoff,
  });

  final double fuelGramsPerSecond;
  final double litresPerHour;
  final double? litresPer100km;
  final double afrUsed;
  final bool afrMeasured;
  final double? speedKph;

  /// Вероятная отсечка подачи топлива: расчет по MAF в этот момент завышен.
  final bool possibleCutoff;

  String get formula => 'MAF / AFR * 3600 / $kPetrolDensityGramsPerLitre';
}

FuelEstimate? estimateFuel({
  double? maf,
  double? afr,
  double? speed,
  double? rpm,
  double? pedal,
}) {
  if (maf == null || !maf.isFinite || maf <= 0) return null;
  final measured = afr != null && afr.isFinite && afr >= 8 && afr <= 25;
  final ratio = measured ? afr : kStoichAfr;
  final grams = maf / ratio;
  if (!grams.isFinite || grams < 0) return null;
  final litresPerHour = grams * 3600 / kPetrolDensityGramsPerLitre;
  double? per100;
  if (speed != null && speed.isFinite && speed >= 5) {
    per100 = litresPerHour / speed * 100;
  }
  final cutoff = pedal != null && pedal <= 1 &&
      rpm != null && rpm > 1500 &&
      speed != null && speed > 5;
  return FuelEstimate(
    fuelGramsPerSecond: grams,
    litresPerHour: litresPerHour,
    litresPer100km: per100,
    afrUsed: ratio,
    afrMeasured: measured,
    speedKph: speed,
    possibleCutoff: cutoff,
  );
}
'''

FILES["lib/analyzer.dart"] = r'''
// Анализатор журнала. Это прозрачные правила с фиксированными порогами,
// а не машинное обучение. Каждый вывод содержит измеренные числа,
// чтобы его можно было проверить вручную.

import 'pids.dart';
import 'protocol.dart';
import 'samples.dart';

enum FindingLevel { info, warning, critical }

class Finding {
  const Finding(this.level, this.title, this.detail, this.evidence, this.advice);
  final FindingLevel level;
  final String title, detail, evidence, advice;
}

class LogAnalysis {
  const LogAnalysis(this.findings, this.sampleCount, this.spanSeconds, this.channels);
  final List<Finding> findings;
  final int sampleCount, channels;
  final double spanSeconds;
  int count(FindingLevel level) => findings.where((f) => f.level == level).length;
}

double _avg(List<double> values) =>
    values.reduce((a, b) => a + b) / values.length;

String _time(DateTime value) =>
    '${value.hour.toString().padLeft(2, '0')}:'
    '${value.minute.toString().padLeft(2, '0')}:'
    '${value.second.toString().padLeft(2, '0')}';

PidSample? _nearest(List<PidSample> list, DateTime time, int skewMs) {
  PidSample? best;
  var bestDelta = skewMs;
  for (final sample in list) {
    final delta = (sample.time.difference(time).inMilliseconds).abs();
    if (delta <= bestDelta) {
      best = sample;
      bestDelta = delta;
    }
  }
  return best;
}

LogAnalysis analyzeLog(
  Map<String, List<PidSample>> history, {
  Map<String, PidSample> attempts = const <String, PidSample>{},
}) {
  final findings = <Finding>[];
  final clean = <String, List<PidSample>>{};
  var total = 0;
  DateTime? first, last;

  history.forEach((id, samples) {
    final usable = samples.where((s) => s.good && !s.allOnes).toList();
    if (usable.isNotEmpty) clean[id] = usable;
    for (final sample in samples) {
      total++;
      if (first == null || sample.time.isBefore(first!)) first = sample.time;
      if (last == null || sample.time.isAfter(last!)) last = sample.time;
    }
  });
  final span = first == null || last == null
      ? 0.0
      : last!.difference(first!).inMilliseconds / 1000;

  List<double> values(String id) =>
      (clean[id] ?? const <PidSample>[]).map((s) => s.value!).toList();
  bool has(String id) => (clean[id] ?? const <PidSample>[]).isNotEmpty;

  // 1. Достаточно ли данных для выводов.
  if (total < 20) {
    findings.add(Finding(FindingLevel.info, 'Мало данных',
        'Собрано $total значений за ${span.toStringAsFixed(1)} с. '
        'Пороговые правила включаются, но статистика может быть случайной.',
        'Для устойчивых выводов нужно хотя бы 60 секунд записи.',
        'Запустите опрос на прогретом двигателе и повторите анализ.'));
  }

  // 2. Ответы 0xFF: адрес почти наверняка не поддерживается этим ROM.
  history.forEach((id, samples) {
    final ones = samples.where((s) => s.allOnes).length;
    if (ones >= 3 && ones >= samples.length * 0.6) {
      final pid = SubaruPidLibrary.byId(id);
      findings.add(Finding(FindingLevel.warning, '$id: ответ 0xFF',
          'ECU вернул все байты 0xFF в $ones из ${samples.length} чтений. '
          'Это не измерение, а признак неподдерживаемого адреса.',
          'Адрес 0x${hexAddress(pid.address)}, ${pid.bytesCount} байт, '
          'формула ${pid.formulaText}.',
          'Отключите этот PID или сверьте адрес с def-файлом своего ROM.'));
    }
  });

  // 3. Замерший канал: значение не меняется, хотя запись длинная.
  clean.forEach((id, samples) {
    if (samples.length < 12 || span < 10) return;
    final list = samples.map((s) => s.value!).toList();
    if (list.every((v) => v == list.first)) {
      findings.add(Finding(FindingLevel.info, '$id: значение не меняется',
          'За ${samples.length} чтений значение осталось '
          '${list.first.toStringAsFixed(2)}.',
          'Первое чтение ${_time(samples.first.time)}, последнее '
          '${_time(samples.last.time)}.',
          'На холостом ходу это нормально. Если значение статично и в движении, '
          'проверьте адрес и формулу.'));
    }
  });

  // 4. Детонация: отрицательная коррекция опережения.
  for (final id in const <String>['FBKC', 'KNOCK_ADV', 'FKL']) {
    if (!has(id)) continue;
    final samples = clean[id]!;
    final events = samples.where((s) => s.value! <= -0.5).toList();
    if (events.isEmpty) continue;
    final worst = events.reduce((a, b) => a.value! < b.value! ? a : b);
    final rpm = _nearest(clean['RPM'] ?? const <PidSample>[], worst.time, 1500);
    final load = _nearest(clean['LOAD'] ?? const <PidSample>[], worst.time, 1500);
    findings.add(Finding(FindingLevel.critical, '$id: коррекция по детонации',
        'Зафиксировано ${events.length} событий с отрицательной коррекцией. '
        'Минимум ${worst.value!.toStringAsFixed(2)} градуса.',
        'Время ${_time(worst.time)}, сырые байты '
        '${worst.raw.map(hex2).join(' ')}'
        '${rpm == null ? '' : ', RPM ${rpm.value!.toStringAsFixed(0)}'}'
        '${load == null ? '' : ', нагрузка ${load.value!.toStringAsFixed(1)}'}.',
        'Проверьте качество топлива, температуру впуска и наддув. '
        'Повторите запись и сравните с этой.'));
  }

  // 5. Бедная смесь под наддувом.
  if (has('AFR') && has('MAP_REL')) {
    final risky = <PidSample>[];
    for (final boost in clean['MAP_REL']!) {
      if (boost.value! < 0.3) continue;
      final afr = _nearest(clean['AFR']!, boost.time, 1200);
      if (afr != null && afr.value! > 12.5) risky.add(afr);
    }
    if (risky.isNotEmpty) {
      final worst = risky.reduce((a, b) => a.value! > b.value! ? a : b);
      findings.add(Finding(FindingLevel.critical, 'Бедная смесь под наддувом',
          'В ${risky.length} точках AFR превышал 12.5 при наддуве выше 0.3 bar. '
          'Максимум ${worst.value!.toStringAsFixed(2)}.',
          'Время ${_time(worst.time)}, сырые байты ${worst.raw.map(hex2).join(' ')}.',
          'Сверьте показания широкополосного датчика и топливоподачу, '
          'прежде чем продолжать нагружать двигатель.'));
    }
  }

  // 6. Суммарная топливная коррекция.
  if (has('STFT') || has('LTFT')) {
    final short = has('STFT') ? _avg(values('STFT')) : 0.0;
    final long = has('LTFT') ? _avg(values('LTFT')) : 0.0;
    final sum = short + long;
    if (sum.abs() >= 10) {
      findings.add(Finding(
          sum.abs() >= 20 ? FindingLevel.critical : FindingLevel.warning,
          'Топливная коррекция ${sum.toStringAsFixed(1)} %',
          'Средняя STFT ${short.toStringAsFixed(1)} %, '
          'LTFT ${long.toStringAsFixed(1)} %.',
          'Порог предупреждения 10 %, критический 20 %.',
          sum > 0
              ? 'Смесь обогащается: возможен подсос воздуха или заниженный MAF.'
              : 'Смесь обедняется: проверьте форсунки, давление топлива и MAF.'));
    }
  }

  // 7. Перегрев охлаждающей жидкости.
  if (has('ECT')) {
    final peak = values('ECT').reduce((a, b) => a > b ? a : b);
    if (peak >= 100) {
      findings.add(Finding(
          peak >= 105 ? FindingLevel.critical : FindingLevel.warning,
          'Температура ОЖ ${peak.toStringAsFixed(0)} C',
          'Максимум за запись ${peak.toStringAsFixed(0)} C.',
          'Порог предупреждения 100 C, критический 105 C.',
          'Проверьте термостат, вентиляторы и уровень охлаждающей жидкости.'));
    }
  }

  // 8. Температура впуска.
  if (has('IAT')) {
    final peak = values('IAT').reduce((a, b) => a > b ? a : b);
    if (peak >= 60) {
      findings.add(Finding(FindingLevel.warning,
          'Температура впуска ${peak.toStringAsFixed(0)} C',
          'Горячий воздух повышает риск детонации.',
          'Максимум ${peak.toStringAsFixed(0)} C, порог 60 C.',
          'Дайте интеркулеру остыть между заездами.'));
    }
  }

  // 9. Бортовое напряжение.
  if (has('BATT')) {
    final list = values('BATT');
    final low = list.reduce((a, b) => a < b ? a : b);
    final high = list.reduce((a, b) => a > b ? a : b);
    if (low < 11.8 || high > 15.2) {
      findings.add(Finding(FindingLevel.warning,
          'Бортсеть ${low.toStringAsFixed(2)}...${high.toStringAsFixed(2)} В',
          'Выход за диапазон 11.8...15.2 В.',
          'Минимум ${low.toStringAsFixed(2)} В, максимум ${high.toStringAsFixed(2)} В.',
          'Проверьте генератор, ремень и состояние аккумулятора.'));
    }
  }

  // 10. Отклонение наддува от цели.
  if (has('BOOST') && has('BOOST_TGT')) {
    final deltas = <double>[];
    for (final actual in clean['BOOST']!) {
      final target = _nearest(clean['BOOST_TGT']!, actual.time, 1200);
      if (target != null) deltas.add(actual.value! - target.value!);
    }
    if (deltas.isNotEmpty) {
      final mean = _avg(deltas);
      if (mean.abs() > 0.15) {
        findings.add(Finding(FindingLevel.warning,
            'Наддув отличается от цели на ${mean.toStringAsFixed(2)} bar',
            'Среднее отклонение по ${deltas.length} парам значений.',
            'Порог 0.15 bar. Сравнение по ближайшим по времени чтениям, '
            'не по одновременному снимку.',
            mean > 0
                ? 'Перенаддув: проверьте вестгейт и его привод.'
                : 'Недобор наддува: ищите утечки впускного тракта.'));
      }
    }
  }

  // 11. Скорость обновления каналов.
  clean.forEach((id, samples) {
    if (samples.length < 4 || span < 5) return;
    final seconds =
        samples.last.time.difference(samples.first.time).inMilliseconds / 1000;
    if (seconds <= 0) return;
    final hz = (samples.length - 1) / seconds;
    if (hz < 0.5) {
      findings.add(Finding(FindingLevel.info, '$id: ${hz.toStringAsFixed(2)} Гц',
          'Канал обновляется реже двух секунд.',
          'Получено ${samples.length} значений за ${seconds.toStringAsFixed(1)} с.',
          'Уменьшите число выбранных PID: каждый адрес читается отдельным запросом.'));
    }
  });

  // 12. Ошибки обмена.
  final failed = attempts.values.where((s) => s.error.isNotEmpty).toList();
  if (failed.isNotEmpty) {
    findings.add(Finding(FindingLevel.info,
        'Ошибки чтения: ${failed.length} каналов',
        'Последние попытки этих PID завершились ошибкой.',
        failed.take(4).map((s) => '${s.pid.id}: ${s.error}').join('; '),
        'Смотрите вкладку диагностики: там видны TX, RX и таймауты.'));
  }

  const order = <FindingLevel, int>{
    FindingLevel.critical: 0,
    FindingLevel.warning: 1,
    FindingLevel.info: 2,
  };
  findings.sort((a, b) => order[a.level]!.compareTo(order[b.level]!));
  return LogAnalysis(findings, total, span, clean.length);
}
'''

FILES["analysis_options.yaml"] = r'''
analyzer:
  exclude:
    - vendor/**
    - transport_templates/**
  language:
    strict-casts: true
    strict-raw-types: true
'''

# Both transport templates must exist before selection, including variant B.
def generate_project():
    for relative, content in FILES.items():
        if relative.endswith(".py"):
            ast.parse(content, filename=relative)
            compile(content, relative, "exec")
    if APP.exists():
        backup = APP.with_name(APP.name + f"_backup_{time.time_ns()}")
        APP.rename(backup)
        print("Предыдущий новый проект сохранен:", backup)
    APP.mkdir(parents=True)
    run([FLUTTER / "bin/flutter", "create", "--no-pub", "--platforms=android",
         "--org", "com.subaru", "--project-name", "subaru_ssm2", APP])
    for relative in ["lib", "test", "android/app/src/main/kotlin"]:
        shutil.rmtree(APP / relative, ignore_errors=True)
    for relative in ["android/build.gradle", "android/settings.gradle", "android/app/build.gradle"]:
        (APP / relative).unlink(missing_ok=True)
    for relative, content in FILES.items():
        write(relative, content)
    write("android/local.properties", f"sdk.dir={CFG['sdk']}\nflutter.sdk={CFG['flutter']}\n")
    write("build_config.json", json.dumps({**CFG, "bt_package": BT_PACKAGE, "revision": "0.6"}, indent=2))
    write("pid_catalog.json", json.dumps(PIDS, ensure_ascii=False, indent=2))
    run([sys.executable, APP / "tool/prepare_bt.py", BT_PACKAGE])
    run([FLUTTER / "bin/flutter", "pub", "get"])
    print(f"\nСоздано {len(FILES)} файлов. Полная библиотека: 28 PID (20 обычных + 8 Extended).")
    print("Активный транспорт:", BT_PACKAGE)
    print("Проект:", APP)
    print("Далее ячейка 3: анализатор, тесты, сборка и проверка APK.")

FILES["transport_templates/serial.dart.txt"] = r'''
import 'dart:async';
import 'dart:typed_data';
import 'package:flutter_bluetooth_serial/flutter_bluetooth_serial.dart';
import 'bt_transport.dart';

class SelectedTransport implements BtTransport {
  BluetoothConnection? _connection;
  StreamSubscription<Uint8List>? _rx;
  final _data = StreamController<Uint8List>.broadcast();
  final _status = StreamController<bool>.broadcast();
  @override
  String get name => 'B / flutter_bluetooth_serial 0.4.0 (local Android patch)';
  @override
  bool get connected => _connection?.isConnected ?? false;
  @override
  Stream<Uint8List> get data => _data.stream;
  @override
  Stream<bool> get status => _status.stream;
  @override
  Future<List<BtDevice>> paired() async {
    await requestBluetoothPermissions();
    final devices = await FlutterBluetoothSerial.instance.getBondedDevices();
    return devices.map((d) => BtDevice(d.name ?? '', d.address)).toList();
  }
  @override
  Future<void> connect(String address) async {
    await requestBluetoothPermissions();
    await disconnect();
    final connection = await BluetoothConnection.toAddress(address);
    _connection = connection;
    _rx = connection.input!.listen((b) => _data.add(Uint8List.fromList(b)),
      onDone: () => _status.add(false),
      onError: (Object e) { _data.addError(e); _status.add(false); });
    _status.add(true);
  }
  @override
  Future<void> write(String ascii) async {
    final connection = _connection;
    if (connection == null || !connection.isConnected) throw StateError('SPP disconnected');
    connection.output.add(Uint8List.fromList(ascii.codeUnits));
    await connection.output.allSent;
  }
  @override
  Future<void> disconnect() async {
    final connection = _connection;
    _connection = null;
    await _rx?.cancel(); _rx = null;
    if (connection != null) await connection.close();
  }
  @override
  Future<void> dispose() async {
    try { await disconnect(); } finally { await _data.close(); await _status.close(); }
  }
}
BtTransport createTransport() => SelectedTransport();
'''

generate_project()

Предыдущий новый проект сохранен: /content/subaru_ssm2_fixed_backup_1789831733437041783
Creating project ....
Wrote 35 files.

All done!
You can find general documentation for Flutter at: https://docs.flutter.dev/
Detailed API documentation is available at: https://api.flutter.dev/
If you prefer video documentation, consider: https://www.youtube.com/c/flutterdev

In order to run your application, type:

  $ cd .
  $ flutter run

Your application code is in ./lib/main.dart.


/content/subaru_ssm2_fixed/tool/prepare_bt.py:57: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(vendor, members=members)
[PATCH] A: immutable RX chunk, EOF, safe close, propagated write error
[OK] Selected bluetooth_classic 0.0.4; common pub-cache untouched

Resolving dependencies...
+ async 2.13.1
+ bluetooth_classic 0.0.4 from path vendor/bluetooth_classic
+ boolean_selec

In [12]:
# @title 2b/3 | Map Lab - интеллектуальный анализатор карт в приложение { display-mode: "form" }
# Выполнять ПОСЛЕ ячейки 2/3 (проект сгенерирован) и ДО ячейки 3/3 (проверка и сборка).
# Что делает ячейка:
#   1) записывает модуль lib/maps_lab/* — разбор дефиниций RomRaider БЕЗ внешних пакетов
#      (свой мини-XML-парсер), парсер ROM SH7058, нормализация CSV-лога, правила вердиктов,
#      2D-таблица и 3D-просмотрщик карты на CustomPainter, экран Map Lab;
#   2) скачивает A2TB100B.xml / A2TB100K.xml / 32BITBASE.xml из TD-D/SubaruDefs в assets;
#   3) мягко патчит pubspec (только секция assets — новых плагинов НЕ добавляем,
#      поэтому prepare_bt не может сломать сборку) и манифест (MANAGE_EXTERNAL_STORAGE,
#      опционально; без него просто работаем из папки приложения);
#   4) пишет test/maps_lab_test.dart (имя пакета берётся из вашего pubspec).
# Ваша ячейка 3/3 затем сама прогонит pub get / analyze / test и соберёт APK с анализатором.

import json
import re
import urllib.request
from pathlib import Path

CONFIG = Path("/content/ssm2_fixed_env.json")
if not CONFIG.exists():
    raise RuntimeError("Сначала выполните ячейку 1")
CFG = json.loads(CONFIG.read_text(encoding="utf-8"))
APP = Path(CFG["app"])
if not (APP / "pubspec.yaml").exists():
    raise RuntimeError("Проект не найден. Сначала выполните ячейку 2/3.")
FILES = {}
FILES["lib/maps_lab/maps_lab_core.dart"] = r"""
/// Map Lab core — чистый Dart без Flutter.
/// Разбор include-цепочки дефиниций RomRaider (A2TB100B → A2TB100K → 32BITBASE)
/// и извлечение матриц карт из бинарника прошивки SH7058 (big-endian).
library;

import 'dart:convert';
import 'dart:typed_data';

// ────────────────────────────────────────────────────────── выражения скейлинга
/// Безопасный вычислитель формул вида `(x*.3515625)-20`, `14.7/(1+x*.0078125)`.
/// Dart не имеет eval — здесь маленький рекурсивный парсер (+ - * / скобки).
class SafeExpr {
  static final RegExp _ok = RegExp(r'^[0-9xX\.\+\-\*/\(\)\s]+$');

  static double Function(double) compile(String? src) {
    if (src == null || src.trim().isEmpty) return (v) => v;
    final s = src.replaceAll('X', 'x');
    if (!_ok.hasMatch(s)) return (v) => v;
    try {
      final parser = _ExprParser(s);
      final node = parser.parse();
      return (x) => node(x);
    } catch (_) {
      return (v) => v;
    }
  }
}

class _ExprParser {
  _ExprParser(this.s);
  final String s;
  int i = 0;

  double Function(double) parse() {
    final n = _expr();
    _ws();
    if (i != s.length) throw FormatException('лишние символы: ${s.substring(i)}');
    return n;
  }

  void _ws() {
    while (i < s.length && s[i] == ' ') i++;
  }

  bool _eat(String ch) {
    _ws();
    if (i < s.length && s[i] == ch) {
      i++;
      return true;
    }
    return false;
  }

  double Function(double) _expr() {
    var node = _term();
    while (true) {
      if (_eat('+')) {
        final r = _term();
        final l = node;
        node = (x) => l(x) + r(x);
      } else if (_eat('-')) {
        final r = _term();
        final l = node;
        node = (x) => l(x) - r(x);
      } else {
        return node;
      }
    }
  }

  double Function(double) _term() {
    var node = _factor();
    while (true) {
      if (_eat('*')) {
        final r = _factor();
        final l = node;
        node = (x) => l(x) * r(x);
      } else if (_eat('/')) {
        final r = _factor();
        final l = node;
        node = (x) => r(x) == 0 ? double.nan : l(x) / r(x);
      } else {
        return node;
      }
    }
  }

  static final RegExp _num = RegExp(r'(\d+\.?\d*|\.\d+)');

  double Function(double) _factor() {
    _ws();
    if (_eat('-')) {
      final n = _factor();
      return (x) => -n(x);
    }
    if (_eat('+')) return _factor();
    if (_eat('(')) {
      final n = _expr();
      if (!_eat(')')) throw const FormatException('нет закрывающей скобки');
      return n;
    }
    _ws();
    if (i < s.length && s[i] == 'x') {
      i++;
      return (x) => x;
    }
    final m = _num.matchAsPrefix(s, i);
    if (m == null) throw FormatException('ожидалось число @ $i');
    i = m.end;
    final v = double.parse(m.group(0)!);
    return (x) => v;
  }
}

// ────────────────────────────────────────────────────────── скейлинги
enum Stype { u8, i8, u16, i16, f32 }

class Scaling {
  Scaling({
    required this.name,
    required this.units,
    required this.type,
    required this.bigEndian,
    required this.to,
    required this.fr,
    required this.format,
  });

  factory Scaling.fromAttrs(String name, Map<String, String> a) {
    final st = switch (a['storagetype'] ?? 'uint8') {
      'int8' => Stype.i8,
      'uint16' => Stype.u16,
      'int16' => Stype.i16,
      'float' => Stype.f32,
      _ => Stype.u8,
    };
    return Scaling(
      name: name,
      units: a['units'] ?? '',
      type: st,
      bigEndian: (a['endian'] ?? 'big') == 'big',
      to: SafeExpr.compile(a['toexpr']),
      fr: SafeExpr.compile(a['frexpr']),
      format: a['format'] ?? '%.2f',
    );
  }

  final String name;
  final String units;
  final Stype type;
  final bool bigEndian;
  final double Function(double) to;
  final double Function(double) fr;
  final String format;

  int get bytes => switch (type) { Stype.f32 => 4, Stype.u16 || Stype.i16 => 2, _ => 1 };
}

// ────────────────────────────────────────────────────────── модель дефиниций
class AxisDef {
  String name = '';
  String? scalingName;
  int? address;
  int? elements;
  Scaling? scaling;
}

class TableDef {
  TableDef(this.name);
  final String name;
  String? type;
  String? category;
  String? scalingName;
  int? dataAddress;
  Scaling? dataScaling;
  final List<AxisDef> axes = [];
}

class DefMeta {
  String xmlid = '';
  String internalIdAddress = '2000';
  String internalIdString = '';
  String ecuid = '';
  String checksumModule = '';
}

class _RawTable {
  _RawTable(this.attrs, this.children);
  final Map<String, String> attrs;
  final List<Map<String, String>> children;
}

class _RawDef {
  final meta = DefMeta();
  final includes = <String>[];
  final scalings = <String, Map<String, String>>{};
  final tables = <_RawTable>[];
}

/// Построение объединённого набора дефиниций из XML-текстов.
/// [xmlById] — карта xmlid → текст XML (A2TB100B, A2TB100K, 32BITBASE).
class DefSet {
  DefSet._();
  final meta = DefMeta();
  final chain = <String>[];
  final scalings = <String, Scaling>{};
  final tables = <String, TableDef>{};

  static const _generic = {'x', 'y', 'z', ''};

  // Мини-парсер XML без внешних пакетов: формат дефиниций RomRaider машинный
  // и регулярный (без namespace, без '>' внутри значений атрибутов).
  static final _attrRe = RegExp(r'([\w:-]+)="([^"]*)"');
  static final _tableTok = RegExp(r'<table\s[^>]*?>|</table>');
  static final _scalingRe = RegExp(r'<scaling\s+([^>]*?)/>');
  static final _romidRe = RegExp(r'<romid>([\s\S]*?)</romid>');
  static final _innerTag = RegExp(r'<(\w+)>([^<]*)</\1>');
  static final _includeRe = RegExp(r'<include>\s*([^<]+?)\s*</include>');

  static Map<String, String> _attrs(String tag) =>
      {for (final m in _attrRe.allMatches(tag)) m.group(1)!: m.group(2)!};

  static _RawDef _parse(String raw) {
    final def = _RawDef();

    final romid = _romidRe.firstMatch(raw);
    if (romid != null) {
      for (final m in _innerTag.allMatches(romid.group(1)!)) {
        final v = m.group(2)!.trim();
        if (v.isEmpty) continue;
        switch (m.group(1)) {
          case 'xmlid':
            def.meta.xmlid = v;
          case 'internalidaddress':
            def.meta.internalIdAddress = v;
          case 'internalidstring':
            def.meta.internalIdString = v;
          case 'ecuid':
            def.meta.ecuid = v;
          case 'checksummodule':
            def.meta.checksumModule = v;
        }
      }
    }
    for (final m in _includeRe.allMatches(raw)) {
      def.includes.add(m.group(1)!);
    }
    for (final m in _scalingRe.allMatches(raw)) {
      final a = _attrs(m.group(0)!);
      if (a['storagetype'] == 'bloblist') continue;
      final n = a['name'];
      if (n != null && n.isNotEmpty) def.scalings[n] = a;
    }

    // Стек по глубине: <rom> → <table> (depth 1) → axis <table ../> (depth 2, самозакрытые).
    var depth = 0;
    for (final m in _tableTok.allMatches(raw)) {
      final tok = m.group(0)!;
      if (tok == '</table>') {
        if (depth > 0) depth--;
        continue;
      }
      final selfClose = tok.endsWith('/>');
      final a = _attrs(tok);
      if (depth == 0) {
        def.tables.add(_RawTable(a, []));
        if (!selfClose) depth = 1;
      } else {
        def.tables.last.children.add(a);
        // вложенных глубже осей в формате нет
      }
    }
    return def;
  }

  static DefSet build(Map<String, String> xmlById, String rootId) {
    final raw = <String, _RawDef>{};
    for (final e in xmlById.entries) {
      raw[e.key] = _parse(e.value);
    }

    // Цепочка include: базовые → производные.
    final order = <String>[];
    final seen = <String>{};
    void walk(String id) {
      if (!seen.add(id)) return;
      for (final inc in raw[id]?.includes ?? const <String>[]) {
        if (raw.containsKey(inc)) walk(inc);
      }
      order.add(id);
    }
    walk(rootId);

    final set = DefSet._();
    set.chain.addAll(order);
    if (raw.containsKey(rootId)) {
      final m = raw[rootId]!.meta;
      set.meta
        ..xmlid = m.xmlid
        ..internalIdAddress = m.internalIdAddress
        ..internalIdString = m.internalIdString
        ..ecuid = m.ecuid
        ..checksumModule = m.checksumModule;
    }

    for (final id in order) {
      final d = raw[id]!;
      d.scalings.forEach((n, a) {
        set.scalings[n] = Scaling.fromAttrs(n, a);
      });
      for (final rt in d.tables) {
        final name = (rt.attrs['name'] ?? '').trim();
        if (name.isEmpty) continue;
        final t = set.tables.putIfAbsent(name, () => TableDef(name));
        for (final k in ['type', 'category', 'level', 'scaling']) {
          final v = rt.attrs[k];
          if (v != null && v.isNotEmpty) {
            switch (k) {
              case 'type':
                t.type = v;
              case 'category':
                t.category = v;
              case 'scaling':
                t.scalingName = v;
            }
          }
        }
        final addr = rt.attrs['address'];
        if (addr != null && addr.isNotEmpty) t.dataAddress = int.parse(addr, radix: 16);
        for (var i = 0; i < rt.children.length; i++) {
          while (t.axes.length <= i) {
            t.axes.add(AxisDef());
          }
          final a = t.axes[i];
          final ch = rt.children[i];
          final cn = (ch['name'] ?? '').trim();
          if (!_generic.contains(cn.toLowerCase())) a.name = cn;
          if ((ch['scaling'] ?? '').isNotEmpty) a.scalingName = ch['scaling'];
          if ((ch['type'] ?? '').isNotEmpty) {} // тип оси нам не критичен
          if ((ch['elements'] ?? '').isNotEmpty) a.elements = int.parse(ch['elements']!);
          if ((ch['address'] ?? '').isNotEmpty) a.address = int.parse(ch['address']!, radix: 16);
        }
      }
    }

    // Привязка скейлингов.
    for (final t in set.tables.values) {
      t.dataScaling = set.scalings[t.scalingName];
      for (final a in t.axes) {
        a.scaling = set.scalings[a.scalingName];
      }
    }
    return set;
  }

  /// Таблица с данными, адресом и полным набором осей — годна для чтения из ROM.
  bool isReadable3D(TableDef t) =>
      t.dataAddress != null &&
      t.dataScaling != null &&
      t.axes.length >= 2 &&
      t.axes[0].address != null &&
      t.axes[0].elements != null &&
      t.axes[0].scaling != null &&
      t.axes[1].address != null &&
      t.axes[1].elements != null &&
      t.axes[1].scaling != null;
}

// ────────────────────────────────────────────────────────── ROM
class AxisVals {
  AxisVals(this.name, this.values, this.guess);
  final String name;
  final List<double> values;
  final String guess;
}

class MapGrid {
  MapGrid({
    required this.name,
    required this.kind,
    required this.units,
    required this.addr,
    required this.x,
    required this.y,
    required this.data,
  });
  final String name;
  final String kind;
  final String units;
  final int addr;
  final AxisVals x;
  final AxisVals y;

  /// data[y][x]
  final List<List<double>> data;

  int get rows => data.length;
  int get cols => data.isEmpty ? 0 : data[0].length;
  double get vmin => data.expand((r) => r).reduce((a, b) => a < b ? a : b);
  double get vmax => data.expand((r) => r).reduce((a, b) => a > b ? a : b);
}

class RomParser {
  RomParser(this.rom);
  final Uint8List rom;

  String readRomId(int addr) {
    if (addr < 0 || addr + 16 > rom.length) return '';
    final bytes = <int>[];
    for (var i = addr; i < addr + 16; i++) {
      final b = rom[i];
      if (b == 0) break;
      bytes.add(b);
    }
    return ascii.decode(bytes, allowInvalid: true).trim();
  }

  List<double> _read(Scaling sc, int addr, int count) {
    final size = sc.bytes;
    if (addr < 0 || addr + count * size > rom.length) {
      throw FormatException(
          'диапазон 0x${addr.toRadixString(16)} + $count×$size вне ROM (${rom.length} байт)');
    }
    final bd = ByteData.sublistView(rom, addr, addr + count * size);
    final en = sc.bigEndian ? Endian.big : Endian.little;
    final out = List<double>.filled(count, 0);
    for (var i = 0; i < count; i++) {
      final v = switch (sc.type) {
        Stype.u8 => bd.getUint8(i).toDouble(),
        Stype.i8 => bd.getInt8(i).toDouble(),
        Stype.u16 => bd.getUint16(i * 2, en).toDouble(),
        Stype.i16 => bd.getInt16(i * 2, en).toDouble(),
        Stype.f32 => bd.getFloat32(i * 4, en),
      };
      out[i] = sc.to(v);
    }
    return out;
  }

  static String guessAxis(List<double> v) {
    if (v.isEmpty) return '?';
    var lo = v.first, hi = v.first;
    for (final x in v) {
      if (x < lo) lo = x;
      if (x > hi) hi = x;
    }
    if (hi > 800) return 'rpm';
    if (hi > 50 && hi <= 600) return 'нм/у.е.';
    if (hi <= 5.5 && lo >= -0.5) return 'г/об·бар';
    if (hi <= 14 && lo >= 0) return 'вольты/%';
    return '?';
  }

  static String classify(String name, String units) {
    final n = name.toLowerCase();
    if (n.contains('wastegate duty')) return 'wgdc';
    if (n.contains('target boost')) return 'boost';
    if (n.contains('requested torque')) return 'torque';
    if (n.contains('knock correction')) return 'knockadv';
    if (n.contains('fueling') || n.contains('fuel')) return 'fuel';
    if (n.contains('timing')) return 'timing';
    return 'other';
  }

  double _round4(double v) => (v * 10000).roundToDouble() / 10000;

  MapGrid? extract(TableDef t) {
    if (t.dataAddress == null || t.dataScaling == null) return null;
    if (t.axes.length < 2) return null;
    final ax = t.axes[0], ay = t.axes[1];
    if (ax.address == null || ax.elements == null || ax.scaling == null) return null;
    if (ay.address == null || ay.elements == null || ay.scaling == null) return null;

    final xv = _read(ax.scaling!, ax.address!, ax.elements!);
    final yv = _read(ay.scaling!, ay.address!, ay.elements!);
    final cols = ax.elements!, rows = ay.elements!;
    final flat = _read(t.dataScaling!, t.dataAddress!, rows * cols);
    final grid = <List<double>>[
      for (var r = 0; r < rows; r++) [for (var c = 0; c < cols; c++) _round4(flat[r * cols + c])],
    ];
    return MapGrid(
      name: t.name,
      kind: classify(t.name, t.dataScaling!.units),
      units: t.dataScaling!.units,
      addr: t.dataAddress!,
      x: AxisVals(ax.name.isEmpty ? 'X' : ax.name, [for (final v in xv) _round4(v)], guessAxis(xv)),
      y: AxisVals(ay.name.isEmpty ? 'Y' : ay.name, [for (final v in yv) _round4(v)], guessAxis(yv)),
      data: grid,
    );
  }

  /// Список карт тюнинга A2TB100B (имена — как в дефиниции TD-D/SubaruDefs).
  static const keyTables = [
    'Base Timing Primary Cruise',
    'Base Timing Primary Non-Cruise',
    'Primary Open Loop Fueling',
    'Target Boost_',
    'Initial Wastegate Duty_',
    'Max Wastegate Duty_',
    'Knock Correction Advance Max Non-Cruise',
    'Requested Torque A (Accelerator Pedal) SI-DRIVE Sport',
  ];

  Map<String, MapGrid> extractKeys(DefSet defs) {
    final out = <String, MapGrid>{};
    for (final n in keyTables) {
      final t = defs.tables[n];
      if (t == null) continue;
      try {
        final g = extract(t);
        if (g != null) out[n] = g;
      } catch (_) {
        // карта битая/вне ROM — пропускаем, UI покажет что извлеклось
      }
    }
    return out;
  }
}
"""
FILES["lib/maps_lab/maps_lab_log.dart"] = r"""
/// Map Lab: лог и правила.
/// Парсер CSV (любой разделитель, десятичная запятая), канонизация колонок,
/// здоровье лога и правила вердиктов по ячейкам карт — зеркало Python-версии.
library;

import 'dart:convert';
import 'dart:math' as math;

import 'maps_lab_core.dart';

// ────────────────────────────────────────────────────────── лог
class LogData {
  final cols = <String, List<double?>>{};
  int get rows => cols.isEmpty ? 0 : cols.values.first.length;

  bool has(String k) => cols.containsKey(k);
  List<double?>? operator [](String k) => cols[k];

  static const aliases = <String, List<String>>{
    'time': ['time', 'timestamp', 'time s', 'elapsed'],
    'rpm': ['engine speed', 'rpm', 'engine speed rpm'],
    'load': ['engine load g/rev', 'load_4b', 'engine load 4-byte', 'calculated load', 'engine load', 'load'],
    'fbkc': ['feedback knock correction', 'fbkc'],
    'flkc': ['fine learning knock correction', 'fkl', 'fine learning knock advance', 'flkc'],
    'iam': ['iam', 'ignition advance multiplier'],
    'timing': ['total ignition timing', 'ignition timing', 'timing'],
    'afr': ['afr', 'a/f sensor #1', 'a/f sensor 1', 'air/fuel ratio', 'estimated afr', 'lambda'],
    'boost': ['manifold relative pressure', 'boost', 'boost_rel'],
    'tgt': ['target boost', 'boost_tgt'],
    'berr': ['boost error', 'boost_err'],
    'wgdc': ['primary wastegate duty', 'wastegate duty', 'wgdc', 'boost control solenoid duty'],
    'tq': ['requested torque', 'cl_target', 'demand torque'],
    'thr': ['throttle opening angle', 'throttle plate', 'throttle', 'throttle position'],
    'iat': ['intake air temperature', 'iat'],
    'ect': ['coolant temperature', 'ect', 'engine coolant temperature'],
    'loop': ['cl/ol', 'fueling status', 'closed loop', 'loop'],
  };

  static String _canon(String s) {
    var c = s.toLowerCase().trim();
    final p = c.indexOf('(');
    if (p > 0) c = c.substring(0, p);
    final b = c.indexOf('[');
    if (b > 0) c = c.substring(0, b);
    return c.replaceAll('*', '').replaceAll(RegExp(r'\s+'), ' ').trim();
  }

  static LogData parse(String text) {
    final head = text.substring(0, math.min(text.length, 4096));
    final sep = ';'.allMatches(head).length > ','.allMatches(head).length ? ';' : ',';
    final decComma = sep == ';' && ','.allMatches(head).length > '.'.allMatches(head).length;

    final lines = const LineSplitter().convert(text).where((l) => l.trim().isNotEmpty).toList();
    if (lines.isEmpty) return LogData();
    final header = lines.first.split(sep).map((h) => h.trim().replaceAll('"', '')).toList();

    final raw = List<List<double?>>.generate(header.length, (_) => []);
    final canonHead = header.map(_canon).toList();

    for (var li = 1; li < lines.length; li++) {
      final parts = lines[li].split(sep);
      for (var i = 0; i < header.length; i++) {
        if (i >= parts.length) {
          raw[i].add(null);
          continue;
        }
        var tok = parts[i].trim().replaceAll('"', '');
        if (decComma) tok = tok.replaceAll(',', '.');
        raw[i].add(double.tryParse(tok));
      }
    }

    final log = LogData();
    final usedNames = <String>{};
    for (final entry in aliases.entries) {
      final canon = entry.key;
      final variants = entry.value.map(_canon).toSet();
      for (var i = 0; i < header.length; i++) {
        if (usedNames.contains(header[i])) continue;
        if (variants.contains(canonHead[i])) {
          log.cols[canon] = raw[i];
          usedNames.add(header[i]);
          break;
        }
      }
    }

    // Производные.
    final afr = log.cols['afr'];
    if (afr != null) {
      final vals = afr.whereType<double>().toList()..sort();
      if (vals.isNotEmpty && vals[vals.length ~/ 2] <= 2.2) {
        log.cols['afr'] = [for (final v in afr) v == null ? null : v * 14.7];
      }
    }
    final boost = log.cols['boost'], tgt = log.cols['tgt'];
    if (boost != null && tgt != null && log.cols['berr'] == null) {
      log.cols['berr'] = [
        for (var i = 0; i < boost.length; i++)
          (boost[i] != null && tgt[i] != null) ? boost[i]! - tgt[i]! : null,
      ];
    }
    return log;
  }
}

// ────────────────────────────────────────────────────────── здоровье лога
class LogHealth {
  final notes = <String>[];
  final missing = <String, List<String>>{};
  bool blockReady(String b) => (missing[b] ?? const []).isEmpty;
}

class LogAudit {
  static const _needFor = {
    'timing': ['rpm', 'load', 'fbkc', 'flkc', 'iat'],
    'fuel': ['rpm', 'load', 'afr', 'boost'],
    'wgdc': ['rpm', 'tq', 'berr'],
  };

  static const _hint = {
    'wgdc': 'Primary Wastegate Duty',
    'tq': 'Requested Torque',
    'thr': 'Throttle Opening Angle',
    'afr': 'A/F Sensor #1',
    'loop': 'CL/OL Fueling Status',
  };

  static LogHealth check(LogData log) {
    final h = LogHealth();
    h.notes.add('строк: ${log.rows}');

    double? maxOf(String k) {
      final c = log[k];
      if (c == null) return null;
      double? m;
      for (final v in c) {
        if (v != null && (m == null || v > m)) m = v;
      }
      return m;
    }

    double? minOf(String k) {
      final c = log[k];
      if (c == null) return null;
      double? m;
      for (final v in c) {
        if (v != null && (m == null || v < m)) m = v;
      }
      return m;
    }

    final mr = maxOf('rpm');
    if (mr != null) h.notes.add('RPM max: ${mr.toStringAsFixed(0)}');
    final iamMin = minOf('iam');
    if (iamMin != null) {
      h.notes.add('IAM min: ${iamMin.toStringAsFixed(2)}'
          '${iamMin < 0.99 ? ' — ЛОГ НЕ ГОДИТСЯ: сначала доучить ЭБУ' : ' — ок'}');
    }
    final mb = maxOf('boost');
    if (mb != null) h.notes.add('буст max: ${mb.toStringAsFixed(2)} бар');

    final t = log['time'];
    if (t != null) {
      final dts = <double>[];
      for (var i = 1; i < t.length; i++) {
        if (t[i] != null && t[i - 1] != null) dts.add(t[i]! - t[i - 1]!);
      }
      dts.sort();
      if (dts.isNotEmpty) {
        final med = dts[dts.length ~/ 2];
        final gaps = dts.where((d) => d > 0.3).length;
        h.notes.add('частота: медиана ${(1 / med).toStringAsFixed(1)} Гц · провалов >300 мс: $gaps');
        if (med > 0.25) h.notes.add('!! реже 4 Гц — сократите набор PID до 10–12');
      }
    }

    _needFor.forEach((block, req) {
      final miss = req.where((r) => !log.has(r)).toList();
      h.missing[block] = miss;
    });
    final allMiss = {for (final v in h.missing.values) ...v};
    for (final m in allMiss) {
      final hint = _hint[m];
      if (hint != null) h.notes.add('добавить в логгер PID: $hint');
    }
    return h;
  }
}

// ────────────────────────────────────────────────────────── правила
class CellNote {
  int n = 0;
  double? fbkc, flkc, iat, iam, afr, target, berr;
  String why = '';
}

class MapResult {
  MapResult(int rows, int cols)
      : delta = List.generate(rows, (_) => List.filled(cols, 0.0)),
        info = List.generate(rows, (_) => List<CellNote?>.filled(cols, null));
  final List<List<double>> delta;
  final List<List<CellNote?>> info;

  int get dec {
    var c = 0;
    for (final r in delta) {
      for (final v in r) {
        if (v < 0) c++;
      }
    }
    return c;
  }

  int get inc {
    var c = 0;
    for (final r in delta) {
      for (final v in r) {
        if (v > 0) c++;
      }
    }
    return c;
  }
}

class AnalyzerConfig {
  const AnalyzerConfig({
    this.minN = 6,
    this.fbkcEvent = -1.5,
    this.flkcEvent = -1.5,
    this.timingStep = 0.5,
    this.timingMaxCut = -3.0,
    this.allowTimingAdd = false,
    this.afrErr = 0.40,
    this.afrMaxCut = -0.8,
    this.boostErr = 0.04,
    this.wgdcMaxDelta = 6.0,
    this.wotLoad = 2.2,
    this.targetBoostGain = 0.0,
  });
  final int minN;
  final double fbkcEvent, flkcEvent, timingStep, timingMaxCut;
  final bool allowTimingAdd;
  final double afrErr, afrMaxCut, boostErr, wgdcMaxDelta, wotLoad, targetBoostGain;
}

class Analyzer {
  Analyzer(this.cfg);
  final AnalyzerConfig cfg;

  double _roundStep(double v, double s) => (v / s).roundToDouble() * s;

  int _binIdx(List<double> axis, double v) {
    if (axis.length <= 1) return 0;
    var lo = 0, hi = axis.length - 1;
    // бинарный поиск ближайшей середины
    while (lo < hi - 1) {
      final mid = (lo + hi) >> 1;
      if (v >= axis[mid]) {
        lo = mid;
      } else {
        hi = mid;
      }
    }
    final mid = (axis[lo] + axis[hi]) / 2;
    return v < mid ? lo : hi;
  }

  MapResult? analyzeMap(MapGrid g, LogData log) {
    final kind = g.kind;
    if (!{'timing', 'knockadv', 'fuel', 'wgdc', 'boost'}.contains(kind)) return null;

    // RPM-ось
    final xIsRpm = g.x.guess == 'rpm' || (g.y.guess != 'rpm' && g.x.values.last > 800);
    final rpmAxis = xIsRpm ? g.x.values : g.y.values;
    final otherAxis = xIsRpm ? g.y.values : g.x.values;

    final need = switch (kind) {
      'timing' || 'knockadv' => ['rpm', 'load', 'fbkc', 'flkc'],
      'fuel' => ['rpm', 'load', 'afr', 'fbkc', 'flkc'],
      _ => ['rpm', 'tq', 'berr'],
    };
    if (need.any((k) => !log.has(k))) return null;

    final rpmCol = log['rpm']!;
    final otherCol = switch (kind) {
      'timing' || 'knockadv' || 'fuel' => log['load']!,
      _ => log['tq']!,
    };

    final rows = g.rows, cols = g.cols;
    final cnt = List<int>.filled(rows * cols, 0);
    final fbkcMin = List<double>.filled(rows * cols, 0);
    final flkcSum = List<double>.filled(rows * cols, 0);
    final iatMax = List<double>.filled(rows * cols, -999);
    final iamMin = List<double>.filled(rows * cols, 999);
    final afrSum = List<double>.filled(rows * cols, 0);
    final berrSum = List<double>.filled(rows * cols, 0);

    final fbkc = log['fbkc'], flkc = log['flkc'], iat = log['iat'];
    final iam = log['iam'], afr = log['afr'], berr = log['berr'];

    for (var i = 0; i < log.rows; i++) {
      final rv = rpmCol[i], ov = otherCol[i];
      if (rv == null || ov == null) continue;
      final ri = _binIdx(xIsRpm ? otherAxis : rpmAxis, xIsRpm ? ov : rv);
      final ci = _binIdx(xIsRpm ? rpmAxis : otherAxis, xIsRpm ? rv : ov);
      final idx = ri * cols + ci;
      cnt[idx]++;
      final f = fbkc?[i];
      if (f != null && (cnt[idx] == 1 || f < fbkcMin[idx])) fbkcMin[idx] = f;
      final fl = flkc?[i];
      if (fl != null) flkcSum[idx] += fl;
      final it = iat?[i];
      if (it != null && it > iatMax[idx]) iatMax[idx] = it;
      final im = iam?[i];
      if (im != null && im < iamMin[idx]) iamMin[idx] = im;
      final a = afr?[i];
      if (a != null) afrSum[idx] += a;
      final be = berr?[i];
      if (be != null) berrSum[idx] += be;
    }

    final res = MapResult(rows, cols);
    for (var ri = 0; ri < rows; ri++) {
      for (var ci = 0; ci < cols; ci++) {
        final idx = ri * cols + ci;
        final n = cnt[idx];
        if (n < cfg.minN) continue;
        final note = CellNote()
          ..n = n
          ..fbkc = fbkcMin[idx]
          ..flkc = flkcSum[idx] / n
          ..iat = iatMax[idx] < -900 ? null : iatMax[idx]
          ..iam = iamMin[idx] > 900 ? null : iamMin[idx]
          ..afr = afrSum[idx] > 0 ? afrSum[idx] / n : null
          ..berr = berrSum[idx] != 0 ? berrSum[idx] / n : null;

        if (kind == 'timing' || kind == 'knockadv') {
          if (note.iam != null && note.iam! < 0.99) {
            res.info[ri][ci] = note..why = 'IAM=${note.iam!.toStringAsFixed(2)} — сначала доучить';
            continue;
          }
          final knock = math.min(note.fbkc!, note.flkc! * 1.4);
          if (knock <= cfg.fbkcEvent) {
            final d = math.max(cfg.timingMaxCut,
                math.min(-cfg.timingStep, _roundStep(knock * 0.6, cfg.timingStep)));
            res.delta[ri][ci] = d;
            res.info[ri][ci] = note..why = 'детон-кластер · FBKC ${note.fbkc!.toStringAsFixed(1)}°, '
                'FLKC ${note.flkc!.toStringAsFixed(1)}° · снять ${d.abs().toStringAsFixed(1)}° здесь и сгладить соседей';
          } else if (cfg.allowTimingAdd &&
              n >= 18 &&
              (note.iat == null || note.iat! <= 45)) {
            res.delta[ri][ci] = cfg.timingStep;
            res.info[ri][ci] = note..why = 'чисто, IAM=1.0 — опционально +0.5°';
          }
        } else if (kind == 'fuel') {
          final loadAxisValue = xIsRpm ? g.y.values[ri] : g.x.values[ci];
          if (loadAxisValue < cfg.wotLoad || note.afr == null) continue;
          var target = g.data[ri][ci];
          if (target < 9.0) target *= 14.7; // карта в λ
          note.target = target;
          final err = note.afr! - target;
          final knock = math.min(note.fbkc!, note.flkc!);
          if (err > cfg.afrErr) {
            final d = math.max(cfg.afrMaxCut, _roundStep(-err, 0.1));
            res.delta[ri][ci] = d;
            res.info[ri][ci] = note
              ..why = 'факт ${note.afr!.toStringAsFixed(2)} против цели ${target.toStringAsFixed(2)} '
                  '— обогатить на ${d.abs().toStringAsFixed(1)}; если не помогает — MAF/давление';
          } else if (knock <= cfg.fbkcEvent) {
            res.delta[ri][ci] = -0.3;
            res.info[ri][ci] = note..why = 'детон при цели ~совпадает — запас −0.3 AFR';
          }
        } else {
          // wgdc / target boost
          final e = note.berr;
          if (e == null) continue;
          if (kind == 'wgdc') {
            if (e.abs() > cfg.boostErr) {
              final d = _roundStep(-e * 45, 1)
                  .clamp(-cfg.wgdcMaxDelta, cfg.wgdcMaxDelta - 1)
                  .toDouble();
              if (d != 0) {
                res.delta[ri][ci] = d;
                res.info[ri][ci] = note
                  ..why = '${e > 0 ? 'овербуст' : 'недобор'} ${e.toStringAsFixed(2)} бар '
                      '→ WGDC ${d > 0 ? '+' : ''}${d.toStringAsFixed(0)}%';
              }
            }
          } else {
            if (e > cfg.boostErr) {
              res.info[ri][ci] = note..why = 'овербуст ${e.toStringAsFixed(2)} бар — чинить через WGDC/TD, не таргет';
            }
            if (cfg.targetBoostGain > 0) {
              res.delta[ri][ci] = cfg.targetBoostGain;
              res.info[ri][ci] = note
                ..why = 'политика +${cfg.targetBoostGain} бар (после фикса WGDC и чистого детон-лога)';
            }
          }
        }
      }
    }
    return res;
  }

  Map<String, MapResult> run(Map<String, MapGrid> maps, LogData log) {
    final out = <String, MapResult>{};
    for (final e in maps.entries) {
      final r = analyzeMap(e.value, log);
      if (r != null) out[e.key] = r;
    }
    return out;
  }
}
"""
FILES["lib/maps_lab/map3d_view.dart"] = r"""
/// 3D-визуализатор карты в стиле RomRaider (без внешних пакетов):
/// painter's algorithm, вращение жестом, тап по ячейке.
library;

import 'dart:math' as math;

import 'package:flutter/material.dart';

import 'maps_lab_core.dart';
import 'maps_lab_log.dart';

class Map3DView extends StatefulWidget {
  const Map3DView({
    super.key,
    required this.grid,
    this.result,
    this.selRi,
    this.selCi,
    this.onCell,
  });

  final MapGrid grid;
  final MapResult? result;
  final int? selRi;
  final int? selCi;
  final void Function(int ri, int ci)? onCell;

  @override
  State<Map3DView> createState() => Map3DViewState();
}

class Map3DViewState extends State<Map3DView> with SingleTickerProviderStateMixin {
  double _yaw = 0.9, _pitch = 0.62;
  bool _auto = true;
  late final AnimationController _ticker;
  final _painter = _SurfacePainter();

  @override
  void initState() {
    super.initState();
    _ticker = AnimationController(vsync: this, duration: const Duration(seconds: 1))
      ..addListener(() {
        if (_auto) _yaw += 0.0028;
        setState(() {});
      })
      ..repeat();
  }

  @override
  void dispose() {
    _ticker.dispose();
    super.dispose();
  }

  @override
  Widget build(BuildContext context) {
    return GestureDetector(
      onPanStart: (_) => _auto = false,
      onPanUpdate: (d) {
        _yaw += d.delta.dx * 0.006;
        _pitch = _pitch.clamp(0.25, 1.25) + d.delta.dy * 0.004;
      },
      onTapUp: (d) {
        final hit = _painter.pick(d.localPosition);
        if (hit != null && widget.onCell != null) widget.onCell!(hit.$1, hit.$2);
      },
      child: ClipRect(
        child: CustomPaint(
          painter: _painter
            ..update(
              grid: widget.grid,
              result: widget.result,
              yaw: _yaw,
              pitch: _pitch,
              selRi: widget.selRi,
              selCi: widget.selCi,
            ),
          size: Size.infinite,
        ),
      ),
    );
  }
}

class _Proj {
  _Proj(this.sx, this.sy, this.depth);
  final double sx, sy, depth;
}

class _Quad {
  _Quad(this.ri, this.ci, this.pts, this.depth, this.cx, this.cy, this.sideA, this.sideB);
  final int ri, ci;
  final List<Offset> pts;
  final List<Offset> sideA, sideB;
  final double depth, cx, cy;
}

class _SurfacePainter extends CustomPainter {
  MapGrid? grid;
  MapResult? result;
  double yaw = 0.9, pitch = 0.62;
  int? selRi, selCi;
  final List<_Quad> _quads = [];

  void update({
    required MapGrid grid,
    MapResult? result,
    required double yaw,
    required double pitch,
    int? selRi,
    int? selCi,
  }) {
    this.grid = grid;
    this.result = result;
    this.yaw = yaw;
    this.pitch = pitch;
    this.selRi = selRi;
    this.selCi = selCi;
  }

  (int, int)? pick(Offset p) {
    _Quad? best;
    var bd = 1e9;
    for (final q in _quads) {
      final d = math.sqrt(math.pow(q.cx - p.dx, 2) + math.pow(q.cy - p.dy, 2));
      if (d < 26 && d < bd) {
        bd = d;
        best = q;
      }
    }
    return best == null ? null : (best.ri, best.ci);
  }

  Color _colorFor(double t, double delta, int severity, bool sel) {
    var r = 24 + t * 30, g = 60 + t * 80, b = 96 + t * 100;
    if (delta < 0) {
      final k = 0.25 + severity * 0.22;
      r = r * (1 - k) + 255 * k;
      g = g * (1 - k) + 92 * k;
      b = b * (1 - k) + 92 * k;
    } else if (delta > 0) {
      final k = 0.25 + severity * 0.2;
      r = r * (1 - k) + 168 * k;
      g = g * (1 - k) + 255 * k;
      b = b * (1 - k) + 62 * k;
    }
    if (sel) {
      r = r * 0.35 + 235 * 0.65;
      g = g * 0.35 + 245 * 0.65;
      b = b * 0.35 + 255 * 0.65;
    }
    return Color.fromARGB(255, r.round(), g.round(), b.round());
  }

  int _severityOf(int ri, int ci) {
    final inf = result?.info[ri][ci];
    final d = result?.delta[ri][ci] ?? 0;
    if (d != 0 && d.abs() >= 2) return 3;
    if (d != 0) return 2;
    if (inf != null) return 1;
    return 0;
  }

  @override
  void paint(Canvas canvas, Size size) {
    final g = grid;
    if (g == null || g.rows == 0 || g.cols == 0) return;
    _quads.clear();

    final rows = g.rows, cols = g.cols;
    final vmin = g.vmin, vmax = g.vmax;
    final scale = math.min(size.width, size.height) * 0.34;
    final cx = size.width / 2, cy = size.height * 0.52;
    const cam = 3.4;

    _Proj proj(double x, double y, double z) {
      final x1 = x * math.cos(yaw) + z * math.sin(yaw);
      final z1 = -x * math.sin(yaw) + z * math.cos(yaw);
      final y2 = y * math.cos(pitch) - z1 * math.sin(pitch);
      final z2 = y * math.sin(pitch) + z1 * math.cos(pitch);
      final f = cam / (cam - z2);
      return _Proj(cx + x1 * scale * f, cy - y2 * scale * f, z2);
    }

    double hgt(double v) => 0.12 + ((v - vmin) / ((vmax - vmin) == 0 ? 1 : (vmax - vmin))) * 0.72;

    // Пол
    final floorPaint = Paint()
      ..color = const Color(0xFF1B2330)
      ..style = PaintingStyle.stroke
      ..strokeWidth = 1;
    final floor = [
      proj(-1, 0, -1),
      proj(1, 0, -1),
      proj(1, 0, 1),
      proj(-1, 0, 1),
    ];
    final fp = Path()
      ..moveTo(floor[0].sx, floor[0].sy)
      ..lineTo(floor[1].sx, floor[1].sy)
      ..lineTo(floor[2].sx, floor[2].sy)
      ..lineTo(floor[3].sx, floor[3].sy)
      ..close();
    canvas.drawPath(fp, floorPaint);

    for (var ri = 0; ri < rows; ri++) {
      for (var ci = 0; ci < cols; ci++) {
        final gz0 = (ri / (rows - 1)) * 2 - 1;
        final gz1 = rows > 1 ? ((ri + 1) / (rows - 1)) * 2 - 1 : gz0;
        final gx0 = (ci / (cols - 1)) * 2 - 1;
        final gx1 = cols > 1 ? ((ci + 1) / (cols - 1)) * 2 - 1 : gx0;
        const scx = 0.96;
        final x0 = gx0 + ((gx1 - gx0) * (1 - scx)) / 2;
        final x1 = gx1 - ((gx1 - gx0) * (1 - scx)) / 2;
        final z0 = gz0 + ((gz1 - gz0) * (1 - scx)) / 2;
        final z1 = gz1 - ((gz1 - gz0) * (1 - scx)) / 2;
        final h = hgt(g.data[ri][ci]);
        final p = [
          proj(x0, h, z0),
          proj(x1, h, z0),
          proj(x1, h, z1),
          proj(x0, h, z1),
        ];
        final depth = (p[0].depth + p[1].depth + p[2].depth + p[3].depth) / 4;
        final sideA = [
          proj(x0, h, z1),
          proj(x1, h, z1),
          proj(x1, 0, z1),
          proj(x0, 0, z1),
        ];
        final sideB = [
          proj(x1, h, z0),
          proj(x1, h, z1),
          proj(x1, 0, z1),
          proj(x1, 0, z0),
        ];
        _quads.add(_Quad(
          ri,
          ci,
          [for (final q in p) Offset(q.sx, q.sy)],
          depth,
          (p[0].sx + p[2].sx) / 2,
          (p[0].sy + p[2].sy) / 2,
          [for (final q in sideA) Offset(q.sx, q.sy)],
          [for (final q in sideB) Offset(q.sx, q.sy)],
        ));
      }
    }

    _quads.sort((a, b) => a.depth.compareTo(b.depth));

    final sidePaint = Paint()..color = const Color.fromARGB(235, 10, 14, 20);
    final edgePaint = Paint()
      ..style = PaintingStyle.stroke
      ..strokeWidth = 0.6
      ..color = const Color.fromARGB(140, 6, 8, 12);

    for (final q in _quads) {
      void poly(List<Offset> pts, Paint fill) {
        final path = Path()..moveTo(pts[0].dx, pts[0].dy);
        for (var i = 1; i < pts.length; i++) {
          path.lineTo(pts[i].dx, pts[i].dy);
        }
        path.close();
        canvas.drawPath(path, fill);
      }

      poly(q.sideA, sidePaint);
      poly(q.sideB, sidePaint);

      final v = g.data[q.ri][q.ci];
      final t = (v - vmin) / ((vmax - vmin) == 0 ? 1 : (vmax - vmin));
      final d = result?.delta[q.ri][q.ci] ?? 0;
      final sel = selRi == q.ri && selCi == q.ci;
      poly(
        q.pts,
        Paint()..color = _colorFor(t, d, _severityOf(q.ri, q.ci), sel),
      );
      final topPath = Path()..moveTo(q.pts[0].dx, q.pts[0].dy);
      for (var i = 1; i < q.pts.length; i++) {
        topPath.lineTo(q.pts[i].dx, q.pts[i].dy);
      }
      topPath.close();
      if (sel) {
        canvas.drawPath(
            topPath,
            Paint()
              ..style = PaintingStyle.stroke
              ..strokeWidth = 1.6
              ..color = const Color(0xFFF0F9FF));
      } else {
        canvas.drawPath(topPath, edgePaint);
      }

      final dAbs = d.abs();
      if (d != 0 && dAbs >= 2) {
        canvas.drawCircle(
          Offset(q.cx, q.cy),
          2.4,
          Paint()..color = d < 0 ? const Color(0xFFFF5C5C) : const Color(0xFFA8FF3E),
        );
      }
    }

    // Подписи осей и шкалы.
    void label(String text, Offset at,
        {double size = 10, Color color = const Color(0xD98FA3BE), bool center = true}) {
      final tp = TextPainter(
        text: TextSpan(
            text: text, style: TextStyle(color: color, fontSize: size, fontFamily: 'monospace')),
        textDirection: TextDirection.ltr,
      )..layout();
      final off = center ? Offset(at.dx - tp.width / 2, at.dy) : at;
      tp.paint(canvas, off);
    }

    final ax0 = proj(-1.15, 0, 1.12);
    final ax1 = proj(1.12, 0, 1.12);
    label('${g.x.name} · ${g.x.guess}', Offset((ax0.sx + ax1.sx) / 2, math.max(ax0.sy, ax1.sy) + 14));

    final ay0 = proj(-1.15, 0, -1.05);
    canvas.save();
    canvas.translate(ay0.sx - 16, (ay0.sy + ax0.sy) / 2);
    canvas.rotate(-math.pi / 2);
    label('${g.y.name} · ${g.y.guess}', Offset.zero);
    canvas.restore();

    final hTop = proj(-1, 0.92, -1);
    label(vmax.toStringAsFixed(2), Offset(hTop.sx + 6, hTop.sy), color: const Color(0xCC4BE1FF), center: false);
    final hBot = proj(-1, 0.06, -1);
    label(vmin.toStringAsFixed(2), Offset(hBot.sx + 6, hBot.sy), center: false);
  }

  @override
  bool shouldRepaint(covariant _SurfacePainter oldDelegate) => true;
}
"""
FILES["lib/maps_lab/maps_lab_page.dart"] = r"""
/// Экран Map Lab: ROM + CSV-лог → вердикты по ячейкам карт, 2D/3D вид, экспорт.
///
/// Ноль новых pub-зависимостей: плагинов не требуется, кроме уже имеющихся
/// в проекте (path_provider, share_plus, permission_handler).
///
/// Подключение: добавьте в своё приложение кнопку
///   Navigator.push(context, MaterialPageRoute(builder: (_) => const MapLabPage()));
library;

import 'dart:convert';
import 'dart:io';

import 'package:flutter/material.dart';
import 'package:flutter/services.dart' show rootBundle;
import 'package:path_provider/path_provider.dart';
import 'package:permission_handler/permission_handler.dart';
import 'package:share_plus/share_plus.dart';

import 'map3d_view.dart';
import 'maps_lab_core.dart';
import 'maps_lab_log.dart';

/// Точки входа файлов: сначала своя папка приложения (нулевые разрешения),
/// общие папки — только если выдан MANAGE_EXTERNAL_STORAGE.
class MapLabPage extends StatefulWidget {
  const MapLabPage({super.key});

  @override
  State<MapLabPage> createState() => _MapLabPageState();
}

class _MapLabPageState extends State<MapLabPage> {
  static const _defUrls = {
    'A2TB100B':
        'https://raw.githubusercontent.com/TD-D/SubaruDefs/Stable/ECUFlash/subaru%20metric/Legacy%20GT/A2TB100B.xml',
    'A2TB100K':
        'https://raw.githubusercontent.com/TD-D/SubaruDefs/Stable/ECUFlash/subaru%20metric/Legacy%20GT%20spec.B/A2TB100K.xml',
    '32BITBASE':
        'https://raw.githubusercontent.com/TD-D/SubaruDefs/Stable/ECUFlash/subaru%20metric/Bases/32BITBASE.xml',
  };

  final _journal = <String>[];
  DefSet? _defs;
  Map<String, MapGrid>? _maps;
  LogData? _log;
  Map<String, MapResult>? _results;
  String? _mapName;
  int? _selRi, _selCi;
  bool _view3D = false;
  String? _docsPath;

  @override
  void initState() {
    super.initState();
    // стартуем после первого кадра — context гарантированно валиден
    WidgetsBinding.instance.addPostFrameCallback((_) => _loadDefs());
  }

  void _say(String s) {
    if (!mounted) return;
    setState(() => _journal.add('${TimeOfDay.now().format(context)}  $s'));
  }

  // ─────────────────────────────────────────────── дефиниции: assets → кэш → github
  Future<String> _fetch(String url) async {
    final c = HttpClient();
    try {
      final req = await c.getUrl(Uri.parse(url));
      final res = await req.close();
      if (res.statusCode != 200) throw HttpException('HTTP ${res.statusCode}');
      return await res.transform(utf8.decoder).join();
    } finally {
      c.close();
    }
  }

  Future<void> _loadDefs() async {
    final docs = await getApplicationDocumentsDirectory();
    _docsPath = docs.path;
    final dir = Directory('${docs.path}/defs')..createSync(recursive: true);
    final xmlById = <String, String>{};
    var src = 'assets APK';
    for (final e in _defUrls.entries) {
      try {
        xmlById[e.key] = await rootBundle.loadString('assets/defs/${e.key}.xml');
        continue;
      } catch (_) {}
      final cache = File('${dir.path}/${e.key}.xml');
      if (cache.existsSync() && cache.lengthSync() > 1000) {
        xmlById[e.key] = await cache.readAsString();
        src = 'кэш телефона';
        continue;
      }
      try {
        xmlById[e.key] = await _fetch(e.value);
        await cache.writeAsString(xmlById[e.key]!);
        src = 'github (кэшировано)';
      } catch (err) {
        _say('!! ${e.key}: $err');
      }
    }
    if (xmlById.length == 3) {
      _defs = DefSet.build(xmlById, 'A2TB100B');
      _say('дефиниции [$src]: ${_defs!.chain.join(' → ')} · '
          'ecuid ${_defs!.meta.ecuid} · ${_defs!.tables.length} таблиц');
    } else {
      _say('!! дефиниций не хватает (${xmlById.length}/3) — без них ROM не распарсить');
    }
  }

  // ─────────────────────────────────────────────── выбор файлов без плагинов
  Future<Directory?> _docsDir() async {
    _docsPath ??= (await getApplicationDocumentsDirectory()).path;
    return Directory(_docsPath!);
  }

  Future<List<File>> _scanFiles(List<String> exts) async {
    final dirs = <Directory>[];
    final docs = await _docsDir();
    if (docs != null) dirs.add(docs);
    var sdOk = false;
    try {
      sdOk = await Permission.manageExternalStorage.isGranted;
    } catch (_) {}
    if (sdOk) {
      for (final p in ['/storage/emulated/0/Download', '/sdcard/Download', '/sdcard/Documents']) {
        final d = Directory(p);
        if (d.existsSync()) dirs.add(d);
      }
    }
    final out = <File>[];
    for (final d in dirs) {
      try {
        await for (final e in d.list(recursive: true, followLinks: false)) {
          if (e is File && exts.any((x) => e.path.toLowerCase().endsWith(x))) out.add(e);
        }
      } catch (_) {}
    }
    out.sort((a, b) => b.lastModifiedSync().compareTo(a.lastModifiedSync()));
    return out;
  }

  Future<File?> _choose(List<String> exts, String title) async {
    final found = await _scanFiles(exts);
    if (!mounted) return null;
    if (found.isEmpty) {
      ScaffoldMessenger.of(context).showSnackBar(SnackBar(
        content: Text('Файлы $exts не найдены в $_docsPath. '
            'Положите их в папку приложения или разрешите «доступ ко всем файлам» в настройках.'),
      ));
      return null;
    }
    return showModalBottomSheet<File>(
      context: context,
      isScrollControlled: true,
      builder: (ctx) => SafeArea(
        child: SizedBox(
          height: MediaQuery.of(ctx).size.height * 0.6,
          child: Column(
            children: [
              Padding(
                padding: const EdgeInsets.all(12),
                child: Text(title, style: const TextStyle(fontWeight: FontWeight.w600)),
              ),
              const Divider(height: 1),
              Expanded(
                child: ListView.builder(
                  itemCount: found.length,
                  itemBuilder: (_, i) {
                    final f = found[i];
                    final sizeKb = f.lengthSync() / 1024;
                    return ListTile(
                      dense: true,
                      leading: const Icon(Icons.insert_drive_file_outlined, size: 20),
                      title: Text(f.uri.pathSegments.last,
                          style: const TextStyle(fontFamily: 'monospace', fontSize: 12)),
                      subtitle: Text(
                        '${f.parent.path} · ${sizeKb.toStringAsFixed(0)} КБ',
                        style: const TextStyle(fontSize: 10),
                        overflow: TextOverflow.ellipsis,
                      ),
                      onTap: () => Navigator.of(ctx).pop(f),
                    );
                  },
                ),
              ),
            ],
          ),
        ),
      ),
    );
  }

  Future<void> _pickRom() async {
    final f = await _choose(['.bin', '.hex', '.rom'], 'Прошивка — выберите .bin рид ЭБУ');
    if (f == null) return;
    try {
      final bytes = await f.readAsBytes();
      setState(() {
        _results = null;
        _selRi = _selCi = null;
      });
      final parser = RomParser(bytes);
      final idAddr = int.tryParse(_defs?.meta.internalIdAddress ?? '2000', radix: 16) ?? 0x2000;
      final romId = parser.readRomId(idAddr);
      final matched = romId == (_defs?.meta.internalIdString ?? '');
      _say('ROM: ${f.uri.pathSegments.last} · ${(bytes.length / 1024).toStringAsFixed(0)} КБ · '
          'ID="$romId"${matched ? '' : '  (ожидался ${_defs?.meta.internalIdString}!)'}');
      if (_defs == null) {
        _say('!! дефиниции не загружены — парсинг невозможен');
        return;
      }
      _maps = parser.extractKeys(_defs!);
      _mapName = _maps!.keys.firstOrNull;
      _say('карт извлечено: ${_maps!.length} из ${RomParser.keyTables.length}');
    } catch (e) {
      _say('!! ошибка парсинга ROM: $e');
    }
  }

  Future<void> _pickLog() async {
    final f = await _choose(['.csv', '.txt', '.log'], 'Лог — выберите CSV из SSM2 Fixed');
    if (f == null) return;
    try {
      final text = utf8.decode(await f.readAsBytes(), allowMalformed: true);
      _log = LogData.parse(text);
      final health = LogAudit.check(_log!);
      _say('лог: ${f.uri.pathSegments.last} · ${_log!.rows} строк');
      for (final n in health.notes) {
        _say('   $n');
      }
      for (final e in health.missing.entries) {
        _say('   блок ${e.key}: ${e.value.isEmpty ? 'OK' : 'ПРОПУСК — нет ${e.value.join(', ')}'}');
      }
    } catch (e) {
      _say('!! ошибка чтения лога: $e');
    }
  }

  void _runAnalysis() {
    if (_maps == null || _log == null) return;
    const cfg = AnalyzerConfig();
    _results = Analyzer(cfg).run(_maps!, _log!);
    var dec = 0, inc = 0;
    for (final r in _results!.values) {
      dec += r.dec;
      inc += r.inc;
    }
    _say('анализ: вердикты по ${_results!.length} картам · убавить $dec, прибавить $inc ячеек');
    setState(() {});
  }

  Future<void> _export() async {
    if (_maps == null || _results == null) return;
    final dir = await getApplicationDocumentsDirectory();
    final stamp = DateTime.now().toIso8601String().replaceAll(':', '-').split('.').first;
    final outDir = Directory('${dir.path}/maplab_$stamp')..createSync(recursive: true);
    final files = <XFile>[];
    final md = StringBuffer('# Map Lab A2TB100B · рекомендации\n');
    for (final e in _maps!.entries) {
      final g = e.value;
      final res = _results![e.key];
      if (res == null) continue;
      final head = '\t${g.x.values.map((v) => v.toStringAsFixed(2)).join('\t')}';
      final rows = <String>[
        for (var ri = 0; ri < g.rows; ri++)
          '${g.y.values[ri].toStringAsFixed(2)}\t${[
            for (var ci = 0; ci < g.cols; ci++)
              (g.data[ri][ci] + res.delta[ri][ci]).toStringAsFixed(3),
          ].join('\t')}',
      ];
      final tsv = '$head\n${rows.join('\n')}\n';
      final file = File('${outDir.path}/${e.key.replaceAll(RegExp(r'[/ ]'), '_')}_recommended.tsv')
        ..writeAsStringSync(tsv);
      files.add(XFile(file.path));
      md.writeln('\n## ${e.key} (${g.units})');
      for (var ri = 0; ri < g.rows; ri++) {
        for (var ci = 0; ci < g.cols; ci++) {
          final d = res.delta[ri][ci];
          if (d == 0) continue;
          final inf = res.info[ri][ci];
          md.writeln('- ${g.x.values[ci]} × ${g.y.values[ri]}: ${g.data[ri][ci]} → '
              '${(g.data[ri][ci] + d).toStringAsFixed(2)} ${g.units} (${d > 0 ? '+' : ''}$d)'
              '${inf != null && inf.why.isNotEmpty ? ' — ${inf.why}' : ''}');
        }
      }
    }
    final mf = File('${outDir.path}/recommendations.md')..writeAsStringSync(md.toString());
    files.insert(0, XFile(mf.path));
    _say('экспорт: ${files.length} файлов → ${outDir.path}');
    await Share.shareXFiles(files, text: 'Map Lab A2TB100B — рекомендации анализатора');
  }

  // ─────────────────────────────────────────────── UI
  @override
  Widget build(BuildContext context) {
    final theme = Theme.of(context);
    final map = _maps == null ? null : _maps![_mapName];
    MapResult? res;
    if (map != null) res = _results?[map.name];

    return Scaffold(
      appBar: AppBar(
        title: const Text('Map Lab · A2TB100B'),
        actions: [
          if (_results != null)
            IconButton(icon: const Icon(Icons.ios_share), onPressed: _export, tooltip: 'Экспорт'),
        ],
      ),
      body: ListView(
        padding: const EdgeInsets.all(12),
        children: [
          Wrap(
            spacing: 8,
            runSpacing: 8,
            children: [
              _stepBtn('1 · Прошивка (.bin)', Icons.memory, _pickRom, ok: _maps != null),
              _stepBtn('2 · Лог (.csv)', Icons.description, _pickLog, ok: _log != null),
              _stepBtn('3 · Анализ', Icons.psychology,
                  (_maps != null && _log != null) ? _runAnalysis : null,
                  ok: _results != null),
              _stepBtn('Экспорт', Icons.ios_share, _results != null ? _export : null),
            ],
          ),
          const SizedBox(height: 12),
          if (_maps != null)
            SizedBox(
              height: 44,
              child: ListView(
                scrollDirection: Axis.horizontal,
                children: [
                  for (final e in _maps!.entries)
                    Padding(
                      padding: const EdgeInsets.only(right: 8),
                      child: ChoiceChip(
                        selected: _mapName == e.key,
                        onSelected: (_) => setState(() {
                          _mapName = e.key;
                          _selRi = _selCi = null;
                        }),
                        label: Column(
                          crossAxisAlignment: CrossAxisAlignment.start,
                          children: [
                            Text(e.key, style: const TextStyle(fontSize: 11)),
                            Text(e.value.units,
                                style: TextStyle(fontSize: 9, color: theme.hintColor)),
                          ],
                        ),
                      ),
                    ),
                ],
              ),
            ),
          if (map != null) ...[
            Row(
              children: [
                Expanded(
                  child: Text(
                    '${map.name} · 0x${map.addr.toRadixString(16).toUpperCase().padLeft(6, '0')} · '
                    '${map.rows}×${map.cols}',
                    style: theme.textTheme.bodySmall,
                    overflow: TextOverflow.ellipsis,
                  ),
                ),
                SegmentedButton<bool>(
                  segments: const [
                    ButtonSegment(value: false, icon: Icon(Icons.table_chart, size: 16)),
                    ButtonSegment(value: true, icon: Icon(Icons.threed_rotation, size: 16)),
                  ],
                  selected: {_view3D},
                  onSelectionChanged: (s) => setState(() => _view3D = s.first),
                ),
              ],
            ),
            const SizedBox(height: 8),
            SizedBox(
              height: 380,
              child: Card(
                clipBehavior: Clip.antiAlias,
                child: _view3D
                    ? Map3DView(
                        grid: map,
                        result: res,
                        selRi: _selRi,
                        selCi: _selCi,
                        onCell: (ri, ci) => setState(() {
                          _selRi = ri;
                          _selCi = ci;
                        }),
                      )
                    : _Table2D(
                        grid: map,
                        result: res,
                        selRi: _selRi,
                        selCi: _selCi,
                        onCell: (ri, ci) => setState(() {
                          _selRi = ri;
                          _selCi = ci;
                        }),
                      ),
              ),
            ),
            if (_selRi != null && _selCi != null) _inspector(map, res),
            const SizedBox(height: 8),
            Wrap(spacing: 12, runSpacing: 4, children: [
              _legend(const Color(0xFFFF5C5C), 'убавить (детон/овербуст/богато)'),
              _legend(const Color(0xFFA8FF3E), 'прибавить (чисто/недобор)'),
              _legend(const Color(0xFF3D6484), 'нейтральные'),
            ]),
          ],
          const SizedBox(height: 12),
          ExpansionTile(
            initiallyExpanded: _maps == null,
            tilePadding: EdgeInsets.zero,
            title: Text('Журнал', style: theme.textTheme.titleSmall),
            children: [
              Container(
                width: double.infinity,
                constraints: const BoxConstraints(maxHeight: 260),
                padding: const EdgeInsets.all(10),
                decoration: BoxDecoration(
                  color: theme.colorScheme.surfaceContainerHighest.withValues(alpha: 0.35),
                  borderRadius: BorderRadius.circular(8),
                ),
                child: ListView(
                  shrinkWrap: true,
                  children: [
                    for (final j in _journal)
                      Text(j, style: const TextStyle(fontFamily: 'monospace', fontSize: 11)),
                  ],
                ),
              ),
            ],
          ),
          const SizedBox(height: 24),
        ],
      ),
    );
  }

  Widget _stepBtn(String label, IconData icon, VoidCallback? onTap, {bool ok = false}) {
    return FilledButton.tonalIcon(
      onPressed: onTap,
      icon: Icon(ok ? Icons.check_circle : icon, size: 18),
      label: Text(label, style: const TextStyle(fontSize: 12)),
    );
  }

  Widget _legend(Color c, String t) => Row(mainAxisSize: MainAxisSize.min, children: [
        Container(
            width: 12,
            height: 8,
            decoration: BoxDecoration(color: c, borderRadius: BorderRadius.circular(2))),
        const SizedBox(width: 6),
        Text(t, style: const TextStyle(fontSize: 10.5)),
      ]);

  Widget _inspector(MapGrid g, MapResult? res) {
    final ri = _selRi!, ci = _selCi!;
    final v = g.data[ri][ci];
    final d = res?.delta[ri][ci] ?? 0;
    final inf = res?.info[ri][ci];
    final col = d < 0 ? const Color(0xFFFF5C5C) : d > 0 ? const Color(0xFFA8FF3E) : null;
    return Card(
      margin: const EdgeInsets.only(top: 8),
      child: Padding(
        padding: const EdgeInsets.all(12),
        child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
          Text('Ячейка: ${g.x.name}=${g.x.values[ci]} · ${g.y.name}=${g.y.values[ri]}',
              style: const TextStyle(fontFamily: 'monospace', fontSize: 12)),
          const SizedBox(height: 6),
          Row(children: [
            Text(v.toStringAsFixed(2),
                style: const TextStyle(
                    fontSize: 20, fontFamily: 'monospace', decoration: TextDecoration.lineThrough)),
            const SizedBox(width: 8),
            const Icon(Icons.arrow_forward, size: 16),
            const SizedBox(width: 8),
            Text('${(v + d).toStringAsFixed(2)} ${g.units}',
                style: TextStyle(fontSize: 24, fontFamily: 'monospace', color: col)),
            if (d != 0)
              Padding(
                padding: const EdgeInsets.only(left: 8),
                child: Chip(
                  visualDensity: VisualDensity.compact,
                  label: Text('${d > 0 ? '+' : ''}${d.toStringAsFixed(1)}'),
                ),
              ),
          ]),
          if (inf != null) ...[
            const SizedBox(height: 6),
            Text(
                'n=${inf.n}'
                '${inf.fbkc != null ? ' · FBKC ${inf.fbkc!.toStringAsFixed(1)}°' : ''}'
                '${inf.flkc != null ? ' · FLKC ${inf.flkc!.toStringAsFixed(1)}°' : ''}'
                '${inf.iat != null ? ' · IAT ${inf.iat!.toStringAsFixed(0)}°C' : ''}',
                style: const TextStyle(fontSize: 11.5)),
            if (inf.why.isNotEmpty)
              Padding(
                padding: const EdgeInsets.only(top: 4),
                child: Text(inf.why, style: const TextStyle(fontSize: 12)),
              ),
          ] else
            const Padding(
              padding: EdgeInsets.only(top: 4),
              child: Text('Вне статистики — в логе нет сэмплов этой зоны.',
                  style: TextStyle(fontSize: 12)),
            ),
        ]),
      ),
    );
  }
}

// ────────────────────────────────────────────────────────── 2D таблица
class _Table2D extends StatelessWidget {
  const _Table2D({required this.grid, this.result, this.selRi, this.selCi, this.onCell});
  final MapGrid grid;
  final MapResult? result;
  final int? selRi, selCi;
  final void Function(int, int)? onCell;

  @override
  Widget build(BuildContext context) {
    final g = grid;
    return InteractiveViewer(
      constrained: false,
      child: Padding(
        padding: const EdgeInsets.all(8),
        child: Table(
          defaultColumnWidth: const IntrinsicColumnWidth(),
          children: [
            TableRow(children: [
              _head('${g.y.guess}↓ ${g.x.guess}→'),
              for (final v in g.x.values) _head(v.toStringAsFixed(0), accent: true),
            ]),
            for (var ri = 0; ri < g.rows; ri++)
              TableRow(children: [
                _head(g.y.values[ri].toStringAsFixed(2), accent: true),
                for (var ci = 0; ci < g.cols; ci++) _cell(ri, ci),
              ]),
          ],
        ),
      ),
    );
  }

  Widget _head(String t, {bool accent = false}) => Container(
        padding: const EdgeInsets.symmetric(horizontal: 6, vertical: 4),
        decoration: BoxDecoration(border: Border.all(color: Colors.white12)),
        child: Text(t,
            style: TextStyle(
                fontSize: 9.5,
                fontFamily: 'monospace',
                color: accent ? const Color(0xCC4BE1FF) : Colors.white54)),
      );

  Widget _cell(int ri, int ci) {
    final v = grid.data[ri][ci];
    final d = result?.delta[ri][ci] ?? 0;
    final sel = selRi == ri && selCi == ci;
    Color? bg;
    if (d < 0) bg = const Color(0x33FF5C5C);
    if (d > 0) bg = const Color(0x2AA8FF3E);
    final inf = result?.info[ri][ci];
    return InkWell(
      onTap: () => onCell?.call(ri, ci),
      child: Container(
        constraints: const BoxConstraints(minWidth: 44),
        padding: const EdgeInsets.symmetric(horizontal: 4, vertical: 4),
        decoration: BoxDecoration(
          color: bg,
          border: Border.all(
              color: sel
                  ? Colors.white
                  : inf != null
                      ? Colors.white24
                      : Colors.white10,
              width: sel ? 1.5 : 0.5),
        ),
        child: Column(children: [
          Text(v.toStringAsFixed(1),
              style: const TextStyle(fontSize: 10, fontFamily: 'monospace')),
          if (d != 0)
            Text('${d > 0 ? '+' : ''}${d.toStringAsFixed(1)}',
                style: TextStyle(
                    fontSize: 8,
                    fontFamily: 'monospace',
                    color: d < 0 ? const Color(0xFFFF5C5C) : const Color(0xFFA8FF3E))),
        ]),
      ),
    );
  }
}
"""
FILES["test/maps_lab_test.dart"] = r"""
// Тесты Map Lab: формулы скейлинга, merge include-цепочки, декод ROM, правила.
import 'dart:typed_data';

import 'package:flutter_test/flutter_test.dart';
import 'package:__PKG__/maps_lab/maps_lab_core.dart';
import 'package:__PKG__/maps_lab/maps_lab_log.dart';

void main() {
  group('SafeExpr', () {
    test('uint8 timing: (x*.3515625)-20', () {
      final f = SafeExpr.compile('(x*.3515625)-20');
      expect(f(100), closeTo(15.15625, 0.0001));
      expect(f(0), -20.0);
    });
    test('estimated AFR: 14.7/(1+x*.0078125)', () {
      final f = SafeExpr.compile('14.7/(1+x*.0078125)');
      expect(f(80), closeTo(9.0461, 0.001));
    });
    test('float identity + garbage guard', () {
      expect(SafeExpr.compile('x')(42.5), 42.5);
      expect(SafeExpr.compile('eval(x)+1')(3), 3); // формула отброшена → identity
    });
  });

  const baseXml = '''
<rom>
  <romid><xmlid>32BITBASE</xmlid></romid>
  <scaling name="Timing8" units="deg" toexpr="(x*.3515625)-20" frexpr="(x+20)/.3515625" format="%.2f" storagetype="uint8" endian="big"/>
  <scaling name="RPM" units="RPM" toexpr="x" frexpr="x" format="%.0f" storagetype="float" endian="big"/>
  <table name="Base Timing" category="Ignition" type="3D" level="4" scaling="Timing8">
    <table name="Engine Load" type="X Axis" elements="2" scaling="RPM"/>
    <table name="Engine Speed" type="Y Axis" elements="3" scaling="RPM"/>
  </table>
</rom>''';

  const derivedXml = '''
<rom>
  <romid><xmlid>TEST_ROM</xmlid><internalidaddress>2000</internalidaddress><internalidstring>TEST_ROM</internalidstring></romid>
  <include>32BITBASE</include>
  <table name="Base Timing" address="1000">
    <table name="X" address="2400" elements="4"/>
    <table name="Y" address="2500"/>
  </table>
</rom>''';

  test('merge include-цепочки: адрес у производного, скейлинг наследуется', () {
    final defs = DefSet.build({'32BITBASE': baseXml, 'TEST_ROM': derivedXml}, 'TEST_ROM');
    expect(defs.chain, ['32BITBASE', 'TEST_ROM']);
    final t = defs.tables['Base Timing']!;
    expect(t.dataAddress, 0x1000);
    expect(t.dataScaling!.units, 'deg');
    expect(t.axes[0].name, 'Engine Load'); // имя наследовано, «X» его не затирает
    expect(t.axes[0].elements, 4); // elements у производного перевесили базовые 2
    expect(t.axes[0].address, 0x2400);
    expect(t.axes[1].elements, 3); // унаследовано из базы
    expect(defs.isReadable3D(t), isTrue);
  });

  test('ROM: декод float32-осей и uint8-данных big-endian', () {
    final rom = Uint8List(0x3000);
    final bd = ByteData.view(rom.buffer);
    // ID калибровки
    for (var i = 0; i < 8; i++) {
      bd.setUint8(0x2000 + i, 'TEST_ROM'.codeUnitAt(i));
    }
    // X axis: 4 float32 BE = 1000,2000,3000,4000
    final xs = [1000.0, 2000.0, 3000.0, 4000.0];
    for (var i = 0; i < 4; i++) {
      bd.setFloat32(0x2400 + i * 4, xs[i], Endian.big);
    }
    final ys = [0.8, 1.6, 2.4];
    for (var i = 0; i < 3; i++) {
      bd.setFloat32(0x2500 + i * 4, ys[i], Endian.big);
    }
    // данные 3×4 uint8: значение x=100 → 15.15625°
    for (var i = 0; i < 12; i++) {
      bd.setUint8(0x1000 + i, 100);
    }

    final defs = DefSet.build({'32BITBASE': baseXml, 'TEST_ROM': derivedXml}, 'TEST_ROM');
    final parser = RomParser(rom);
    expect(parser.readRomId(0x2000), 'TEST_ROM');
    final grid = parser.extract(defs.tables['Base Timing']!)!;
    expect(grid.rows, 3);
    expect(grid.cols, 4);
    expect(grid.x.values.last, 4000);
    expect(grid.y.values.first, closeTo(0.8, 0.001));
    expect(grid.data[2][3], closeTo(15.15625, 0.001));
    expect(grid.kind, 'timing');
  });

  group('LogData + Analyzer', () {
    LogData syntheticLog({required double fbkc}) {
      // 600 сэмплов: rpm 4000±40, load 3.0 — кластер в одной ячейке
      final rpm = <double?>[], load = <double?>[], f = <double?>[], fl = <double?>[];
      for (var i = 0; i < 600; i++) {
        rpm.add(4000 + (i % 80) - 40);
        load.add(3.0);
        f.add(fbkc);
        fl.add(fbkc * 0.4);
      }
      final log = LogData()
        ..cols['rpm'] = rpm
        ..cols['load'] = load
        ..cols['fbkc'] = f
        ..cols['flkc'] = fl;
      return log;
    }

    MapGrid smallGrid() => MapGrid(
          name: 'Base Timing Primary Non-Cruise',
          kind: 'timing',
          units: 'deg',
          addr: 0,
          x: AxisVals('Engine Speed', [3000, 4000, 5000], 'rpm'),
          y: AxisVals('Engine Load', [2.0, 3.0], 'г/об·бар'),
          data: [
            [20, 18, 16],
            [16, 14, 12],
          ],
        );

    test('детон-кластер → отрицательная дельта только в своей ячейке', () {
      final res = Analyzer(const AnalyzerConfig()).analyzeMap(smallGrid(), syntheticLog(fbkc: -4.0))!;
      final d = res.delta[1][1];
      expect(d, lessThan(0));
      expect(d, greaterThanOrEqualTo(-3.0));
      expect(res.delta[0][0], 0.0);
      expect(res.info[1][1]!.why, contains('детон'));
    });

    test('чистый лог при выключенном allowTimingAdd → ноль правок', () {
      final res = Analyzer(const AnalyzerConfig()).analyzeMap(smallGrid(), syntheticLog(fbkc: 0))!;
      final anyDelta = res.delta.expand((r) => r).any((v) => v != 0);
      expect(anyDelta, isFalse);
    });

    test('CSV: автоопределение ; и десятичной запятой, алиасы колонок', () {
      const csv = 'Time (s);Engine Speed (RPM);Feedback Knock Correction;Load_4B\r\n'
          '0,0;3000;0,0;3,00\r\n'
          '0,1;3100;-2,5;3,10\r\n';
      final log = LogData.parse(csv);
      expect(log['rpm']![1], 3100);
      expect(log['fbkc']![1], -2.5);
      expect(log['load']![1], 3.1);
    });
  });
}
"""

# ── pubspec: только assets (дефиниции в APK). Новых плагинов НЕТ намеренно:
# prepare_bt перезаписывает pubspec, а наше ядро работает без единого нового пакета.
pub = APP / "pubspec.yaml"
text = pub.read_text(encoding="utf-8")

if "assets/defs/" not in text:
    try:
        if re.search(r"(?m)^  assets:\s*$", text):
            text = re.sub(r"(?m)^(  assets:\s*\n)", r"\1    - assets/defs/\n", text, count=1)
        elif re.search(r"(?m)^flutter:\s*$", text):
            text = re.sub(r"(?m)^(flutter:\s*\n)", r"\1  assets:\n    - assets/defs/\n", text, count=1)
        else:
            text += "\nflutter:\n  uses-material-design: true\n  assets:\n    - assets/defs/\n"
        pub.write_text(text, encoding="utf-8")
        print("[pubspec] assets/defs добавлены")
    except Exception as e:
        print(f"[pubspec] пропуск ({e}) — дефиниции скачаются на телефоне")
else:
    print("[pubspec] assets уже на месте")

# мягкий патч манифеста (опционально): доступ к Download, если разрешите руками
man = APP / "android/app/src/main/AndroidManifest.xml"
try:
    if man.exists():
        m = man.read_text(encoding="utf-8")
        if "MANAGE_EXTERNAL_STORAGE" not in m:
            m = m.replace("<application",
                '    <uses-permission android:name="android.permission.MANAGE_EXTERNAL_STORAGE"/>\n    <application', 1)
            man.write_text(m, encoding="utf-8")
            print("[manifest] MANAGE_EXTERNAL_STORAGE добавлен (если prepare_bt перезапишет — не страшно)")
except Exception as e:
    print(f"[manifest] пропуск ({e}) — приложение будет работать из своей папки")

pkg_match = re.search(r"(?m)^name:\s*(\S+)", text)
PKG = pkg_match.group(1) if pkg_match else "subaru_ssm2_fixed"

# ── запись dart-файлов ───────────────────────────────────────────────────────
for rel, src in FILES.items():
    dest = APP / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_text(src.replace("__PKG__", PKG) + "\n", encoding="utf-8")
    print("[dart]", rel)

# ── assets: дефиниции в APK ─────────────────────────────────────────────────-
RAW = "https://raw.githubusercontent.com/TD-D/SubaruDefs/Stable/ECUFlash/subaru%20metric"
DEFS = {
    "A2TB100B.xml": f"{RAW}/Legacy%20GT/A2TB100B.xml",
    "A2TB100K.xml": f"{RAW}/Legacy%20GT%20spec.B/A2TB100K.xml",
    "32BITBASE.xml": f"{RAW}/Bases/32BITBASE.xml",
}
defs_dir = APP / "assets/defs"
defs_dir.mkdir(parents=True, exist_ok=True)
for name, url in DEFS.items():
    dest = defs_dir / name
    if not dest.exists() or dest.stat().st_size < 1000:
        urllib.request.urlretrieve(url, dest)
    print("[asset]", name, f"{dest.stat().st_size/1024:.0f} КБ")

print(f"""
Готово. Пакет приложения: {PKG}

Остался один ручной шаг — кнопка входа в Map Lab. Откройте lib/main.dart и добавьте:

    import 'package:{PKG}/maps_lab/maps_lab_page.dart';

    // например, в AppBar вашего экрана логирования:
    IconButton(
      icon: const Icon(Icons.analytics),
      tooltip: 'Map Lab',
      onPressed: () => Navigator.push(
        context, MaterialPageRoute(builder: (_) => const MapLabPage())),
    ),

Дальше выполните вашу ячейку 3/3: она сделает pub get, analyze, тесты и соберёт APK.
""")

[pubspec] assets/defs добавлены
[dart] lib/maps_lab/maps_lab_core.dart
[dart] lib/maps_lab/maps_lab_log.dart
[dart] lib/maps_lab/map3d_view.dart
[dart] lib/maps_lab/maps_lab_page.dart
[dart] test/maps_lab_test.dart
[asset] A2TB100B.xml 2 КБ
[asset] A2TB100K.xml 28 КБ
[asset] 32BITBASE.xml 489 КБ

Готово. Пакет приложения: subaru_ssm2

Остался один ручной шаг — кнопка входа в Map Lab. Откройте lib/main.dart и добавьте:

    import 'package:subaru_ssm2/maps_lab/maps_lab_page.dart';

    // например, в AppBar вашего экрана логирования:
    IconButton(
      icon: const Icon(Icons.analytics),
      tooltip: 'Map Lab',
      onPressed: () => Navigator.push(
        context, MaterialPageRoute(builder: (_) => const MapLabPage())),
    ),

Дальше выполните вашу ячейку 3/3: она сделает pub get, analyze, тесты и соберёт APK.



In [13]:
# @title 3/3 | SSM2 0.6 - Анализ, тесты, выбранная сборка и проверка APK { display-mode: "form" }
import hashlib
import json
import os
from pathlib import Path
import re
import shutil
import subprocess
import sys
import time
import urllib.request

BUILD_VARIANT = "release"  # @param ["release", "debug", "profile"]
# Выбор другого транспорта восстанавливает его полный исходник из templates.
TRANSPORT = "from_cell_2"  # @param ["from_cell_2", "bluetooth_classic", "flutter_bluetooth_serial"]
CLEAN_BEFORE = False  # @param {type:"boolean"}
APP = Path("/content/subaru_ssm2_fixed")
CONFIG = APP / "build_config.json"
if not CONFIG.exists():
    raise RuntimeError("Не найден новый проект. Выполните новые ячейки 1 и 2.")
cfg = json.loads(CONFIG.read_text(encoding="utf-8"))
if cfg.get("revision") != "0.6":
    raise RuntimeError("Смешаны версии ячеек. Используйте весь комплект 0.6.")
FLUTTER = Path(cfg["flutter"])
SDK = Path(cfg["sdk"])
JAVA = Path(cfg["java"])
bt = cfg["bt_package"] if TRANSPORT == "from_cell_2" else TRANSPORT
os.environ.update(JAVA_HOME=str(JAVA), ANDROID_HOME=str(SDK), ANDROID_SDK_ROOT=str(SDK))
os.environ["PATH"] = os.pathsep.join([str(JAVA / "bin"), str(FLUTTER / "bin"),
                                    str(SDK / "cmdline-tools/latest/bin"), os.environ.get("PATH", "")])
stamp = time.strftime("%Y%m%d_%H%M%S")
REPORT = Path(f"/content/ssm2_build_{stamp}")
REPORT.mkdir(parents=True, exist_ok=True)
report = {"revision": "0.6", "transport": bt, "variant": BUILD_VARIANT,
          "flutter": cfg["flutter_version"], "stages": {}, "hardware_tested": False}


def save_report():
    (REPORT / "report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")


def stage(name, args, timeout=3600):
    logfile = REPORT / f"{name}.log"
    print(f"\n=== {name} ===\n> {' '.join(map(str, args))}\nЖурнал: {logfile}")
    started = time.monotonic()
    try:
        with logfile.open("w", encoding="utf-8") as out:
            # Output streams directly to disk, not into an unbounded memory buffer.
            p = subprocess.run(list(map(str, args)), cwd=APP, text=True,
                               stdout=out, stderr=subprocess.STDOUT, timeout=timeout)
        text = logfile.read_text(encoding="utf-8", errors="replace")
        print(text[-7000:])
        report["stages"][name] = {"exit_code": p.returncode, "seconds": round(time.monotonic() - started, 1)}
        save_report()
        if p.returncode:
            diagnose(text)
            raise RuntimeError(f"Этап {name} не прошел. Полный журнал: {logfile}")
        return text
    except subprocess.TimeoutExpired:
        report["stages"][name] = {"error": "timeout"}
        save_report()
        raise RuntimeError(f"Таймаут этапа {name}. Журнал: {logfile}")


def diagnose(text):
    print("\nДиагностика (без автоматического понижения версий):")
    if "minimum supported version" in text or "Minimum supported Gradle version" in text:
        print("Несовместимые версии Flutter/AGP/Gradle/Kotlin. Смотрите точный минимум в журнале.")
    if "Registrar" in text:
        print("Остался Flutter embedding v1. Проверьте локальный vendor и prepare_bt.log.")
    if "Namespace not specified" in text:
        print("Нужен namespace конкретного модуля. Общий pub-cache не изменяется.")
    if "AAR metadata" in text or "compileSdk" in text:
        print("Проверьте требуемый SDK зависимости. Нельзя лечить это снятием всех версий pubspec.")
    if "Permission denied" in text:
        print("Проверьте разрешения файлов SDK/JDK и исполняемый gradlew.")
    print("При неизвестной ошибке сохраните весь журнал, а не только последние строки.")


def at_least(value, minimum):
    return tuple(map(int, value.split("."))) >= tuple(map(int, minimum.split(".")))


print("SSM2 0.6 | Проверка перед сборкой")
required = [FLUTTER / "bin/flutter", FLUTTER / "bin/dart", JAVA / "bin/java",
            SDK / "platforms/android-36/android.jar", SDK / "build-tools/35.0.0/aapt",
            SDK / "build-tools/35.0.0/apksigner", SDK / "ndk/27.0.12077973/source.properties",
            APP / "transport_templates/classic.dart.txt", APP / "transport_templates/serial.dart.txt",
            APP / "test/protocol_test.dart", APP / "tool/prepare_bt.py",
            APP / "lib/derived.dart", APP / "lib/analyzer.dart"]
for file in required:
    if not file.exists(): raise RuntimeError(f"Отсутствует: {file}. Повторите соответствующую ячейку.")
    print("[OK]", file)

version_log = stage("flutter_version", [FLUTTER / "bin/flutter", "--version", "--machine"], timeout=600)
actual_flutter = json.JSONDecoder().raw_decode(version_log[version_log.index("{"):])[0]
report["flutter"] = actual_flutter["frameworkVersion"]
save_report()

# Reject accidental use of an old 8.7.0 settings file before touching Gradle.
settings_file = APP / "android/settings.gradle.kts"
settings = settings_file.read_text(encoding="utf-8")
for plugin, expected in [("com.android.application", cfg["agp"]), ("org.jetbrains.kotlin.android", cfg["kotlin"])]:
    match = re.search(r'id\("' + re.escape(plugin) + r'"\)\s+version\s+"([\d.]+)"', settings)
    if not match or match.group(1) != expected:
        raise RuntimeError(f"{settings_file}: ожидается {plugin} {expected}. Повторите ячейку 2, не старый автофикс.")
wrapper_file = APP / "android/gradle/wrapper/gradle-wrapper.properties"
wrapper = wrapper_file.read_text(encoding="utf-8")
if f"gradle-{cfg['gradle']}-bin.zip" not in wrapper:
    raise RuntimeError("Gradle wrapper не соответствует комплекту. Повторите ячейку 2.")
checker = FLUTTER / "packages/flutter_tools/gradle/src/main/kotlin/DependencyVersionChecker.kt"
if checker.exists():
    source = checker.read_text(encoding="utf-8")
    for name, key in [("errorAGPVersion", "agp"), ("errorGradleVersion", "gradle"), ("errorKGPVersion", "kotlin")]:
        match = re.search(rf"{name}\s*[^=]*=\s*(?:AndroidPluginVersion|Version)\(\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*\)", source)
        if match and not at_least(cfg[key], ".".join(match.groups())):
            raise RuntimeError(f"Установленный Flutter требует более новый {key}. Повторите ячейку 1 с подходящей версией Flutter.")

properties_file = APP / "android/local.properties"
properties = {}
if properties_file.exists():
    for line in properties_file.read_text(encoding="utf-8").splitlines():
        if "=" in line and not line.lstrip().startswith("#"):
            key, value = line.split("=", 1)
            properties[key.strip()] = value
properties.update({"sdk.dir": str(SDK), "flutter.sdk": str(FLUTTER)})
properties_file.write_text("\n".join(f"{key}={value}" for key, value in properties.items()) + "\n", encoding="utf-8")
gradlew = APP / "android/gradlew"
gradlew.chmod(gradlew.stat().st_mode | 0o111)
sha_url = f"https://services.gradle.org/distributions/gradle-{cfg['gradle']}-bin.zip.sha256"
with urllib.request.urlopen(sha_url, timeout=90) as response:
    wrapper_sha = response.read().decode().strip().split()[0]
if not re.fullmatch(r"[0-9a-fA-F]{64}", wrapper_sha): raise RuntimeError("Invalid Gradle SHA256 metadata")
wrapper = re.sub(r"(?m)^distributionSha256Sum=.*\n?", "", wrapper)
wrapper_file.write_text(wrapper.rstrip() + f"\ndistributionSha256Sum={wrapper_sha}\n", encoding="utf-8")

stage("prepare_bt", [sys.executable, APP / "tool/prepare_bt.py", bt], timeout=600)
cfg["bt_package"] = bt
CONFIG.write_text(json.dumps(cfg, indent=2), encoding="utf-8")
if CLEAN_BEFORE:
    stage("clean", [FLUTTER / "bin/flutter", "clean"], timeout=600)
stage("pub_get", [FLUTTER / "bin/flutter", "pub", "get"], timeout=1200)
stage("format", [FLUTTER / "bin/dart", "format", "lib", "test"], timeout=600)
stage("analyze", [FLUTTER / "bin/flutter", "analyze", "--no-pub", "--no-fatal-infos"], timeout=1200)
stage("tests", [FLUTTER / "bin/flutter", "test", "--no-pub", "--reporter", "expanded"], timeout=1200)

apk = APP / f"build/app/outputs/flutter-apk/app-{BUILD_VARIANT}.apk"
apk.unlink(missing_ok=True)
stage("build", [FLUTTER / "bin/flutter", "build", "apk", f"--{BUILD_VARIANT}", "--no-pub"], timeout=4800)
if not apk.exists() or apk.stat().st_size < 1024 * 1024:
    raise RuntimeError(f"Новый APK не найден: {apk}")
badging = stage("apk_badging", [SDK / "build-tools/35.0.0/aapt", "dump", "badging", apk], timeout=120)
if "package: name='com.subaru.ssm2_fixed'" not in badging:
    raise RuntimeError("Unexpected applicationId in APK")
for permission in ["BLUETOOTH_CONNECT", "BLUETOOTH_SCAN"]:
    if f"android.permission.{permission}" not in badging:
        raise RuntimeError(f"Missing APK permission: {permission}")
if "sdkVersion:'24'" not in badging: raise RuntimeError("Unexpected APK minSdk")
stage("apk_signature", [SDK / "build-tools/35.0.0/apksigner", "verify", "--verbose", apk], timeout=120)
digest = hashlib.sha256(apk.read_bytes()).hexdigest()
short = "A" if bt == "bluetooth_classic" else "B"
destination = Path(f"/content/ssm2_fixed_{short}_{BUILD_VARIANT}.apk")
shutil.copy2(apk, destination)
report.update({"apk": str(destination), "sha256": digest, "apk_verified": True,
               "signing": "local debug key for sideloading", "hardware_tested": False})
save_report()
shutil.copy2(APP / "pubspec.lock", REPORT / "pubspec.lock")
shutil.copy2(CONFIG, REPORT / "build_config.json")
shutil.make_archive(str(REPORT), "zip", REPORT)
print(f"\n[OK] Проверки и сборка завершены: {destination}")
print(f"SHA256: {digest}\nЖурналы: {REPORT}.zip")
print("APK использует отдельный applicationId и не заменяет старое приложение.")
print("Подпись: локальный debug-ключ, даже для release. Для Play Store нужен ваш release-ключ.")
print("Проверка на автомобиле НЕ выполнена. Начните с базовых PID; Extended включайте только после сверки ROM.")
try:
    from IPython.display import FileLink, display
    display(FileLink(str(destination)))
    display(FileLink(str(REPORT) + ".zip"))
except ImportError:
    print("Скачайте файлы через панель Файлы Colab.")

SSM2 0.6 | Проверка перед сборкой
[OK] /content/flutter/bin/flutter
[OK] /content/flutter/bin/dart
[OK] /usr/lib/jvm/java-17-openjdk-amd64/bin/java
[OK] /content/android-sdk/platforms/android-36/android.jar
[OK] /content/android-sdk/build-tools/35.0.0/aapt
[OK] /content/android-sdk/build-tools/35.0.0/apksigner
[OK] /content/android-sdk/ndk/27.0.12077973/source.properties
[OK] /content/subaru_ssm2_fixed/transport_templates/classic.dart.txt
[OK] /content/subaru_ssm2_fixed/transport_templates/serial.dart.txt
[OK] /content/subaru_ssm2_fixed/test/protocol_test.dart
[OK] /content/subaru_ssm2_fixed/tool/prepare_bt.py
[OK] /content/subaru_ssm2_fixed/lib/derived.dart
[OK] /content/subaru_ssm2_fixed/lib/analyzer.dart

=== flutter_version ===
> /content/flutter/bin/flutter --version --machine
Журнал: /content/ssm2_build_20260919_154926/flutter_version.log
{
  "frameworkVersion": "3.35.4",
  "channel": "stable",
  "repositoryUrl": "https://github.com/flutter/flutter.git",
  "frameworkRevision": "d

/content/ssm2_fixed_A_release.apk

/content/ssm2_build_20260919_154926.zip